# BJAI-2026-0104 — reference baselines and analyses

Deposited in response to EIC comment 4 of 19-Nov-2026:

> The released notebooks do not contain the code that constructs the reference
> baselines of Section 3.8 or the analyses of Sections 4.5 to 4.12. Please
> deposit that code, or amend the Data Availability Statement.

Every cell below is taken verbatim from the revision notebook that produced the
published numbers. Nothing has been rewritten, re-ordered or tidied for release,
so that what is deposited is what was executed.

| Manuscript | Where in this notebook |
|---|---|
| Section 3.1–3.3, cohort construction | cells "cohort", "sampler", "folds" |
| Section 3.8, RAM and metadata baselines | cell "baselines" (`ram_indicator`, `metadata_features`) |
| Section 3.9, evaluation protocol | cell "folds", and `hcv_revision.evaluate_balanced` |
| Sections 4.5–4.12, reported analyses | cells "statistics" and "analyses" |

The two modules are written to disk by the first two code cells, exactly as in
the main notebook. Run order is top to bottom.


## The project modules

Written to disk exactly as the main notebook writes them.

In [ ]:
%%writefile hcv_pipeline.py
"""
hcv_pipeline.py — unified, leakage-safe evaluation harness for HCV G4 resistance.

This module implements the experimental core of the paper AND the fixes for the
reviewer comments:

  R1 "How do you know similar sequences did not appear in train and validation?"
      -> splits are grouped by accession (group_id); leakage_audit() proves both
         group-disjointness and low cross-split sequence identity.
  R2 "The model is not clinically validated?"
      -> labels are Geno2pheno[HCV] silver-standard; see CLINICAL_CAVEAT and the
         notebook limitations cell. No clinical outcomes are claimed.
  R5 "Different n per model (1050/1890/1872) -> unfair comparison"
      -> build_common_cohort() evaluates every pipeline on ONE identical set of
         sequences with ONE split scheme and ONE task.

Design rule enforced here: classifiers may only use sequence-DERIVED features
(composition / k-mer / foundation-model embeddings). RAS/mutation annotation
columns are the label source and are NEVER used as inputs (that would be label
leakage). See SEQUENCE_FEATURE_PREFIXES and get_feature_matrix().
"""
from __future__ import annotations

import itertools
import warnings
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, matthews_corrcoef,
    roc_auc_score, average_precision_score, confusion_matrix,
)

warnings.filterwarnings("ignore")


def _free_gpu():
    """Release GPU memory between heavy stages so one run does not accumulate models."""
    try:
        import gc, torch
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass

LABELS_4 = ["susceptible", "polymorphism", "reduced_susceptibility", "resistant"]
# binary RAS-detection: RAS-positive = any resistance-associated call (matches the
# paper's neutral-vs-RAS framing). Configurable via ras_positive_classes.
DEFAULT_RAS_POSITIVE = ("polymorphism", "reduced_susceptibility", "resistant")

CLINICAL_CAVEAT = (
    "Labels are computational (Geno2pheno[HCV]) silver-standard, not clinical "
    "treatment outcomes. Results describe agreement with an in-silico interpreter "
    "on Egyptian G4 isolates, NOT validated clinical resistance. Prospective "
    "validation against SVR/treatment-failure data is required before any clinical use."
)

# columns that are the label source or label-adjacent -> forbidden as model inputs
FORBIDDEN_FEATURE_PREFIXES = (
    "ras_", "total_ras", "has_sof_ras",
    "ns3_n_", "ns5a_n_", "ns5b_n_", "ns3_ras", "ns5a_ras", "ns5b_ras",
    "ns3_scored", "ns5a_scored", "ns5b_scored",
    "ns3_mutations", "ns5a_mutations", "ns5b_mutations",
    "ns3_genotype", "ns5a_genotype", "ns5b_genotype",
)
# allowed sequence-derived feature prefixes present in hcv_enriched.csv
SEQUENCE_FEATURE_PREFIXES = ("a_frac", "c_frac", "g_frac", "t_frac", "n_frac",
                             "ambig_frac", "di_", "kmer", "seq_len", "gc_pct")


# ----------------------------------------------------------------------------- #
# Sequence translation (paper Sec. 3.1.1): multi-frame, longest ORF w/o stops
# ----------------------------------------------------------------------------- #
from Bio.Seq import Seq


def translate_longest_orf(nt: str) -> str:
    """Translate all 3 forward frames; return the aa string of the frame whose
    longest internal stop-free stretch is greatest."""
    nt = str(nt).upper().replace("U", "T")
    best = ""
    for f in range(3):
        sub = nt[f:]
        sub = sub[: len(sub) - (len(sub) % 3)]
        if len(sub) < 3:
            continue
        aa = str(Seq(sub).translate())
        longest = max(aa.split("*"), key=len) if aa else ""
        if len(longest) > len(best):
            best = longest
    return best


# ----------------------------------------------------------------------------- #
# Cohort construction
# ----------------------------------------------------------------------------- #
@dataclass
class Cohort:
    df: pd.DataFrame                 # rows in the cohort (may include augmented)
    task: str                        # "4class" or "binary"
    label_col: str                   # resolved target column
    classes: list                    # ordered class names / [0,1]
    ras_positive: tuple = DEFAULT_RAS_POSITIVE
    notes: dict = field(default_factory=dict)


def build_common_cohort(df: pd.DataFrame, task: str = "4class",
                        real_only_eval: bool = True, min_aa_len: int = 20,
                        ras_positive=DEFAULT_RAS_POSITIVE) -> Cohort:
    """Restrict to a single shared cohort so every pipeline is compared on the
    same sequences (fixes R5). Filters: valid translation of >= min_aa_len aa.
    real_only_eval keeps augmented rows for training but they are flagged; eval
    is later restricted to is_real via eval_eligible."""
    d = df.copy()
    if "aa_seq" not in d.columns:
        d["aa_seq"] = d["nt_sequence"].map(translate_longest_orf)
    d["aa_len"] = d["aa_seq"].str.len()
    keep = d["aa_len"] >= min_aa_len
    d = d[keep].reset_index(drop=True)

    if task == "4class":
        label_col, classes = "label", LABELS_4
        d = d[d["label"].isin(LABELS_4)].reset_index(drop=True)
    elif task == "binary":
        label_col, classes = "y_bin", [0, 1]
        d["y_bin"] = d["label"].isin(ras_positive).astype(int)
    else:
        raise ValueError("task must be '4class' or 'binary'")

    if "eval_eligible" not in d.columns:
        d["eval_eligible"] = (~d.get("is_augmented", False)).astype(bool) if "is_augmented" in d else True

    notes = {
        "n_rows": len(d), "n_real": int(d["eval_eligible"].sum()),
        "n_groups": d["group_id"].nunique(),
        "min_aa_len": min_aa_len, "dropped_short": int((~keep).sum()),
    }
    return Cohort(df=d, task=task, label_col=label_col, classes=list(classes),
                  ras_positive=tuple(ras_positive), notes=notes)


# ----------------------------------------------------------------------------- #
# Feature extractors  (each returns an (n_rows, d) float matrix aligned to cohort.df)
# ----------------------------------------------------------------------------- #
def _kmer_matrix(seqs: pd.Series, k: int, alphabet="ACGT") -> pd.DataFrame:
    cols = ["".join(t) for t in itertools.product(alphabet, repeat=k)]
    idx = {c: j for j, c in enumerate(cols)}
    M = np.zeros((len(seqs), len(cols)), dtype=np.float32)
    for i, s in enumerate(seqs.astype(str)):
        s = s.upper()
        L = len(s)
        if L < k:
            continue
        tot = L - k + 1
        for p in range(tot):
            j = idx.get(s[p:p + k])
            if j is not None:
                M[i, j] += 1.0
        M[i] /= tot
    return pd.DataFrame(M, columns=[f"kmer{k}_{c}" for c in cols], index=seqs.index)


def features_sequence_composition(cohort: Cohort) -> pd.DataFrame:
    """Use the sequence-composition columns already in hcv_enriched.csv."""
    d = cohort.df
    cols = [c for c in d.columns
            if c.startswith(("a_frac", "c_frac", "g_frac", "t_frac", "n_frac",
                             "ambig_frac", "di_"))]
    X = d[cols].astype(np.float32).fillna(0.0)
    _assert_no_leak(X.columns)
    return X


def features_kmer(cohort: Cohort, k: int = 3) -> pd.DataFrame:
    """Composition + k-mer frequencies computed from nt_sequence."""
    base = features_sequence_composition(cohort)
    km = _kmer_matrix(cohort.df["nt_sequence"], k)
    X = pd.concat([base, km], axis=1)
    _assert_no_leak(X.columns)
    return X


def features_esm2_embeddings(cohort: Cohort, model_name="facebook/esm2_t33_650M_UR50D",
                             device=None, batch_size=8, cache=None, max_len=1022,
                             show_progress=True) -> pd.DataFrame:
    """Frozen ESM-2, mean-pooled 1280-d embeddings. model_name may be a fine-tuned dir."""
    import os
    if cache and os.path.exists(cache):
        return pd.read_parquet(cache)
    import torch
    from transformers import AutoTokenizer, AutoModel
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device).eval()
    seqs_all = cohort.df["aa_seq"].tolist()
    uniq = list(dict.fromkeys(seqs_all))
    rng = range(0, len(uniq), batch_size)
    try:
        from tqdm.auto import tqdm
        if show_progress: rng = tqdm(rng, desc="ESM-2 embed")
    except Exception:
        pass
    emb_map = {}
    with torch.no_grad():
        for i in rng:
            batch = [s[:max_len] for s in uniq[i:i + batch_size]]
            enc = tok(batch, return_tensors="pt", padding=True, truncation=True,
                      max_length=max_len + 2).to(device)
            hs = model(**enc).last_hidden_state
            mask = enc["attention_mask"].unsqueeze(-1)
            emb = (hs * mask).sum(1) / mask.sum(1)
            for s, e in zip(uniq[i:i + batch_size], emb.cpu().numpy()):
                emb_map[s] = e
    dim = next(iter(emb_map.values())).shape[0]
    M = np.vstack([emb_map[s] for s in seqs_all])
    X = pd.DataFrame(M, columns=[f"esm2_{j}" for j in range(dim)],
                     index=cohort.df.index).astype(np.float32)
    if cache:
        X.to_parquet(cache)
    del model, tok; _free_gpu()
    return X


def features_structural_placeholder(cohort: Cohort, kind: str) -> pd.DataFrame:
    """Interface for ESMFold (Sec. 3.6) / AlphaFold2 (Sec. 3.7) structural features.
    These require ESMFold / ColabFold external compute; load your cached feature
    parquet here. Raises to avoid silently substituting fake structures."""
    raise NotImplementedError(
        f"{kind} structural features require external GPU/ColabFold extraction. "
        f"Extract per paper Sec. 3.6/3.7, save a parquet aligned to group_id, and "
        f"load it here. Do NOT substitute placeholders (the paper notes zero-filled "
        f"placeholders caused identity leakage and unrealistic scores).")


def features_esmfold(cohort: Cohort, model_name="facebook/esmfold_v1", device=None,
                     max_len=350, cache=None, show_progress=True, trunk_ckpt=None, chunk=32) -> pd.DataFrame:
    """ESMFold structural features (1032-d): 1024 mean-pooled trunk s_s + 3 pLDDT
    stats + 5 Ca-Ca distance stats. Memory-safe for ~14-16GB GPUs:
      - chunked folding (set_chunk_size)
      - fine-tuned trunk loaded from a state_dict FILE (NO second 3B model)
      - per-sequence OOM guard (zero-fill a sequence that fails)
      - automatic CPU fallback if the GPU cannot hold the model."""
    import os, glob
    if cache and os.path.exists(cache):
        return pd.read_parquet(cache)
    import torch
    from transformers import AutoTokenizer, EsmForProteinFolding
    tok = AutoTokenizer.from_pretrained(model_name)
    cols = ([f"esmf_ss_{i}" for i in range(1024)] +
            ["plddt_mean", "plddt_std", "plddt_min",
             "ca_mean", "ca_std", "ca_min", "ca_max", "ca_median"])
    seqs_all = cohort.df["aa_seq"].tolist()
    uniq = list(dict.fromkeys(seqs_all))

    def _load(dev):
        m = EsmForProteinFolding.from_pretrained(model_name, low_cpu_mem_usage=True).to(dev).eval()
        if trunk_ckpt:
            try:
                sf = sorted(glob.glob(os.path.join(trunk_ckpt, "*.safetensors")))
                if sf:
                    from safetensors.torch import load_file
                    sd = {}
                    for f in sf:
                        sd.update(load_file(f))
                else:
                    sd = torch.load(os.path.join(trunk_ckpt, "pytorch_model.bin"), map_location="cpu")
                miss, unexp = m.esm.load_state_dict(sd, strict=False)
                del sd; _free_gpu()
                print(f"loaded fine-tuned ESMFold trunk (missing={len(miss)}, unexpected={len(unexp)})")
            except Exception as e:
                print("fine-tuned trunk load skipped:", e)
        if dev == "cuda":
            try: m.esm = m.esm.half()
            except Exception: pass
        try: m.trunk.set_chunk_size(chunk)
        except Exception: pass
        return m

    def _feat(out):
        s_s = out.s_s[0].float().mean(0).cpu().numpy()
        pr = out.plddt[0].float().cpu().numpy()
        plddt = pr.mean(-1) if pr.ndim == 2 else pr
        pl = np.array([plddt.mean(), plddt.std(), plddt.min()])
        ca = out.positions[-1, 0, :, 1, :].float().cpu().numpy()
        d = np.sqrt(((ca[:, None, :] - ca[None, :, :]) ** 2).sum(-1))
        iu = np.triu_indices(len(ca), k=1)
        dd = d[iu] if len(iu[0]) else np.array([0.0])
        dstats = np.array([dd.mean(), dd.std(), dd.min(), dd.max(), np.median(dd)])
        return np.concatenate([s_s, pl, dstats]).astype(np.float32)

    def _run(dev):
        m = _load(dev)
        it = uniq
        try:
            from tqdm.auto import tqdm
            if show_progress: it = tqdm(uniq, desc=f"ESMFold[{dev}]")
        except Exception:
            pass
        feat = {}
        with torch.no_grad():
            for s in it:
                seq = (str(s)[:max_len]) or "M"
                try:
                    enc = tok([seq], return_tensors="pt", add_special_tokens=False,
                              padding=True, truncation=True, max_length=max_len)
                    out = m(enc["input_ids"].to(dev))
                    feat[s] = _feat(out)
                except Exception:
                    _free_gpu()
                    feat[s] = np.zeros(1032, dtype=np.float32)
        del m; _free_gpu()
        return feat

    _free_gpu()
    dev0 = device or ("cuda" if torch.cuda.is_available() else "cpu")
    try:
        feat = _run(dev0)
    except Exception as e:
        _free_gpu()
        print("ESMFold GPU load failed -> CPU fallback (slower, safe):", str(e)[:100])
        feat = _run("cpu")
    M = np.vstack([feat[s] for s in seqs_all])
    X = pd.DataFrame(M, columns=cols, index=cohort.df.index).astype(np.float32)
    if cache:
        X.to_parquet(cache)
    return X

def features_colabfold_parse(cohort: Cohort, result_dir: str,
                             ras_positions=(28, 30, 31, 93, 282, 316),
                             single_repr_dim=384) -> pd.DataFrame:
    """FAITHFUL paper implementation (Sec. 3.7): parse AlphaFold2/ColabFold outputs
    into leakage-safe structural features:
      mean pLDDT, pLDDT at RAS reference positions, mean-pooled 384-d single
      representation, sequence length, NS5A/NS5B presence flags.

    ColabFold is run separately (it needs its own JAX/CUDA environment and the
    MMseqs2 API). Point result_dir at its output folder. Expected per sequence:
      <name>_unrelaxed_rank_001_*.pdb        (B-factor column = pLDDT)
      <name>_*_single_repr.npy   (optional)  (L, 384) single representation

    Matching is by a 'cf_name' column on cohort.df (set it to the ColabFold job
    name you used per sequence). Rows without a folded structure get NaN features
    (drop or impute downstream)."""
    import glob
    import numpy as np
    d = cohort.df
    if "cf_name" not in d.columns:
        raise ValueError("cohort.df needs a 'cf_name' column mapping each row to its "
                         "ColabFold job name (the FASTA header you folded).")

    def parse_pdb_plddt(pdb):
        plddt, resid = [], []
        for line in open(pdb):
            if line.startswith("ATOM") and line[12:16].strip() == "CA":
                resid.append(int(line[22:26]))
                plddt.append(float(line[60:66]))
        return np.array(resid), np.array(plddt)

    rows = []
    ras_cols = [f"plddt_ras_{p}" for p in ras_positions]
    for _, r in d.iterrows():
        name = r["cf_name"]
        pdbs = sorted(glob.glob(os.path.join(result_dir, f"{name}*rank_001*.pdb"))
                      + glob.glob(os.path.join(result_dir, f"{name}*.pdb")))
        feat = {c: np.nan for c in ["plddt_mean"] + ras_cols + [f"single_{i}" for i in range(single_repr_dim)]}
        feat["seq_len"] = len(str(r["aa_seq"]))
        feat["has_ns5a"] = int(bool(r.get("has_ns5a", False)))
        feat["has_ns5b"] = int(bool(r.get("has_ns5b", False)))
        if pdbs:
            resid, plddt = parse_pdb_plddt(pdbs[0])
            if len(plddt):
                feat["plddt_mean"] = float(plddt.mean())
                pos2plddt = dict(zip(resid, plddt))
                for p, c in zip(ras_positions, ras_cols):
                    feat[c] = float(pos2plddt.get(p, np.nan))
            npy = sorted(glob.glob(os.path.join(result_dir, f"{name}*single*.npy")))
            if npy:
                sr = np.load(npy[0])
                if sr.ndim == 2 and sr.shape[1] == single_repr_dim:
                    for i, v in enumerate(sr.mean(0)):
                        feat[f"single_{i}"] = float(v)
        rows.append(feat)
    X = pd.DataFrame(rows, index=d.index).astype(np.float32)
    return X


import os  # noqa: E402  (used by features_colabfold_parse)


# ----------------------------------------------------------------------------- #
# Continued pretraining (paper Sec. 3.2.1): masked-language-model fine-tuning of
# ESM-2 on the Egyptian G4 corpus. Only the LM head + the last N transformer
# layers are updated; the rest stays frozen. GPU-gated.
# ----------------------------------------------------------------------------- #
def continued_pretrain_esm2(sequences, model_name="facebook/esm2_t33_650M_UR50D",
                            out_dir="esm2_finetuned", n_unfrozen_layers=3, epochs=3,
                            mlm_prob=0.15, batch_size=4, lr=1e-4, max_len=1022,
                            device=None, show_progress=True):
    """Continued MLM pretraining of ESM-2 on `sequences` (list of aa strings).
    Freezes everything except the LM head and the last `n_unfrozen_layers`
    encoder layers, matching the paper. Saves the adapted model to `out_dir` and
    returns that path; feed it back as `model_name` to features_esm2_embeddings."""
    import torch
    from torch.utils.data import DataLoader
    from transformers import (AutoTokenizer, AutoModelForMaskedLM,
                              DataCollatorForLanguageModeling)
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForMaskedLM.from_pretrained(model_name).to(device)

    # discover the last N encoder-layer indices and freeze everything else
    import re as _re
    layer_ids = set()
    for name, _ in model.named_parameters():
        m = _re.search(r"encoder\.layer\.(\d+)\.", name)
        if m:
            layer_ids.add(int(m.group(1)))
    keep = set(sorted(layer_ids)[-n_unfrozen_layers:]) if layer_ids else set()
    for name, p in model.named_parameters():
        train = ("lm_head" in name) or any(f"encoder.layer.{i}." in name for i in keep)
        p.requires_grad = bool(train)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"continued-pretraining: unfreezing layers {sorted(keep)} + lm_head "
          f"({trainable/1e6:.1f}M trainable params)")

    seqs = [s[:max_len] for s in sequences if s and len(s) >= 5]
    enc = tok(seqs, truncation=True, max_length=max_len, padding=False)
    examples = [{"input_ids": ids} for ids in enc["input_ids"]]
    collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=True, mlm_probability=mlm_prob)
    loader = DataLoader(examples, batch_size=batch_size, shuffle=True, collate_fn=collator)

    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.01)
    model.train()
    for ep in range(epochs):
        it = loader
        try:
            from tqdm.auto import tqdm
            if show_progress:
                it = tqdm(loader, desc=f"MLM epoch {ep+1}/{epochs}")
        except Exception:
            pass
        tot = 0.0; n = 0
        for batch in it:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            loss = out.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            opt.step(); opt.zero_grad()
            tot += float(loss); n += 1
        print(f"  epoch {ep+1}: mean MLM loss {tot/max(n,1):.4f}")
    os.makedirs(out_dir, exist_ok=True)
    model.save_pretrained(out_dir); tok.save_pretrained(out_dir)
    print("saved fine-tuned ESM-2 to", out_dir)
    del model, opt; _free_gpu()
    return out_dir


# ----------------------------------------------------------------------------- #
# Feature composition + grid evaluation (classifier x feature-source matrix)
# ----------------------------------------------------------------------------- #
def concat_features(*Xs) -> pd.DataFrame:
    """Concatenate aligned feature matrices column-wise (same row index)."""
    return pd.concat(list(Xs), axis=1)


def run_grid(cohort: Cohort, sources: dict, classifiers, n_splits=5,
             group_col="cluster_group", seed=42):
    """Evaluate every (feature-source, classifier) pair on one cohort/split.
    Returns a tidy DataFrame (mean +/- std across folds + OOF CIs) and the raw
    store keyed by (source, classifier)."""
    rows, store = [], {}
    for sname, X in sources.items():
        for clf in classifiers:
            r = evaluate_pipeline(cohort, X, clf_kind=clf, n_splits=n_splits,
                                  group_col=group_col, seed=seed)
            clabel = clf if isinstance(clf, str) else "+".join(clf)
            store[(sname, clabel)] = r
            fm, mo = r["fold_mean_std"], r["metrics_oof"]
            g = lambda k: fm.get(k, (np.nan, np.nan))
            rows.append({
                "features": sname, "classifier": clabel,
                "AUROC_mean": g("auroc_macro")[0], "AUROC_std": g("auroc_macro")[1],
                "PR_AUC": g("pr_auc")[0], "F1": g("f1_macro")[0], "MCC": g("mcc")[0],
                "BalAcc": g("balanced_acc")[0], "Sens": g("sensitivity")[0],
                "Spec": g("specificity")[0], "Acc": mo["accuracy"],
                "Acc_CI_lo": mo["acc_ci95"][0], "Acc_CI_hi": mo["acc_ci95"][1],
            })
    return pd.DataFrame(rows), store


def _assert_no_leak(cols):
    bad = [c for c in cols if any(c.startswith(p) for p in FORBIDDEN_FEATURE_PREFIXES)]
    if bad:
        raise ValueError(f"Label-leaking features in matrix: {bad[:6]} ... "
                         f"RAS/mutation annotations are the label source; not allowed as inputs.")


# ----------------------------------------------------------------------------- #
# Identity clustering (strong R1 fix): merge near-duplicate accessions into one
# group so highly similar sequences can never be split across train/val.
# ----------------------------------------------------------------------------- #
def add_identity_clusters(cohort: Cohort, threshold=0.90, k=6) -> Cohort:
    """Greedy single-linkage clustering of REAL sequences by k-mer Jaccard.
    Adds a 'cluster_group' column (augmented rows inherit their accession's
    cluster). Use group_col='cluster_group' in splits/audit for the strict fix."""
    d = cohort.df
    reps = (d[d["eval_eligible"]]
            .drop_duplicates("group_id")[["group_id", "nt_sequence"]]
            .reset_index(drop=True))

    def kset(s):
        s = str(s).upper()
        return set(s[i:i + k] for i in range(len(s) - k + 1))

    ksets = [kset(s) for s in reps["nt_sequence"]]
    cluster_of = {}
    clusters = []            # list of (representative_index)
    for i, gi in enumerate(reps["group_id"]):
        placed = False
        for cidx, rep_i in enumerate(clusters):
            a, b = ksets[i], ksets[rep_i]
            if a and b and len(a & b) / len(a | b) >= threshold:
                cluster_of[gi] = cidx
                placed = True
                break
        if not placed:
            cluster_of[gi] = len(clusters)
            clusters.append(i)
    d = d.copy()
    d["cluster_group"] = d["group_id"].map(cluster_of).fillna(-1).astype(int)
    coh = Cohort(df=d, task=cohort.task, label_col=cohort.label_col,
                 classes=cohort.classes, ras_positive=cohort.ras_positive,
                 notes={**cohort.notes, "n_clusters": len(clusters),
                        "n_accessions": len(reps),
                        "merged_accessions": len(reps) - len(clusters)})
    return coh


# ----------------------------------------------------------------------------- #
# Leakage audit (R1)
# ----------------------------------------------------------------------------- #
def leakage_audit(cohort: Cohort, n_splits=5, seed=42, sample_pairs=4000, group_col='group_id') -> dict:
    """Prove leakage control: (a) no group spans folds; (b) max k-mer(6) Jaccard
    identity between real train and real val sequences is bounded well below 1."""
    d = cohort.df
    y = d[cohort.label_col].to_numpy()
    groups = d[group_col].to_numpy()
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    group_disjoint = True
    identities = []
    rng = np.random.default_rng(seed)

    def kset(s, k=6):
        s = str(s).upper()
        return set(s[i:i + k] for i in range(len(s) - k + 1))

    for tr, va in sgkf.split(d, y, groups):
        if set(groups[tr]) & set(groups[va]):
            group_disjoint = False
        # sequence-identity on REAL sequences only
        rtr = d.iloc[tr]; rva = d.iloc[va]
        rtr = rtr[rtr["eval_eligible"]]["nt_sequence"].tolist()
        rva = rva[rva["eval_eligible"]]["nt_sequence"].tolist()
        if not rtr or not rva:
            continue
        for _ in range(min(sample_pairs, len(rtr) * len(rva))):
            a = kset(rtr[rng.integers(len(rtr))])
            b = kset(rva[rng.integers(len(rva))])
            if a and b:
                identities.append(len(a & b) / len(a | b))
    identities = np.array(identities) if identities else np.array([0.0])
    return {
        "group_col": group_col,
        "group_disjoint": bool(group_disjoint),
        "max_train_val_identity": float(identities.max()),
        "p99_identity": float(np.percentile(identities, 99)),
        "mean_identity": float(identities.mean()),
        "n_pairs_checked": int(len(identities)),
    }


# ----------------------------------------------------------------------------- #
# Metrics
# ----------------------------------------------------------------------------- #
def _safe_auroc(y, proba, classes):
    try:
        if len(classes) == 2:
            return roc_auc_score(y, proba[:, 1])
        # roc_auc_score(labels=...) requires ordered labels; reorder proba columns
        # to match sorted class names so probabilities stay aligned to their class.
        order = np.argsort(classes)
        classes_sorted = [classes[i] for i in order]
        proba_sorted = proba[:, order]
        return roc_auc_score(y, proba_sorted, multi_class="ovr",
                             average="macro", labels=classes_sorted)
    except Exception:
        return np.nan


def _safe_prauc(y, proba, classes):
    try:
        if len(classes) == 2:
            return average_precision_score(y, proba[:, 1])
        aps = []
        for j, c in enumerate(classes):
            yc = (np.asarray(y) == c).astype(int)
            if yc.sum() > 0:
                aps.append(average_precision_score(yc, proba[:, j]))
        return float(np.mean(aps)) if aps else np.nan
    except Exception:
        return np.nan


def compute_metrics(y, y_pred, proba, classes):
    from sklearn.metrics import balanced_accuracy_score
    m = {
        "accuracy": accuracy_score(y, y_pred),
        "balanced_acc": balanced_accuracy_score(y, y_pred),
        "auroc_macro": _safe_auroc(y, proba, classes),
        "pr_auc": _safe_prauc(y, proba, classes),
        "f1_macro": f1_score(y, y_pred, average="macro", labels=classes, zero_division=0),
        "mcc": matthews_corrcoef(y, y_pred),
    }
    # sensitivity / specificity are defined for the binary task
    if len(classes) == 2:
        cm = confusion_matrix(y, y_pred, labels=classes)
        tn, fp, fn, tp = cm.ravel()
        m["sensitivity"] = tp / (tp + fn) if (tp + fn) else np.nan
        m["specificity"] = tn / (tn + fp) if (tn + fp) else np.nan
    return m


def wilson_ci(k, n, z=1.96):
    """95% Wilson binomial CI for a proportion (used for accuracy)."""
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (max(0.0, center - half), min(1.0, center + half))


def bootstrap_auroc_ci(y, proba, classes, n_boot=1000, seed=42):
    """Percentile bootstrap 95% CI for AUROC over the pooled OOF predictions."""
    rng = np.random.default_rng(seed)
    y = np.asarray(y)
    n = len(y)
    if n == 0:
        return (np.nan, np.nan)
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yv = y[idx]
        if len(np.unique(yv)) < 2:            # need both classes present
            continue
        a = _safe_auroc(yv, proba[idx], classes)
        if not np.isnan(a):
            vals.append(a)
    if not vals:
        return (np.nan, np.nan)
    return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5)))


# ----------------------------------------------------------------------------- #
# Classifiers (mirror the paper: linear probe / MLP / LightGBM)
# ----------------------------------------------------------------------------- #

# --------------------------------------------------------------------------- #
# Linear-probe constructor. Feature sources that do not cover every record carry
# missing values, so the median imputer runs ahead of the scaler. For the RAM
# vector of Section 3.8 the imputer is a no-op, because that vector's elements
# are 0 or 1 by construction.
# --------------------------------------------------------------------------- #
from sklearn.impute import SimpleImputer as _SimpleImputer
from sklearn.linear_model import LogisticRegression as _LogisticRegression
from sklearn.pipeline import make_pipeline as _make_pipeline
from sklearn.preprocessing import StandardScaler as _StandardScaler


def _linear(**kw):
    return _make_pipeline(_SimpleImputer(strategy="median"), _StandardScaler(),
                          _LogisticRegression(**kw))


def make_classifier(kind: str, n_classes: int, seed=42):
    if kind == "logreg":                # ESM-2 linear-probe analogue
        return _linear(max_iter=2000, class_weight="balanced", C=1.0)
    if kind == "mlp":                   # structural-MLP analogue
        return MLPClassifier(hidden_layer_sizes=(256, 128, 64), alpha=1e-3,
                             max_iter=400, early_stopping=True, random_state=seed)
    if kind == "randomforest":
        from sklearn.ensemble import RandomForestClassifier
        return RandomForestClassifier(n_estimators=300, max_depth=None,
                                      class_weight="balanced", n_jobs=-1, random_state=seed)
    if kind == "xgboost":
        from xgboost import XGBClassifier
        return XGBClassifier(n_estimators=600, learning_rate=0.04, max_depth=6,
                             subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
                             tree_method="hist", random_state=seed, n_jobs=-1)
    if kind == "lightgbm":
        import lightgbm as lgb
        return lgb.LGBMClassifier(n_estimators=600, learning_rate=0.04, max_depth=6,
                                  subsample=0.8, colsample_bytree=0.8,
                                  class_weight="balanced", random_state=seed, verbose=-1)
    raise ValueError(kind)


# conventional-ML baselines reported in the paper's Experiment 4
BASELINE_CLASSIFIERS = ["logreg", "randomforest", "xgboost"]
ALL_CLASSIFIERS = ["logreg", "randomforest", "xgboost", "lightgbm", "mlp"]


# ----------------------------------------------------------------------------- #
# Unified evaluation harness — identical for every pipeline (fixes R5)
# ----------------------------------------------------------------------------- #
def evaluate_pipeline(cohort: Cohort, X: pd.DataFrame, clf_kind="logreg",
                      n_splits=5, seed=42, group_col="group_id"):
    """StratifiedGroupKFold on accession; TRAIN on all rows in train folds
    (real+augmented), EVALUATE on real rows only (eval_eligible). Returns
    out-of-fold predictions + aggregated metrics + per-fold metrics."""
    d = cohort.df
    y = d[cohort.label_col].to_numpy()
    groups = d[group_col].to_numpy()
    classes = cohort.classes
    Xv = X.to_numpy(dtype=np.float32)

    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    n = len(d)
    n_c = len(classes)
    oof_proba = np.full((n, n_c), np.nan)
    oof_pred = np.empty(n, dtype=object)
    per_fold = []

    for k, (tr, va) in enumerate(sgkf.split(d, y, groups)):
        eval_mask = d.iloc[va]["eval_eligible"].to_numpy()
        va_eval = va[eval_mask]
        if len(va_eval) == 0:
            continue
        scaler = StandardScaler().fit(Xv[tr])
        Xtr, Xva = scaler.transform(Xv[tr]), scaler.transform(Xv[va_eval])
        # clf_kind may be a single name or a list -> ensemble by proba averaging
        kinds = [clf_kind] if isinstance(clf_kind, str) else list(clf_kind)
        proba_stack = []
        for kk in kinds:
            clf = make_classifier(kk, n_c, seed)
            clf.fit(Xtr, y[tr])
            cls_order = list(clf.classes_)
            pr = clf.predict_proba(Xva)
            pr_full = np.zeros((len(va_eval), n_c))
            for j, c in enumerate(classes):
                if c in cls_order:
                    pr_full[:, j] = pr[:, cls_order.index(c)]
            proba_stack.append(pr_full)
        proba_full = np.mean(proba_stack, axis=0)
        pred = np.array(classes)[proba_full.argmax(1)]
        oof_proba[va_eval] = proba_full
        oof_pred[va_eval] = pred
        yv = y[va_eval]
        m = compute_metrics(yv, pred, proba_full, classes)
        m["fold"] = k; m["n_val_real"] = int(len(va_eval))
        per_fold.append(m)

    ev = ~pd.isna(oof_proba[:, 0])
    yv = y[ev]
    # rebuild predictions with inferred dtype (str for 4-class, int for binary)
    oof_pred_ev = np.array(list(oof_pred[ev]))
    agg = compute_metrics(yv, oof_pred_ev, oof_proba[ev], classes)
    agg["n_eval_real"] = int(ev.sum())

    # confidence intervals (paper Experiment 4): Wilson for accuracy, bootstrap AUROC
    k_correct = int((oof_pred_ev == yv).sum())
    lo, hi = wilson_ci(k_correct, len(yv))
    agg["acc_ci95"] = (round(lo, 4), round(hi, 4))
    alo, ahi = bootstrap_auroc_ci(yv, oof_proba[ev], classes, seed=seed)
    agg["auroc_ci95"] = (round(alo, 4), round(ahi, 4))

    per_fold_df = pd.DataFrame(per_fold)
    # mean ± std across folds for every numeric metric (paper reports these)
    metric_cols = [c for c in per_fold_df.columns if c not in ("fold", "n_val_real")]
    fold_summary = {c: (float(per_fold_df[c].mean()), float(per_fold_df[c].std()))
                    for c in metric_cols} if len(per_fold_df) else {}
    return {
        "clf": clf_kind, "task": cohort.task, "n_splits": n_splits,
        "metrics_oof": agg,
        "fold_mean_std": fold_summary,
        "per_fold": per_fold_df,
        "oof_true": yv, "oof_pred": oof_pred_ev, "oof_proba": oof_proba[ev],
        "classes": classes,
    }


# ----------------------------------------------------------------------------- #
# Explainable AI: uniform feature attribution across every classifier x feature
# source. Permutation importance is the model-agnostic backbone (works for all);
# SHAP (TreeExplainer) enriches tree models and coefficients explain the linear
# probe. On the SeqKmer3 source features are real k-mer motifs (interpretable);
# on ESM-2/ESMFold/ColabFold sources they are latent dims (influence, not motif).
# ----------------------------------------------------------------------------- #
def explain_model(cohort: Cohort, X: pd.DataFrame, clf_kind, top_k=15, n_repeats=5,
                  seed=42, shap_max=200, scoring=None):
    from sklearn.inspection import permutation_importance
    y = cohort.df[cohort.label_col].to_numpy()
    scaler = StandardScaler().fit(X.values.astype(float))
    Xs = scaler.transform(X.values.astype(float))
    kind = clf_kind if isinstance(clf_kind, str) else clf_kind[0]
    clf = make_classifier(kind, len(cohort.classes), seed)
    clf.fit(Xs, y)
    imp = pd.DataFrame({"feature": list(X.columns)})
    method, shap_summary, sort_col = None, None, None
    try:
        if kind in ("xgboost", "randomforest", "lightgbm"):
            import shap
            expl = shap.TreeExplainer(clf)
            sv = expl.shap_values(Xs[:shap_max])
            if isinstance(sv, list):
                sv = sv[1] if len(sv) > 1 else sv[0]
            sv = np.asarray(sv)
            if sv.ndim == 3:
                sv = sv[:, :, -1]
            imp["mean_abs_shap"] = np.abs(sv).mean(0)
            method, sort_col = "shap_tree", "mean_abs_shap"
            shap_summary = (expl, sv, X.iloc[:shap_max].copy())
        elif kind == "logreg":
            coef = clf.coef_
            imp["abs_coef"] = np.abs(coef).mean(0) if coef.ndim > 1 else np.abs(coef.ravel())
            method, sort_col = "coef", "abs_coef"
    except Exception:
        method = None
    # permutation importance only when no cheaper explainer applied (e.g. MLP) or as fallback
    if method is None:
        scoring = scoring or ("roc_auc" if len(cohort.classes) == 2 else "accuracy")
        pi = permutation_importance(clf, Xs, y, n_repeats=n_repeats, random_state=seed, scoring=scoring)
        imp["perm_importance"] = pi.importances_mean
        imp["perm_std"] = pi.importances_std
        method, sort_col = "permutation", "perm_importance"
    top = imp.sort_values(sort_col, ascending=False).head(top_k).reset_index(drop=True)
    return {"top": top, "full": imp, "clf": clf, "scaler": scaler,
            "method": method, "sort_col": sort_col, "shap_summary": shap_summary}


def explain_grid(cohort: Cohort, sources: dict, classifiers, top_k=15, n_repeats=5, seed=42):
    """Explain every (feature-source, classifier). Returns a tidy top-k table and a
    store of per-model explainers (for plotting)."""
    rows, store = [], {}
    for sname, X in sources.items():
        for clf in classifiers:
            clabel = clf if isinstance(clf, str) else "+".join(clf)
            try:
                res = explain_model(cohort, X, clf, top_k=top_k, n_repeats=n_repeats, seed=seed)
            except Exception as e:
                print(f"  explain {sname}/{clabel} skipped: {e}")
                continue
            store[(sname, clabel)] = res
            for _, r in res["top"].iterrows():
                rows.append({"features": sname, "classifier": clabel,
                             "feature": r["feature"], "importance": float(r[res["sort_col"]]),
                             "method": res["method"]})
    return pd.DataFrame(rows), store


# ----------------------------------------------------------------------------- #
# Explainability for the foundation models (ESM-2 / ESMFold / ColabFold).
# Two biologically grounded views instead of anonymous latent dims:
#   (a) per-residue occlusion of ESM-2 -> which positions drive the resistance
#       call, mapped onto known RAS sites;
#   (b) SHAP on the NAMED structural features (ESMFold pLDDT/Ca geometry,
#       ColabFold pLDDT-at-RAS-position) -> which structural signals matter.
# ----------------------------------------------------------------------------- #
RAS_REFERENCE_POSITIONS = {"NS3": [36, 54, 55, 80, 168],
                           "NS5A": [24, 28, 30, 31, 93],
                           "NS5B": [282, 316]}
ALL_RAS_POSITIONS = sorted({p for v in RAS_REFERENCE_POSITIONS.values() for p in v})


def structural_named_columns(X: pd.DataFrame):
    """Named (interpretable) structural columns, excluding latent embedding dims."""
    latent = ("esm2_", "esmf_ss_", "single_", "kmer", "di_",
              "a_frac", "c_frac", "g_frac", "t_frac", "n_frac", "ambig_frac")
    return [c for c in X.columns if not c.startswith(latent)]


def explain_structural_features(cohort: Cohort, X: pd.DataFrame, clf_kind="xgboost", top_k=15):
    """SHAP/importance on the NAMED structural features only (ESMFold pLDDT & Ca
    stats, ColabFold pLDDT-at-RAS-position, region flags). Returns None if X has
    no named structural columns (e.g. a pure-embedding source)."""
    named = structural_named_columns(X)
    if not named:
        return None
    return explain_model(cohort, X[named], clf_kind, top_k=top_k)


def explain_esm2_positions(cohort: Cohort, clf, scaler,
                           model_name="facebook/esm2_t33_650M_UR50D",
                           sample_n=20, stride=1, max_len=400, batch_size=16,
                           device=None, seed=42):
    """Per-residue occlusion attribution for an ESM-2-based classifier.
    For each sampled sequence: embed the original, then mask each residue (ESM
    <mask> token), re-embed, and record the drop in predicted resistance
    probability. Positive value => that residue supports the resistance call.
    Returns a per-position DataFrame (importance averaged across sequences) plus
    the RAS reference positions for overlay. GPU-gated; cost ~ sample_n x length."""
    import torch
    from transformers import AutoTokenizer, AutoModel
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device).eval()

    d = cohort.df[cohort.df["eval_eligible"]] if "eval_eligible" in cohort.df else cohort.df
    y = d[cohort.label_col].to_numpy()
    rng = np.random.default_rng(seed)
    pos_idx = np.where(y == 1)[0] if len(cohort.classes) == 2 else np.arange(len(d))
    if len(pos_idx) >= sample_n:
        sel = pos_idx[:sample_n]
    else:
        extra = rng.choice(len(d), sample_n - len(pos_idx), replace=False)
        sel = np.concatenate([pos_idx, extra])
    seqs = [d["aa_seq"].iloc[i][:max_len] for i in sel]

    def embed_ids(id_batch, attn_batch):
        out = []
        with torch.no_grad():
            for i in range(0, len(id_batch), batch_size):
                ids = torch.tensor(id_batch[i:i + batch_size]).to(device)
                att = torch.tensor(attn_batch[i:i + batch_size]).to(device)
                hs = model(input_ids=ids, attention_mask=att).last_hidden_state
                m = att.unsqueeze(-1)
                out.append(((hs * m).sum(1) / m.sum(1)).cpu().numpy())
        return np.vstack(out)

    def prob(embs):
        return clf.predict_proba(scaler.transform(embs))[:, 1]

    maxL = max(len(s) for s in seqs)
    agg = np.full((len(seqs), maxL), np.nan)
    for si, s in enumerate(seqs):
        enc = tok(s, truncation=True, max_length=max_len + 2)
        ids0, att0 = enc["input_ids"], enc["attention_mask"]
        L = len(s)
        base = prob(embed_ids([ids0], [att0]))[0]
        positions = list(range(0, L, stride))
        var_ids, var_att = [], []
        for p in positions:
            v = list(ids0); v[p + 1] = tok.mask_token_id     # +1 for CLS
            var_ids.append(v); var_att.append(att0)
        occ = prob(embed_ids(var_ids, var_att))
        for p, o in zip(positions, occ):
            agg[si, p] = base - o
    prof = np.nanmean(np.abs(agg), axis=0)
    out = pd.DataFrame({"position": np.arange(1, maxL + 1), "importance": prof})
    out["is_ras"] = out["position"].isin(ALL_RAS_POSITIONS)
    del model, tok; _free_gpu()
    return out

In [ ]:
# The linear-probe constructor now lives in hcv_pipeline.py, written by the cell
# above, so that make_classifier can reach it. Nothing to run here.
import hcv_pipeline as _hp
assert hasattr(_hp, "_linear"), "hcv_pipeline is missing the linear-probe helper"
print("[check] linear-probe helper available inside hcv_pipeline")

In [ ]:
%%writefile hcv_revision.py
"""
hcv_revision.py — revision experiments for BJAI-2026-0104.

Companion to hcv_pipeline.py. Every checklist item that needs new code lives
here; nothing in hcv_pipeline.py is modified, so the original results remain
reproducible from the original module.

Checklist coverage
------------------
  1  ram_baseline_matrix            RAM-position baseline (deliberately label-adjacent)
  2  metadata_baseline_matrix       dataset-bias baseline
  3  holdout_split / evaluate_holdout
  4  per_protein_cohorts
  5  stricter_cohorts               alternative RAS-positive mappings
  6  evaluate_balanced              class weighting + nested threshold tuning
  7  ablate_colabfold_ras           drop the six pLDDT-at-RAS features
  8  overlap_report                 fold + corpus overlap audit
  9  significance_tests             Wilcoxon / paired t across folds, Holm-corrected
 10  matched_operating_points, pr_curve_points
 11  bootstrap_metric_cis           95% CIs on every metric + positives per fold
 12  ResourceLog / Timer            wall time, peak VRAM, time per 1000 sequences

Invariant inherited from hcv_pipeline: models TRAIN on real + augmented rows and
are SCORED on real rows only (``eval_eligible``). Every function here keeps that.

Tensor/array shape convention in docstrings: ``N,D`` = rows x features.
"""
from __future__ import annotations

import json
import os
import platform
import time
import warnings
from contextlib import contextmanager
from dataclasses import dataclass, field, asdict
from typing import Iterable, Sequence

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score

import hcv_pipeline as H

warnings.filterwarnings("ignore")


# --------------------------------------------------------------------------- #
# Configuration — one dataclass, no constants hidden in function bodies
# --------------------------------------------------------------------------- #
@dataclass
class RevisionConfig:
    """Every knob the revision experiments use."""

    seed: int = 42
    n_splits: int = 5
    inner_splits: int = 3                   # for nested threshold tuning (item 6)
    cluster_threshold: float = 0.90         # k-mer Jaccard single-linkage (item 21)
    cluster_k: int = 6
    group_col: str = "cluster_group"

    # item 5 — alternative binary mappings of the four Geno2pheno categories
    ras_positive_default: tuple = ("polymorphism", "reduced_susceptibility", "resistant")
    ras_positive_strict: tuple = ("reduced_susceptibility", "resistant")
    ras_positive_strictest: tuple = ("resistant",)

    # item 6 — threshold tuning
    tune_threshold: bool = True
    threshold_metric: str = "youden"        # "youden" | "f1"
    threshold_grid: int = 201

    # item 10 — matched operating points
    spec_targets: tuple = (0.80, 0.90, 0.95)
    sens_targets: tuple = (0.80, 0.90, 0.95)

    # item 11 — interval estimation
    n_boot: int = 2000

    # item 3 — internal held-out split (cluster-disjoint)
    holdout_frac: float = 0.20

    # item 7 — the six pre-defined pLDDT positions the reviewer objects to
    af2_ras_positions: tuple = (28, 30, 31, 93, 282, 316)

    # item 9 — significance testing
    alpha: float = 0.05

    classifiers: tuple = ("logreg", "randomforest", "xgboost", "lightgbm", "mlp")

    def to_dict(self) -> dict:
        return asdict(self)


CFG = RevisionConfig()

# Positive class under the binary task. hcv_pipeline encodes it as integer 1.
POS = 1
NEG = 0


# --------------------------------------------------------------------------- #
# Item 12 — instrumentation. Must wrap the GPU work as it happens; peak VRAM
# cannot be reconstructed after the process exits.
# --------------------------------------------------------------------------- #
@dataclass
class ResourceRecord:
    stage: str
    n_items: int
    wall_s: float
    peak_vram_gb: float
    device: str
    extra: dict = field(default_factory=dict)

    @property
    def seconds_per_1000(self) -> float:
        return (self.wall_s / self.n_items * 1000.0) if self.n_items else float("nan")


class ResourceLog:
    """Append-only record of stage cost, persisted as JSON so it survives the
    session boundary between a GPU kernel and the CPU analysis kernel."""

    def __init__(self, path: str = "compute_cost.json"):
        self.path = path
        self.records: list[ResourceRecord] = []
        if os.path.exists(path):
            try:
                for r in json.load(open(path)):
                    self.records.append(ResourceRecord(**r))
            except Exception as exc:                      # corrupt log must not kill a run
                print(f"[ResourceLog] could not read {path}: {exc}")

    def add(self, rec: ResourceRecord) -> None:
        self.records.append(rec)
        self.save()

    def save(self) -> None:
        json.dump([asdict(r) for r in self.records], open(self.path, "w"), indent=2)

    def table(self) -> pd.DataFrame:
        """Item 12's table: one row per instrumented stage."""
        rows = []
        for r in self.records:
            rows.append({
                "stage": r.stage,
                "n_sequences": r.n_items,
                "wall_clock_s": round(r.wall_s, 1),
                "s_per_1000_seq": round(r.seconds_per_1000, 1),
                "peak_VRAM_GB": round(r.peak_vram_gb, 2),
                "device": r.device,
                **r.extra,
            })
        return pd.DataFrame(rows)


def _device_name() -> str:
    try:
        import torch
        if torch.cuda.is_available():
            return torch.cuda.get_device_name(0)
        return "cpu"
    except Exception:
        return "cpu"


def _reset_peak_vram() -> None:
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
    except Exception:
        pass


def _peak_vram_gb() -> float:
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            return torch.cuda.max_memory_allocated() / 1e9
    except Exception:
        pass
    return 0.0


class GpuSampler:
    """Peak GPU memory for work that happens in a SUBPROCESS.

    ``torch.cuda.max_memory_allocated`` only sees allocations in this process,
    so it reports 0.00 GB for ``colabfold_batch``. Item 12 asks for peak VRAM
    per model, so this polls ``nvidia-smi`` on a background thread instead.
    Returns 0.0 if nvidia-smi is unavailable, rather than failing the stage.
    """

    def __init__(self, interval: float = 5.0):
        self.interval = interval
        self.peak_gb = 0.0
        self._stop = None
        self._thread = None

    def _poll(self):
        import subprocess
        while not self._stop.is_set():
            try:
                out = subprocess.run(
                    ["nvidia-smi", "--query-gpu=memory.used",
                     "--format=csv,noheader,nounits"],
                    capture_output=True, text=True, timeout=10)
                for line in out.stdout.strip().splitlines():
                    mib = float(line.strip())
                    self.peak_gb = max(self.peak_gb, mib / 1024.0)
            except Exception:
                pass
            self._stop.wait(self.interval)

    def __enter__(self):
        import threading
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._poll, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, *exc):
        if self._stop is not None:
            self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=15)
        return False


@contextmanager
def Timer(log: ResourceLog, stage: str, n_items: int, vram_fn=None, **extra):
    """Time a stage and record peak VRAM. Use around every feature extractor and
    every model fit that should appear in the item-12 table.

    ``vram_fn`` overrides the in-process VRAM reading — pass a GpuSampler's
    ``peak_gb`` for work that runs in a subprocess.
    """
    _reset_peak_vram()
    t0 = time.perf_counter()
    try:
        yield
    finally:
        wall = time.perf_counter() - t0
        try:
            vram = float(vram_fn()) if vram_fn is not None else _peak_vram_gb()
        except Exception:
            vram = _peak_vram_gb()
        log.add(ResourceRecord(stage=stage, n_items=int(n_items), wall_s=wall,
                               peak_vram_gb=vram, device=_device_name(),
                               extra=extra))
        print(f"[cost] {stage}: {wall:.1f}s "
              f"({wall / max(n_items, 1) * 1000:.1f}s/1000) peak VRAM {vram:.2f} GB")


def plddt_summary(result_dir: str, pattern: str = "*rank_001*.pdb") -> dict:
    """Per-structure mean pLDDT across a ColabFold output directory.

    AlphaFold bands pLDDT as very low (<50), low (50–70), confident (70–90) and
    very high (>90); below 50 the prediction should not be read as a structure.
    Run this after folding: if the distribution sits in the very-low band the
    structural features are not carrying structural information, whatever the
    downstream AUROC says, and the methods must state it.
    """
    import glob as _glob
    out = {"result_dir": result_dir, "n_structures": 0}
    means = []
    for pdb in sorted(_glob.glob(os.path.join(result_dir, pattern))):
        vals = []
        try:
            with open(pdb) as fh:
                for line in fh:
                    if line.startswith("ATOM") and line[12:16].strip() == "CA":
                        vals.append(float(line[60:66]))
        except Exception:
            continue
        if vals:
            means.append(float(np.mean(vals)))
    if not means:
        print(f"[pLDDT] no structures matched {pattern} in {result_dir}")
        return out
    arr = np.array(means)
    bands = {"very_low_lt50": int((arr < 50).sum()),
             "low_50_70": int(((arr >= 50) & (arr < 70)).sum()),
             "confident_70_90": int(((arr >= 70) & (arr < 90)).sum()),
             "very_high_ge90": int((arr >= 90).sum())}
    out.update(n_structures=len(arr), mean=float(arr.mean()),
               median=float(np.median(arr)), min=float(arr.min()),
               max=float(arr.max()), bands=bands)
    print(f"[pLDDT] {len(arr)} structures | mean {arr.mean():.1f} "
          f"median {np.median(arr):.1f} range {arr.min():.1f}-{arr.max():.1f}")
    for k, v in bands.items():
        print(f"         {k:18s} {v:4d} ({100 * v / len(arr):.0f}%)")
    if bands["very_low_lt50"] / len(arr) > 0.5:
        print("  WARNING: most structures are below pLDDT 50, which AlphaFold "
              "treats as too unreliable to interpret. Features derived from "
              "these are not structural evidence — say so in the methods, and "
              "consider re-folding with an MSA.")
    return out


def carry_forward_cost(work_dir: str, patterns: Sequence[str] = (
        "/kaggle/input/**/compute_cost.json",)) -> ResourceLog:
    """Open the cost log for this session, seeding it from an earlier stage.

    Item 12 spans sessions: the GPU timings are recorded in the ``features`` and
    ``fold`` stages, the table is assembled in ``analysis``. A fresh Kaggle
    session starts with an empty working directory, so without this the GPU rows
    are silently lost and the item-12 table ends up holding CPU training times
    only. Records are merged on (stage, n_items) so re-running a stage updates
    its row instead of duplicating it.
    """
    path = os.path.join(work_dir, "compute_cost.json")
    log = ResourceLog(path)
    seen = {(r.stage, r.n_items) for r in log.records}
    import glob as _glob
    for pat in patterns:
        for prev in sorted(_glob.glob(pat, recursive=True)):
            if os.path.abspath(prev) == os.path.abspath(path):
                continue
            try:
                for raw in json.load(open(prev)):
                    rec = ResourceRecord(**raw)
                    if (rec.stage, rec.n_items) not in seen:
                        log.records.append(rec)
                        seen.add((rec.stage, rec.n_items))
                print(f"[cost] carried forward {prev}")
            except Exception as exc:
                print(f"[cost] could not read {prev}: {exc}")
    log.save()
    print(f"[cost] log holds {len(log.records)} record(s)")
    return log


# --------------------------------------------------------------------------- #
# Crash safety: atomic writes, resumable feature extraction, session budgets
# --------------------------------------------------------------------------- #
def atomic_write(path: str, write_fn) -> None:
    """Write via a temporary file and rename, so a kill mid-write cannot leave a
    half-written parquet or JSON that the next session will happily load."""
    tmp = f"{path}.tmp{os.getpid()}"
    try:
        write_fn(tmp)
        os.replace(tmp, path)
    finally:
        if os.path.exists(tmp):
            try:
                os.remove(tmp)
            except OSError:
                pass


def atomic_json(path: str, obj) -> None:
    atomic_write(path, lambda p: json.dump(obj, open(p, "w"), indent=2, default=str))


def atomic_parquet(path: str, frame: pd.DataFrame) -> None:
    atomic_write(path, lambda p: frame.to_parquet(p))


@dataclass
class SessionBudget:
    """Wall-clock guard. Kaggle kills a session at its limit with no warning and
    no chance to flush, so long extraction loops stop themselves early, save, and
    report what remains for the next run."""

    minutes: float = 9 * 60
    started: float = field(default_factory=time.perf_counter)

    @property
    def elapsed_min(self) -> float:
        return (time.perf_counter() - self.started) / 60.0

    @property
    def remaining_min(self) -> float:
        return self.minutes - self.elapsed_min

    def expired(self) -> bool:
        return self.remaining_min <= 0


class RunReport:
    """What happened in this session, written out whatever goes wrong.

    A Kaggle "Save Version" run saves nothing at all if a cell raises, so every
    substantial cell runs inside ``guard`` and records its outcome here instead
    of propagating. The report is the first thing to read after an unattended
    run: it says which steps ran, which failed and why, and what to attach to
    the next run.
    """

    def __init__(self, path: str):
        self.path = path
        self.steps: list[dict] = []

    def record(self, name: str, status: str, seconds: float = 0.0,
               error: str = "", note: str = "") -> None:
        self.steps.append({"step": name, "status": status,
                           "seconds": round(seconds, 1), "error": error, "note": note})
        self.save()

    def save(self) -> None:
        try:
            atomic_json(self.path, self.steps)
        except Exception:
            pass

    @property
    def failures(self) -> list[dict]:
        return [s for s in self.steps if s["status"] == "failed"]

    def table(self) -> pd.DataFrame:
        return pd.DataFrame(self.steps)


@contextmanager
def guard(name: str, report: RunReport | None = None, verbose: bool = True):
    """Run a notebook step without letting it kill an unattended commit.

    An exception is caught, printed with its traceback, recorded, and swallowed,
    so later cells still run and Kaggle still saves the output directory. Steps
    that depend on a failed one check their inputs with ``have`` and skip.
    """
    import traceback
    t0 = time.perf_counter()
    if verbose:
        print(f"===== {name}")
    try:
        yield
    except Exception as exc:
        secs = time.perf_counter() - t0
        tb = traceback.format_exc()
        print(f"!!! step '{name}' failed after {secs:.1f}s — continuing so the "
              f"session still saves its output\n{tb}")
        if report is not None:
            report.record(name, "failed", secs, error=f"{type(exc).__name__}: {exc}")
    else:
        secs = time.perf_counter() - t0
        if report is not None:
            report.record(name, "ok", secs)
        if verbose:
            print(f"----- {name}: ok ({secs:.1f}s)")


def _seq_key(seq: str) -> str:
    import hashlib
    return hashlib.sha1(str(seq).encode("utf-8")).hexdigest()


class FeatureCache:
    """Per-sequence feature cache keyed by sequence hash, flushed incrementally.

    The extractors in hcv_pipeline compute everything and save once at the end,
    so an out-of-memory error or a session timeout at 95% costs the whole run.
    This keeps a parquet keyed by sequence hash and flushes every ``flush_every``
    new sequences, so a re-run resumes from the last flush.
    """

    def __init__(self, path: str, columns: Sequence[str], flush_every: int = 32):
        self.path = path
        self.columns = list(columns)
        self.flush_every = flush_every
        self.data: dict[str, np.ndarray] = {}
        self.failed: set[str] = set()
        self._since_flush = 0
        self._load()

    @property
    def _fail_path(self) -> str:
        return os.path.splitext(self.path)[0] + "_failed.json"

    def _load(self) -> None:
        if os.path.exists(self.path):
            try:
                frame = pd.read_parquet(self.path)
                self.data = {k: row.to_numpy(dtype=np.float32)
                             for k, row in frame.iterrows()}
                print(f"[cache] resumed {len(self.data)} sequences from {self.path}")
            except Exception as exc:
                print(f"[cache] {self.path} unreadable ({exc}); starting fresh")
        if os.path.exists(self._fail_path):
            try:
                self.failed = set(json.load(open(self._fail_path)))
            except Exception:
                pass

    def has(self, seq: str) -> bool:
        return _seq_key(seq) in self.data

    def put(self, seq: str, vec: np.ndarray) -> None:
        self.data[_seq_key(seq)] = np.asarray(vec, dtype=np.float32)
        self._since_flush += 1
        if self._since_flush >= self.flush_every:
            self.flush()

    def mark_failed(self, seq: str) -> None:
        """Record a sequence that could not be processed. It is NOT zero-filled:
        a zero vector is indistinguishable from a real feature downstream, and
        the pipeline's own docstring warns that placeholder structures caused
        false results before."""
        self.failed.add(_seq_key(seq))

    def flush(self) -> None:
        if self.data:
            frame = pd.DataFrame.from_dict(self.data, orient="index",
                                           columns=self.columns).astype(np.float32)
            atomic_parquet(self.path, frame)
        if self.failed:
            atomic_json(self._fail_path, sorted(self.failed))
        self._since_flush = 0

    def matrix(self, seqs: Sequence[str], index) -> tuple[pd.DataFrame, list[int]]:
        """Assemble the ``N,D`` matrix for ``seqs``. Rows with no cached vector
        come back as NaN and their positions are returned, so the caller can
        report coverage rather than discover a silent zero row later."""
        rows, missing = [], []
        for i, s in enumerate(seqs):
            k = _seq_key(s)
            if k in self.data:
                rows.append(self.data[k])
            else:
                rows.append(np.full(len(self.columns), np.nan, dtype=np.float32))
                missing.append(i)
        X = pd.DataFrame(np.vstack(rows), columns=self.columns, index=index)
        return X.astype(np.float32), missing


def features_esm2_checkpointed(cohort: H.Cohort, model_name: str, cache_path: str,
                               max_len: int = 1022, batch_size: int = 8,
                               flush_every: int = 64,
                               budget: SessionBudget | None = None) -> tuple[pd.DataFrame, dict]:
    """ESM-2 mean-pooled embeddings, resumable.

    Identical arithmetic to ``H.features_esm2_embeddings`` — truncate to
    ``max_len``, attention-mask-weighted mean over the last hidden state — but
    flushed to disk as it goes and restartable after a crash or a timeout.

    Returns the ``N,D`` matrix and a status dict carrying coverage.
    """
    import torch
    from transformers import AutoTokenizer, AutoModel

    seqs_all = cohort.df["aa_seq"].tolist()
    uniq = list(dict.fromkeys(seqs_all))
    dim = 1280
    cache = FeatureCache(cache_path, [f"esm2_{j}" for j in range(dim)], flush_every)
    todo = [s for s in uniq if not cache.has(s)]
    print(f"[ESM-2] {len(uniq) - len(todo)} cached, {len(todo)} to embed")

    stopped_early = False
    if todo:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tok = AutoTokenizer.from_pretrained(model_name)
        model = AutoModel.from_pretrained(model_name).to(device).eval()
        try:
            from tqdm.auto import tqdm
            bar = tqdm(range(0, len(todo), batch_size), desc="ESM-2 embed")
        except Exception:
            bar = range(0, len(todo), batch_size)
        with torch.no_grad():
            for i in bar:
                if budget is not None and budget.expired():
                    stopped_early = True
                    print(f"[ESM-2] session budget reached with {len(todo) - i} "
                          f"sequences left; flushing and stopping cleanly")
                    break
                batch = [s[:max_len] for s in todo[i:i + batch_size]]
                try:
                    enc = tok(batch, return_tensors="pt", padding=True,
                              truncation=True, max_length=max_len + 2).to(device)
                    hs = model(**enc).last_hidden_state
                    mask = enc["attention_mask"].unsqueeze(-1)
                    emb = ((hs * mask).sum(1) / mask.sum(1)).cpu().numpy()
                    for s, e in zip(todo[i:i + batch_size], emb):
                        cache.put(s, e)
                except Exception as exc:                 # OOM on one batch only
                    H._free_gpu()
                    print(f"[ESM-2] batch at {i} failed ({str(exc)[:80]}); "
                          f"retrying one sequence at a time")
                    for s in todo[i:i + batch_size]:
                        try:
                            enc = tok([s[:max_len]], return_tensors="pt",
                                      truncation=True, max_length=max_len + 2).to(device)
                            hs = model(**enc).last_hidden_state
                            mask = enc["attention_mask"].unsqueeze(-1)
                            cache.put(s, ((hs * mask).sum(1) / mask.sum(1)).cpu().numpy()[0])
                        except Exception:
                            H._free_gpu()
                            cache.mark_failed(s)
        cache.flush()
        del model, tok
        H._free_gpu()

    X, missing = cache.matrix(seqs_all, cohort.df.index)
    status = {"source": "ESM2", "n_rows": len(seqs_all), "n_missing": len(missing),
              "coverage": 1.0 - len(missing) / max(len(seqs_all), 1),
              "n_failed_sequences": len(cache.failed), "stopped_early": stopped_early,
              "cache": cache_path}
    print(f"[ESM-2] coverage {status['coverage']:.1%}"
          f"{' — re-run this cell to continue' if stopped_early or missing else ''}")
    return X, status


def features_esmfold_checkpointed(cohort: H.Cohort, cache_path: str,
                                  model_name: str = "facebook/esmfold_v1",
                                  max_len: int = 300, chunk: int = 16,
                                  flush_every: int = 8,
                                  budget: SessionBudget | None = None
                                  ) -> tuple[pd.DataFrame, dict]:
    """ESMFold structural features, resumable and honest about failures.

    Same 1032-d feature vector as ``H.features_esmfold`` (1024 mean-pooled trunk
    ``s_s``, 3 pLDDT stats, 5 Ca-Ca distance stats), with two differences:

    * the cache is flushed every ``flush_every`` sequences, so an OOM or a
      session timeout costs at most that many folds rather than the whole run;
    * a sequence that fails to fold is recorded as failed and left as NaN.
      ``H.features_esmfold`` zero-fills it, which is indistinguishable
      downstream from a genuine all-zero structure.

    Falls back to CPU if the model will not load on the GPU.
    """
    import torch
    from transformers import AutoTokenizer, EsmForProteinFolding

    cols = ([f"esmf_ss_{i}" for i in range(1024)] +
            ["plddt_mean", "plddt_std", "plddt_min",
             "ca_mean", "ca_std", "ca_min", "ca_max", "ca_median"])
    seqs_all = cohort.df["aa_seq"].tolist()
    uniq = list(dict.fromkeys(seqs_all))
    cache = FeatureCache(cache_path, cols, flush_every)
    todo = [s for s in uniq if not cache.has(s) and _seq_key(s) not in cache.failed]
    print(f"[ESMFold] {len(uniq) - len(todo)} cached, {len(todo)} to fold")

    stopped_early = False
    if todo:
        tok = AutoTokenizer.from_pretrained(model_name)

        def load(dev):
            m = EsmForProteinFolding.from_pretrained(
                model_name, low_cpu_mem_usage=True).to(dev).eval()
            if dev == "cuda":
                try:
                    m.esm = m.esm.half()
                except Exception:
                    pass
            try:
                m.trunk.set_chunk_size(chunk)
            except Exception:
                pass
            return m

        device = "cuda" if torch.cuda.is_available() else "cpu"
        try:
            model = load(device)
        except Exception as exc:
            H._free_gpu()
            print(f"[ESMFold] GPU load failed ({str(exc)[:80]}) — falling back to CPU")
            device = "cpu"
            model = load(device)

        def featurise(out) -> np.ndarray:
            s_s = out.s_s[0].float().mean(0).cpu().numpy()
            pr = out.plddt[0].float().cpu().numpy()
            plddt = pr.mean(-1) if pr.ndim == 2 else pr
            ca = out.positions[-1, 0, :, 1, :].float().cpu().numpy()
            d = np.sqrt(((ca[:, None, :] - ca[None, :, :]) ** 2).sum(-1))
            iu = np.triu_indices(len(ca), k=1)
            dd = d[iu] if len(iu[0]) else np.array([0.0])
            return np.concatenate([
                s_s,
                [plddt.mean(), plddt.std(), plddt.min()],
                [dd.mean(), dd.std(), dd.min(), dd.max(), np.median(dd)],
            ]).astype(np.float32)

        try:
            from tqdm.auto import tqdm
            bar = tqdm(todo, desc=f"ESMFold[{device}]")
        except Exception:
            bar = todo
        with torch.no_grad():
            for n, s in enumerate(bar):
                if budget is not None and budget.expired():
                    stopped_early = True
                    print(f"[ESMFold] session budget reached with {len(todo) - n} "
                          f"sequences left; flushing and stopping cleanly")
                    break
                seq = (str(s)[:max_len]) or "M"
                try:
                    enc = tok([seq], return_tensors="pt", add_special_tokens=False,
                              padding=True, truncation=True, max_length=max_len)
                    cache.put(s, featurise(model(enc["input_ids"].to(device))))
                except Exception as exc:
                    H._free_gpu()
                    print(f"[ESMFold] fold failed for one sequence: {str(exc)[:80]}")
                    cache.mark_failed(s)
        cache.flush()
        del model, tok
        H._free_gpu()

    X, missing = cache.matrix(seqs_all, cohort.df.index)
    status = {"source": "ESMFold", "n_rows": len(seqs_all), "n_missing": len(missing),
              "coverage": 1.0 - len(missing) / max(len(seqs_all), 1),
              "n_failed_sequences": len(cache.failed), "stopped_early": stopped_early,
              "cache": cache_path}
    print(f"[ESMFold] coverage {status['coverage']:.1%}, "
          f"{len(cache.failed)} sequence(s) failed to fold"
          f"{' — re-run this cell to continue' if stopped_early or missing else ''}")
    return X, status


def finalise_source(X: pd.DataFrame, name: str, min_coverage: float = 0.80,
                    status: dict | None = None) -> pd.DataFrame | None:
    """Decide whether a partially extracted feature source may enter the grid.

    Rows that were never computed are NaN. Below ``min_coverage`` the source is
    refused outright rather than reported on whichever sequences happened to
    succeed, because that subset is not random — long or difficult sequences are
    exactly the ones that fail. Above it, the remaining NaNs are mean-imputed and
    the imputation is stated.
    """
    n_missing = int(X.isna().all(axis=1).sum())
    coverage = 1.0 - n_missing / max(len(X), 1)
    if coverage < min_coverage:
        print(f"[{name}] coverage {coverage:.0%} < {min_coverage:.0%}: EXCLUDED from the "
              f"grid. Reporting it on the {len(X) - n_missing} sequences that succeeded "
              f"would be a biased subset, not a smaller sample.")
        return None
    if n_missing:
        print(f"[{name}] {n_missing} row(s) mean-imputed ({coverage:.1%} genuine). "
              f"State this in the methods.")
    out = X.fillna(X.mean(numeric_only=True))
    if status is not None:
        status["entered_grid"] = True
        status["imputed_rows"] = n_missing
    return out


# --------------------------------------------------------------------------- #
# Dataset loading — tolerant of the is_augmented / eval_eligible column drift
# --------------------------------------------------------------------------- #
def load_dataset(path: str) -> pd.DataFrame:
    """Read the enriched CSV and guarantee an ``eval_eligible`` boolean column.

    Older exports carried ``is_augmented``; the current file carries
    ``eval_eligible`` directly. Exactly one of them must be present.
    """
    df = pd.read_csv(path, low_memory=False)
    if "eval_eligible" in df.columns:
        df["eval_eligible"] = df["eval_eligible"].astype(bool)
    elif "is_augmented" in df.columns:
        df["eval_eligible"] = ~df["is_augmented"].astype(bool)
    else:
        raise ValueError("CSV has neither 'eval_eligible' nor 'is_augmented'; "
                         "cannot tell real sequences from augmented ones.")
    return df


def cohort_counts(df: pd.DataFrame, ras_positive: Sequence[str]) -> dict:
    """Item 20: per-category counts on real sequences plus the binary collapse."""
    real = df[df["eval_eligible"]]
    per_cat = real["label"].value_counts().reindex(H.LABELS_4).fillna(0).astype(int)
    pos = int(real["label"].isin(ras_positive).sum())
    return {
        "n_rows_total": int(len(df)),
        "n_real": int(len(real)),
        "n_augmented": int(len(df) - len(real)),
        "n_accessions": int(real["accession"].nunique()) if "accession" in real else
                        int(real["group_id"].nunique()),
        "per_category": per_cat.to_dict(),
        "ras_positive_classes": list(ras_positive),
        "n_positive": pos,
        "n_negative": int(len(real) - pos),
    }


# --------------------------------------------------------------------------- #
# Item 6 — class balancing and nested threshold tuning
# --------------------------------------------------------------------------- #
def make_balanced_classifier(kind: str, y_train: np.ndarray, seed: int = 42):
    """Classifier with imbalance handling appropriate to its library.

    logreg / randomforest / lightgbm already take ``class_weight="balanced"`` in
    hcv_pipeline.make_classifier. The two that did not were xgboost (no
    ``scale_pos_weight``) and mlp (sklearn's MLPClassifier has no sample_weight
    in ``fit`` at all) — those are handled here and by ``_fit_balanced``.
    """
    n_pos = int((np.asarray(y_train) == POS).sum())
    n_neg = int((np.asarray(y_train) == NEG).sum())
    if kind in ("knn1", "knn5"):
        # Nearest-relative baseline: label a sequence by its closest neighbours
        # in composition space. For viral sequences this is the alignment-free
        # analogue of "what is the nearest known isolate", which is the implicit
        # baseline any sequence-similarity lookup provides. Cosine distance
        # suits frequency vectors; no class weighting applies to a vote.
        from sklearn.neighbors import KNeighborsClassifier
        k = 1 if kind == "knn1" else 5
        return KNeighborsClassifier(n_neighbors=k, metric="cosine",
                                    weights="distance" if k > 1 else "uniform")
    if kind == "xgboost":
        from xgboost import XGBClassifier
        spw = (n_neg / n_pos) if n_pos else 1.0
        return XGBClassifier(n_estimators=600, learning_rate=0.04, max_depth=6,
                             subsample=0.8, colsample_bytree=0.8,
                             eval_metric="logloss", tree_method="hist",
                             scale_pos_weight=spw, random_state=seed, n_jobs=-1)
    return H.make_classifier(kind, n_classes=2, seed=seed)


def _fit_balanced(kind: str, Xtr: np.ndarray, ytr: np.ndarray, seed: int = 42):
    """Fit with imbalance handling. MLP cannot be weighted, so its minority rows
    are oversampled with replacement to parity — the resampling arm of item 6."""
    clf = make_balanced_classifier(kind, ytr, seed)
    if kind == "mlp":
        rng = np.random.default_rng(seed)
        pos = np.where(ytr == POS)[0]
        neg = np.where(ytr == NEG)[0]
        if len(pos) and len(neg) and len(pos) < len(neg):
            extra = rng.choice(pos, len(neg) - len(pos), replace=True)
            idx = np.concatenate([np.arange(len(ytr)), extra])
            Xtr, ytr = Xtr[idx], ytr[idx]
    clf.fit(Xtr, ytr)
    return clf


def _threshold_score(y: np.ndarray, p: np.ndarray, thr: float, metric: str) -> float:
    pred = (p >= thr).astype(int)
    tp = int(((pred == POS) & (y == POS)).sum())
    fp = int(((pred == POS) & (y == NEG)).sum())
    fn = int(((pred == NEG) & (y == POS)).sum())
    tn = int(((pred == NEG) & (y == NEG)).sum())
    if metric == "f1":
        denom = 2 * tp + fp + fn
        return (2 * tp / denom) if denom else 0.0
    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    return sens + spec - 1.0                               # Youden's J


def pick_threshold(y: np.ndarray, p: np.ndarray, metric: str = "youden",
                   grid: int = 201) -> float:
    """Threshold maximising ``metric``. Called only on INNER-fold predictions so
    the outer validation fold never informs its own operating point."""
    if len(np.unique(y)) < 2:
        return 0.5
    cand = np.unique(np.concatenate([np.linspace(0.0, 1.0, grid), p]))
    scores = [_threshold_score(y, p, t, metric) for t in cand]
    return float(cand[int(np.argmax(scores))])


def _inner_threshold(cohort: H.Cohort, Xv: np.ndarray, tr_idx: np.ndarray,
                     clf_kind: str, cfg: RevisionConfig) -> float:
    """Nested threshold selection: inner StratifiedGroupKFold over the training
    rows only, scored on the real rows of each inner validation fold."""
    d = cohort.df
    y = d[cohort.label_col].to_numpy()
    groups = d[cfg.group_col].to_numpy()
    eligible = d["eval_eligible"].to_numpy()

    y_tr, g_tr = y[tr_idx], groups[tr_idx]
    if len(np.unique(y_tr)) < 2 or len(np.unique(g_tr)) < cfg.inner_splits:
        return 0.5

    inner = StratifiedGroupKFold(n_splits=cfg.inner_splits, shuffle=True,
                                 random_state=cfg.seed)
    ys, ps = [], []
    for itr, iva in inner.split(tr_idx, y_tr, g_tr):
        a, b = tr_idx[itr], tr_idx[iva]
        b = b[eligible[b]]
        if len(b) == 0 or len(np.unique(y[a])) < 2:
            continue
        sc = StandardScaler().fit(Xv[a])
        clf = _fit_balanced(clf_kind, sc.transform(Xv[a]), y[a], cfg.seed)
        proba = clf.predict_proba(sc.transform(Xv[b]))
        col = list(clf.classes_).index(POS) if POS in list(clf.classes_) else 1
        ys.append(y[b])
        ps.append(proba[:, col])
    if not ys:
        return 0.5
    return pick_threshold(np.concatenate(ys), np.concatenate(ps),
                          cfg.threshold_metric, cfg.threshold_grid)


# --------------------------------------------------------------------------- #
# Items 6 + 11 — the evaluation harness used by every revision experiment
# --------------------------------------------------------------------------- #
def evaluate_balanced(cohort: H.Cohort, X: pd.DataFrame, clf_kind: str = "logreg",
                      cfg: RevisionConfig = CFG, balance: bool = True,
                      tune_threshold: bool | None = None) -> dict:
    """Binary evaluation with class balancing, nested threshold tuning, per-fold
    positive counts and bootstrap CIs.

    Mirrors hcv_pipeline.evaluate_pipeline (same splits, same train-on-all /
    score-on-real invariant) and adds what items 6, 10 and 11 require.

    X: ``N,D`` aligned positionally to ``cohort.df``.
    """
    if cohort.task != "binary":
        raise ValueError("evaluate_balanced is binary-only; use H.evaluate_pipeline "
                         "for the 4-class task.")
    tune = cfg.tune_threshold if tune_threshold is None else tune_threshold

    d = cohort.df
    y = d[cohort.label_col].to_numpy().astype(int)
    groups = d[cfg.group_col].to_numpy()
    eligible = d["eval_eligible"].to_numpy()
    Xv = X.to_numpy(dtype=np.float32)

    sgkf = StratifiedGroupKFold(n_splits=cfg.n_splits, shuffle=True,
                                random_state=cfg.seed)
    n = len(d)
    oof_p = np.full(n, np.nan)                    # P(positive)
    oof_pred = np.full(n, -1, dtype=int)
    per_fold = []

    for k, (tr, va) in enumerate(sgkf.split(d, y, groups)):
        va_eval = va[eligible[va]]
        if len(va_eval) == 0:
            continue
        thr = _inner_threshold(cohort, Xv, tr, clf_kind, cfg) if tune else 0.5

        sc = StandardScaler().fit(Xv[tr])
        fit = _fit_balanced if balance else _fit_plain
        clf = fit(clf_kind, sc.transform(Xv[tr]), y[tr], cfg.seed)
        proba = clf.predict_proba(sc.transform(Xv[va_eval]))
        col = list(clf.classes_).index(POS) if POS in list(clf.classes_) else 1
        p = proba[:, col]

        oof_p[va_eval] = p
        oof_pred[va_eval] = (p >= thr).astype(int)

        yv = y[va_eval]
        m = _binary_metrics(yv, oof_pred[va_eval], p)
        m.update(fold=k, threshold=round(thr, 4),
                 n_val_real=int(len(va_eval)),
                 n_pos_val=int((yv == POS).sum()),       # item 11
                 n_neg_val=int((yv == NEG).sum()))
        per_fold.append(m)

    ev = ~np.isnan(oof_p)
    yv, pv, predv = y[ev], oof_p[ev], oof_pred[ev]
    agg = _binary_metrics(yv, predv, pv)
    agg["n_eval_real"] = int(ev.sum())
    agg["n_pos_total"] = int((yv == POS).sum())
    agg.update(bootstrap_metric_cis(yv, predv, pv, cfg))

    pf = pd.DataFrame(per_fold)
    metric_cols = [c for c in pf.columns
                   if c not in ("fold", "n_val_real", "n_pos_val", "n_neg_val", "threshold")]
    fold_summary = {c: (float(pf[c].mean()), float(pf[c].std())) for c in metric_cols} \
        if len(pf) else {}

    return {
        "clf": clf_kind, "balanced": balance, "tuned": tune,
        "metrics_oof": agg, "fold_mean_std": fold_summary, "per_fold": pf,
        "oof_true": yv, "oof_pred": predv, "oof_p": pv,
        "oof_mask": ev, "classes": [NEG, POS],
    }


def _fit_plain(kind: str, Xtr: np.ndarray, ytr: np.ndarray, seed: int = 42):
    """Unbalanced fit — the original behaviour, kept so item 6 can report a
    before/after comparison rather than only the improved number.

    Classifiers that hcv_pipeline does not define (the k-NN baselines) have no
    weighting to switch off, so the balanced constructor is used for them and
    the two settings coincide, as they should.
    """
    try:
        clf = H.make_classifier(kind, n_classes=2, seed=seed)
    except ValueError:
        clf = make_balanced_classifier(kind, ytr, seed)
    clf.fit(Xtr, ytr)
    return clf


def _binary_metrics(y: np.ndarray, pred: np.ndarray, p: np.ndarray) -> dict:
    from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                                 matthews_corrcoef, roc_auc_score, confusion_matrix)
    out = {}
    try:
        out["auroc"] = float(roc_auc_score(y, p))
    except Exception:
        out["auroc"] = float("nan")
    try:
        out["pr_auc"] = float(average_precision_score(y, p))
    except Exception:
        out["pr_auc"] = float("nan")
    out["accuracy"] = float(accuracy_score(y, pred))
    out["balanced_acc"] = float(balanced_accuracy_score(y, pred))
    out["f1"] = float(f1_score(y, pred, pos_label=POS, zero_division=0))
    out["mcc"] = float(matthews_corrcoef(y, pred)) if len(np.unique(y)) > 1 else float("nan")
    cm = confusion_matrix(y, pred, labels=[NEG, POS])
    tn, fp, fn, tp = cm.ravel()
    out["sensitivity"] = float(tp / (tp + fn)) if (tp + fn) else float("nan")
    out["specificity"] = float(tn / (tn + fp)) if (tn + fp) else float("nan")
    out["precision"] = float(tp / (tp + fp)) if (tp + fp) else float("nan")
    out["tp"], out["fp"], out["fn"], out["tn"] = int(tp), int(fp), int(fn), int(tn)
    return out


def bootstrap_metric_cis(y: np.ndarray, pred: np.ndarray, p: np.ndarray,
                         cfg: RevisionConfig = CFG) -> dict:
    """Item 11. Wilson intervals for the three proportions (exact-form, correct at
    small n) and a percentile bootstrap for the rank/threshold-free metrics."""
    out = {}
    tp = int(((pred == POS) & (y == POS)).sum())
    fn = int(((pred == NEG) & (y == POS)).sum())
    tn = int(((pred == NEG) & (y == NEG)).sum())
    fp = int(((pred == POS) & (y == NEG)).sum())
    out["accuracy_ci95"] = tuple(round(v, 4) for v in H.wilson_ci(tp + tn, len(y)))
    out["sensitivity_ci95"] = tuple(round(v, 4) for v in H.wilson_ci(tp, tp + fn))
    out["specificity_ci95"] = tuple(round(v, 4) for v in H.wilson_ci(tn, tn + fp))

    rng = np.random.default_rng(cfg.seed)
    keys = ("auroc", "pr_auc", "f1", "mcc", "balanced_acc")
    draws = {k: [] for k in keys}
    n = len(y)
    for _ in range(cfg.n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y[idx])) < 2:
            continue
        m = _binary_metrics(y[idx], pred[idx], p[idx])
        for k in keys:
            if not np.isnan(m[k]):
                draws[k].append(m[k])
    for k in keys:
        v = draws[k]
        out[f"{k}_ci95"] = (round(float(np.percentile(v, 2.5)), 4),
                            round(float(np.percentile(v, 97.5)), 4)) if v else (np.nan, np.nan)
    return out


# --------------------------------------------------------------------------- #
# Item 1 — RAM-position baseline
# --------------------------------------------------------------------------- #
RAM_POSITION_COLUMNS = (
    "ras_ns3_36", "ras_ns3_54", "ras_ns3_55", "ras_ns3_80", "ras_ns3_168",
    "ras_ns5a_24", "ras_ns5a_28", "ras_ns5a_30", "ras_ns5a_31", "ras_ns5a_93",
    "ras_ns5b_282", "ras_ns5b_316",
)


def ram_baseline_matrix(cohort: H.Cohort) -> pd.DataFrame:
    """Item 1: the twelve known RAM indicator columns, as a feature matrix.

    This deliberately uses columns that ``hcv_pipeline.FORBIDDEN_FEATURE_PREFIXES``
    blocks everywhere else, because the question the editor asked is exactly
    "what do the labelling rules alone achieve?". Geno2pheno derives the labels
    from these positions, so a near-perfect score is the expected and honest
    result — it is not evidence that the baseline is a good model, and it must be
    reported as the ceiling that the sequence models are measured against, never
    folded into the headline comparison.
    """
    d = cohort.df
    cols = [c for c in RAM_POSITION_COLUMNS if c in d.columns]
    missing = set(RAM_POSITION_COLUMNS) - set(cols)
    if missing:
        print(f"[item 1] RAM columns absent from the CSV: {sorted(missing)}")
    if not cols:
        raise ValueError("no RAM position columns found")
    return d[cols].astype(np.float32).fillna(0.0)


def mutation_count_matrix(cohort: H.Cohort) -> pd.DataFrame:
    """A weaker annotation baseline: how many mutations, not which ones.

    Sits between ``ram_baseline_matrix`` (the full labelling rule) and the
    sequence models. It uses the per-region mutation counts but not the identity
    of any position, so it answers a narrower question than item 1: is a
    sequence called resistant simply because it is divergent? Like item 1 it
    draws on annotation columns and is reported as a baseline, never as a model.
    """
    d = cohort.df
    cols = [c for c in ("ns3_n_mut", "ns5a_n_mut", "ns5b_n_mut",
                        "region_count", "has_ns3", "has_ns5a", "has_ns5b")
            if c in d.columns]
    if not cols:
        raise ValueError("no mutation-count columns found")
    return d[cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).astype(np.float32)


def aa_composition_matrix(cohort: H.Cohort, dipeptides: bool = True) -> pd.DataFrame:
    """Amino-acid composition of the translated sequence: 20 residue frequencies
    and, optionally, 400 dipeptide frequencies.

    A classical protein-sequence baseline that owes nothing to a foundation
    model and nothing to the annotation. Pure sequence, so it passes the leakage
    guard — the honest floor that ESM-2 and the structural models must clear.
    """
    AA = "ACDEFGHIKLMNPQRSTVWY"
    seqs = cohort.df["aa_seq"].astype(str)
    feats = {f"aa_{a}": np.zeros(len(seqs), dtype=np.float32) for a in AA}
    for i, s in enumerate(seqs):
        n = max(len(s), 1)
        for a in AA:
            feats[f"aa_{a}"][i] = s.count(a) / n
    X = pd.DataFrame(feats, index=cohort.df.index)

    if dipeptides:
        pairs = [a + b for a in AA for b in AA]
        idx = {p: j for j, p in enumerate(pairs)}
        M = np.zeros((len(seqs), len(pairs)), dtype=np.float32)
        for i, s in enumerate(seqs):
            tot = max(len(s) - 1, 1)
            for p in range(len(s) - 1):
                j = idx.get(s[p:p + 2])
                if j is not None:
                    M[i, j] += 1.0
            M[i] /= tot
        X = pd.concat([X, pd.DataFrame(M, columns=[f"aadi_{p}" for p in pairs],
                                       index=cohort.df.index)], axis=1)
    X = X.astype(np.float32)
    H._assert_no_leak(X.columns)
    return X


def trivial_baseline_metrics(cohort: H.Cohort, cfg: RevisionConfig = CFG) -> pd.DataFrame:
    """The floor every other number must be read against.

    ``majority`` always predicts the negative class; ``stratified`` samples the
    predicted class from the training prevalence; ``prevalence`` is the positive
    rate itself, which is the precision an uninformative ranker achieves and the
    baseline line on a PR curve. On a cohort that is 88% negative, an accuracy of
    0.88 is not a result, and this table is what makes that obvious in print.
    """
    d = cohort.df[cohort.df["eval_eligible"]] if "eval_eligible" in cohort.df else cohort.df
    y = d[cohort.label_col].to_numpy().astype(int)
    rng = np.random.default_rng(cfg.seed)
    rate = float(y.mean())

    rows = []
    for name, pred, proba in (
        ("majority_negative", np.zeros_like(y), np.zeros(len(y), dtype=float)),
        ("stratified_random", rng.binomial(1, rate, len(y)), rng.random(len(y))),
    ):
        m = _binary_metrics(y, pred, proba)
        m["baseline"] = name
        rows.append(m)
    out = pd.DataFrame(rows)
    keep = ["baseline", "accuracy", "balanced_acc", "sensitivity", "specificity",
            "f1", "mcc", "auroc", "pr_auc"]
    out = out[keep]
    out.loc[len(out)] = {"baseline": "positive_prevalence", "accuracy": np.nan,
                         "balanced_acc": np.nan, "sensitivity": np.nan,
                         "specificity": np.nan, "f1": np.nan, "mcc": np.nan,
                         "auroc": 0.5, "pr_auc": rate}
    return out


# --------------------------------------------------------------------------- #
# Item 2 — dataset-bias baseline
# --------------------------------------------------------------------------- #
def metadata_baseline_matrix(cohort: H.Cohort, top_k: int = 15) -> pd.DataFrame:
    """Item 2: metadata only — length, collection year, submitter, organisation,
    sequenced region. No nucleotide content of any kind.

    Submitter and organisation are high-cardinality, so they enter as (a) a
    frequency encoding and (b) one-hot indicators for the ``top_k`` most common
    values. Frequency encoding is computed over the whole cohort rather than per
    fold; it carries no label information, only how often a submitter appears.
    """
    d = cohort.df
    feats: dict[str, np.ndarray] = {}

    if "seq_len" in d:
        feats["meta_seq_len"] = d["seq_len"].astype(np.float32).to_numpy()
    if "Length" in d:
        feats["meta_length"] = pd.to_numeric(d["Length"], errors="coerce").fillna(0).to_numpy()

    if "Collection_Date" in d:
        year = pd.to_datetime(d["Collection_Date"], errors="coerce").dt.year
        if year.isna().all():                     # some exports hold bare years
            year = pd.to_numeric(d["Collection_Date"].astype(str).str[:4], errors="coerce")
        feats["meta_year"] = year.fillna(year.median() if year.notna().any() else 0).to_numpy()

    for c in ("has_ns3", "has_ns5a", "has_ns5b", "region_count"):
        if c in d:
            feats[f"meta_{c}"] = pd.to_numeric(d[c], errors="coerce").fillna(0).to_numpy()

    for c in ("Submitters", "Organization", "Nuc_Completeness", "covered_regions"):
        if c not in d:
            continue
        s = d[c].astype(str).fillna("NA")
        freq = s.map(s.value_counts()).astype(np.float32)
        feats[f"meta_{c}_freq"] = freq.to_numpy()
        for val in s.value_counts().head(top_k).index:
            key = f"meta_{c}_is_{str(val)[:24].replace(' ', '_')}"
            feats[key] = (s == val).astype(np.float32).to_numpy()

    X = pd.DataFrame(feats, index=d.index).astype(np.float32)
    H._assert_no_leak(X.columns)                  # metadata must not smuggle RAS columns
    return X


# --------------------------------------------------------------------------- #
# Item 5 — alternative binary mappings
# --------------------------------------------------------------------------- #
def stricter_cohorts(df: pd.DataFrame, cfg: RevisionConfig = CFG) -> dict:
    """Item 5: the default mapping plus two more conservative ones.

    ``strict``    = reduced_susceptibility + resistant  (22 positives)
    ``strictest`` = resistant only                      (9 positives, underpowered)

    Both are reported; ``strictest`` carries so few positives that its intervals
    will be very wide, and the text must say so rather than quoting a point
    estimate.
    """
    out = {}
    for name, mapping in (("default", cfg.ras_positive_default),
                          ("strict", cfg.ras_positive_strict),
                          ("strictest", cfg.ras_positive_strictest)):
        coh = H.build_common_cohort(df, task="binary", ras_positive=mapping)
        coh = H.add_identity_clusters(coh, threshold=cfg.cluster_threshold,
                                      k=cfg.cluster_k)
        coh = real_only(coh, cfg)
        out[name] = coh
        n_pos = int(coh.df[coh.label_col].sum())
        print(f"[item 5] {name:9s} {mapping} -> {len(coh.df)} real, {n_pos} positive")
    return out


def real_only(coh: H.Cohort, cfg: RevisionConfig = CFG) -> H.Cohort:
    """Drop augmented rows and re-cluster, matching the original notebook's
    ``cohR``. Used wherever an experiment evaluates on real sequences only."""
    d = coh.df[coh.df["eval_eligible"]].reset_index(drop=True)
    c = H.Cohort(df=d, task=coh.task, label_col=coh.label_col, classes=coh.classes,
                 ras_positive=coh.ras_positive, notes=dict(coh.notes))
    return H.add_identity_clusters(c, threshold=cfg.cluster_threshold, k=cfg.cluster_k)


# --------------------------------------------------------------------------- #
# Item 4 — per-protein cohorts
# --------------------------------------------------------------------------- #
def per_protein_cohorts(cohort: H.Cohort, cfg: RevisionConfig = CFG) -> dict:
    """Item 4: NS5A-covering and NS5B-covering subsets, re-clustered.

    Returns only subsets with both classes present and enough groups to split.
    Coverage on this cohort is heavily skewed towards NS5B, so the NS5A model is
    expected to be underpowered; report its intervals, not just its point
    estimates.
    """
    out = {}
    for prot, flag in (("NS5A", "has_ns5a"), ("NS5B", "has_ns5b"), ("NS3", "has_ns3")):
        if flag not in cohort.df:
            continue
        mask = cohort.df[flag].astype(bool)
        d = cohort.df[mask].reset_index(drop=True)
        if len(d) == 0:
            continue
        n_pos = int(d[cohort.label_col].sum())
        sub = H.Cohort(df=d, task=cohort.task, label_col=cohort.label_col,
                       classes=cohort.classes, ras_positive=cohort.ras_positive,
                       notes={**cohort.notes, "protein": prot})
        sub = H.add_identity_clusters(sub, threshold=cfg.cluster_threshold, k=cfg.cluster_k)
        n_groups = sub.df[cfg.group_col].nunique()
        ok = n_pos >= cfg.n_splits and n_groups >= cfg.n_splits
        print(f"[item 4] {prot}: n={len(d)} positives={n_pos} groups={n_groups}"
              f"{'' if ok else '  -> SKIPPED, too few to cross-validate'}")
        if ok:
            out[prot] = sub
    return out


def subset_features(X: pd.DataFrame, cohort: H.Cohort, sub: H.Cohort) -> pd.DataFrame:
    """Re-index a feature matrix built on ``cohort`` onto a subset cohort, by
    group_id. Both cohorts must hold real rows only (one row per accession)."""
    key = cohort.df["group_id"].to_numpy()
    lookup = pd.DataFrame(X.to_numpy(), index=key, columns=X.columns)
    out = lookup.reindex(sub.df["group_id"].to_numpy())
    out.index = sub.df.index
    return out.astype(np.float32)


# --------------------------------------------------------------------------- #
# Item 7 — AlphaFold2 ablation
# --------------------------------------------------------------------------- #
def ablate_colabfold_ras(X: pd.DataFrame, cfg: RevisionConfig = CFG) -> pd.DataFrame:
    """Item 7: drop the six per-position pLDDT features that encode prior
    knowledge ESM-2 never sees. Everything else (mean pLDDT, the 384-d single
    representation, length, region flags) is kept."""
    drop = [f"plddt_ras_{p}" for p in cfg.af2_ras_positions]
    present = [c for c in drop if c in X.columns]
    if not present:
        print("[item 7] no plddt_ras_* columns present; ablation is a no-op")
    return X.drop(columns=present)


# --------------------------------------------------------------------------- #
# Item 3 — cluster-disjoint held-out evaluation
# --------------------------------------------------------------------------- #
def holdout_split(cohort: H.Cohort, cfg: RevisionConfig = CFG) -> tuple[np.ndarray, np.ndarray]:
    """Item 3 (internal): split whole identity-clusters, stratified on the label,
    into train and held-out positional index arrays.

    An internal holdout is NOT an independent test set and must not be described
    as one. It controls optimism from repeated cross-validation on the same data;
    genuine external validation needs sequences from outside this GenBank pull,
    which is what ``evaluate_external`` is for.
    """
    d = cohort.df
    y = d[cohort.label_col].to_numpy().astype(int)
    groups = d[cfg.group_col].to_numpy()

    # label a cluster positive if it holds any positive sequence, then split
    # positive and negative clusters separately to keep the ratio stable
    gdf = pd.DataFrame({"g": groups, "y": y}).groupby("g")["y"].max()
    rng = np.random.default_rng(cfg.seed)
    test_groups: list = []
    for cls in (POS, NEG):
        gs = gdf[gdf == cls].index.to_numpy()
        rng.shuffle(gs)
        k = max(1, int(round(len(gs) * cfg.holdout_frac)))
        test_groups.extend(gs[:k].tolist())
    test_groups = set(test_groups)

    in_test = np.array([g in test_groups for g in groups])
    eligible = d["eval_eligible"].to_numpy()
    test_idx = np.where(in_test & eligible)[0]      # score real rows only
    train_idx = np.where(~in_test)[0]               # augmented rows of train clusters stay
    return train_idx, test_idx


def evaluate_holdout(cohort: H.Cohort, X: pd.DataFrame, clf_kind: str,
                     cfg: RevisionConfig = CFG, balance: bool = True) -> dict:
    """Fit once on the training clusters, score once on the held-out clusters.
    The threshold is chosen on the training part only."""
    tr, te = holdout_split(cohort, cfg)
    y = cohort.df[cohort.label_col].to_numpy().astype(int)
    Xv = X.to_numpy(dtype=np.float32)

    thr = _inner_threshold(cohort, Xv, tr, clf_kind, cfg) if cfg.tune_threshold else 0.5
    sc = StandardScaler().fit(Xv[tr])
    clf = (_fit_balanced if balance else _fit_plain)(clf_kind, sc.transform(Xv[tr]),
                                                     y[tr], cfg.seed)
    proba = clf.predict_proba(sc.transform(Xv[te]))
    col = list(clf.classes_).index(POS) if POS in list(clf.classes_) else 1
    p = proba[:, col]
    pred = (p >= thr).astype(int)

    m = _binary_metrics(y[te], pred, p)
    m.update(bootstrap_metric_cis(y[te], pred, p, cfg))
    m.update(n_train_rows=int(len(tr)), n_test_real=int(len(te)),
             n_test_pos=int((y[te] == POS).sum()), threshold=round(thr, 4),
             n_train_clusters=int(pd.unique(cohort.df[cfg.group_col].to_numpy()[tr]).size),
             n_test_clusters=int(pd.unique(cohort.df[cfg.group_col].to_numpy()[te]).size))
    return m


def evaluate_external(cohort: H.Cohort, X_train: pd.DataFrame,
                      ext_df: pd.DataFrame, feature_fn, clf_kind: str,
                      cfg: RevisionConfig = CFG) -> dict:
    """Item 3 (genuine): train on the full cohort, score on sequences from a
    separate pull. ``feature_fn(cohort)`` must rebuild the same feature source
    for the external cohort.

    Raises if any external accession also appears in the training cohort — a
    silent intersection here would invalidate the only external claim the paper
    can make.
    """
    ext = H.build_common_cohort(ext_df, task="binary",
                                ras_positive=cohort.ras_positive)
    shared = set(ext.df["group_id"]) & set(cohort.df["group_id"])
    if shared:
        raise ValueError(f"{len(shared)} accessions appear in both the training cohort "
                         f"and the external set, e.g. {sorted(shared)[:5]}")
    X_ext = feature_fn(ext)

    y = cohort.df[cohort.label_col].to_numpy().astype(int)
    Xv = X_train.to_numpy(dtype=np.float32)
    all_idx = np.arange(len(y))
    thr = _inner_threshold(cohort, Xv, all_idx, clf_kind, cfg) if cfg.tune_threshold else 0.5

    sc = StandardScaler().fit(Xv)
    clf = _fit_balanced(clf_kind, sc.transform(Xv), y, cfg.seed)
    proba = clf.predict_proba(sc.transform(X_ext.to_numpy(dtype=np.float32)))
    col = list(clf.classes_).index(POS) if POS in list(clf.classes_) else 1
    p = proba[:, col]
    pred = (p >= thr).astype(int)
    y_ext = ext.df[ext.label_col].to_numpy().astype(int)

    m = _binary_metrics(y_ext, pred, p)
    m.update(bootstrap_metric_cis(y_ext, pred, p, cfg))
    m.update(n_external=int(len(y_ext)), n_external_pos=int((y_ext == POS).sum()),
             threshold=round(thr, 4))
    return m


# --------------------------------------------------------------------------- #
# Item 3 — assembling a genuine external set
#
# Three steps, because the middle one cannot be automated: Geno2pheno[hcv] is a
# web form with no API, and it is the label source. Re-deriving labels by rule
# is not an option — see label_consistency_report() below, which shows the
# published labels are not a pure function of the recorded mutations.
#
#   A. fetch_external_genbank()   -> sequences GenBank has that the cohort lacks
#   B. you submit the FASTA to https://hcv.geno2pheno.org and download its CSV
#   C. ingest_geno2pheno()        -> a labelled CSV in this dataset's schema
# --------------------------------------------------------------------------- #
# Calibrated against the cohort: this query recovers all 361 of its accessions
# and finds ~1,500 more, which is the shape the original pull must have had.
# Deliberately does NOT filter on genotype or region in the query text —
# Geno2pheno assigns the subtype and reports which regions it could score, and
# narrowing beforehand on free text discards most of the set (a textual
# "genotype 4" filter recovers only 195 of the 361 known-G4 cohort sequences).
DEFAULT_GENBANK_QUERY = '("Hepacivirus C"[Organism]) AND Egypt[All Fields]'


def fetch_external_genbank(exclude_accessions: Iterable[str], email: str,
                           query: str = DEFAULT_GENBANK_QUERY,
                           mindate: str | None = None, maxdate: str | None = None,
                           retmax: int = 2000, api_key: str | None = None,
                           batch_size: int = 100, require_egypt: bool = True,
                           require_genotype4: bool = False,
                           limit: int | None = None) -> pd.DataFrame:
    """Step A: pull Egyptian HCV G4 records that are not already in the cohort.

    ``exclude_accessions`` is matched on the versionless accession, so AB470005
    and AB470005.1 are the same record. Set ``mindate``/``maxdate``
    (``YYYY/MM/DD``, publication date) to restrict to submissions after the
    original pull — that gives temporal independence, which is a stronger claim
    than merely disjoint accessions, and is what the paper's own conclusion
    promises.

    Returns a frame with the columns this dataset uses, minus the label and the
    per-region annotation, which only Geno2pheno can supply.
    """
    from Bio import Entrez, SeqIO

    Entrez.email = email
    if api_key:
        Entrez.api_key = api_key

    kw = {"db": "nuccore", "term": query, "retmax": retmax, "idtype": "acc"}
    if mindate:
        kw.update(mindate=mindate, maxdate=maxdate or "3000/01/01", datetype="pdat")
    handle = Entrez.esearch(**kw)
    found = Entrez.read(handle)["IdList"]
    handle.close()

    exclude = {str(a).split(".")[0] for a in exclude_accessions}
    todo = [a for a in found if a.split(".")[0] not in exclude]
    print(f"[item 3] GenBank returned {len(found)} records; {len(found) - len(todo)} "
          f"are already in the cohort; {len(todo)} are new")
    if limit and len(todo) > limit:
        todo = todo[:limit]
        print(f"[item 3] capped at {limit} — Geno2pheno is a web form, so the set you "
              f"submit has to be one you are willing to upload in one go")
    print(f"[item 3] fetching {len(todo)}")
    if not todo:
        return pd.DataFrame()

    rows = []
    for i in range(0, len(todo), batch_size):
        chunk = todo[i:i + batch_size]
        with Entrez.efetch(db="nuccore", id=",".join(chunk),
                           rettype="gb", retmode="text") as fh:
            for rec in SeqIO.parse(fh, "genbank"):
                src = next((f for f in rec.features if f.type == "source"), None)
                q = (lambda k: (src.qualifiers.get(k, [""])[0] if src else ""))
                country = q("country") or q("geo_loc_name")
                note = " ".join(src.qualifiers.get("note", [])) if src else ""
                genotype = q("genotype") or note
                refs = rec.annotations.get("references", [])
                rows.append({
                    "accession": rec.id.split(".")[0],
                    "nt_sequence": str(rec.seq).upper(),
                    "GenBank_Title": rec.description,
                    "Length": len(rec.seq),
                    "Country": country.split(":")[0].strip(),
                    "Geo_Location": country,
                    "Host": q("host"),
                    "Isolate": q("isolate") or q("strain"),
                    "Collection_Date": q("collection_date"),
                    "Genotype": genotype,
                    "Submitters": (refs[0].authors if refs else ""),
                    "Organization": (refs[0].journal if refs else ""),
                    "Publications": "; ".join(r.title for r in refs if r.title),
                })
        print(f"  fetched {min(i + batch_size, len(todo))}/{len(todo)}")

    out = pd.DataFrame(rows)
    if len(out) and require_egypt:
        # The /country qualifier is often absent even on records the query
        # matched on "Egypt", so fall back to the title and isolate name rather
        # than discarding most of the set. Both counts are printed so the
        # provenance of the external set stays visible.
        explicit = out["Geo_Location"].str.contains("Egypt", case=False, na=False)
        fallback = (out["GenBank_Title"].fillna("") + " " + out["Isolate"].fillna("")
                    ).str.contains("Egypt", case=False, na=False)
        keep = explicit | fallback
        print(f"  Egypt: {int(explicit.sum())} by country qualifier, "
              f"{int((fallback & ~explicit).sum())} by title/isolate only, "
              f"{int((~keep).sum())} dropped")
        out = out[keep]
    if len(out) and require_genotype4:
        text = (out["Genotype"].fillna("") + " " + out["GenBank_Title"].fillna(""))
        keep = text.str.contains(r"\b4[a-z]?\b|genotype\s*4|subtype\s*4", case=False,
                                 regex=True, na=False)
        print(f"  {int((~keep).sum())} dropped: genotype 4 not evident in the record")
        out = out[keep]
    out = out.reset_index(drop=True)
    print(f"[item 3] {len(out)} external candidate sequences")
    return out


def assign_cf_names(cohort: H.Cohort, max_len: int = 400) -> pd.Series:
    """Map every accession to the accession whose structure represents it.

    40 of the 361 cohort sequences are exact duplicates of another, and
    identical sequences fold to identical structures. Folding each one again
    buys nothing and costs an MSA lookup, which is the expensive part of an
    MSA-mode run. Each distinct truncated sequence is folded once under the
    first accession that carries it, in sorted order so the choice is
    deterministic and both the folding notebook and the analysis notebook
    derive the same mapping without exchanging a file.

    This is not sampling: every accession still receives features, computed
    from the structure of its own sequence.
    """
    d = cohort.df.drop_duplicates("group_id")[["group_id", "aa_seq"]].copy()
    d["trunc"] = d["aa_seq"].astype(str).str[:max_len]
    rep = (d.sort_values("group_id").groupby("trunc")["group_id"].first())
    mapping = d.set_index("group_id")["trunc"].map(rep)
    full = cohort.df["group_id"].map(mapping)
    n_uniq = int(rep.nunique())
    print(f"[colabfold] {len(d)} accessions -> {n_uniq} distinct sequences to fold "
          f"({len(d) - n_uniq} are exact duplicates at max_len={max_len})")
    return full.astype(str)


def write_geno2pheno_fasta(ext: pd.DataFrame, path: str, max_len: int | None = None) -> str:
    """Write the FASTA to upload at https://hcv.geno2pheno.org.

    Headers are the bare accession so the returned CSV can be joined back. The
    site asks for plain or FASTA sequence and warns against patient identifiers;
    accessions are public record identifiers, so they are safe to use.
    """
    n = 0
    with open(path, "w") as fh:
        for _, r in ext.iterrows():
            seq = str(r["nt_sequence"])[:max_len] if max_len else str(r["nt_sequence"])
            if len(seq) < 100:
                continue
            fh.write(f">{r['accession']}\n{seq}\n")
            n += 1
    print(f"wrote {n} sequences to {path}\n"
          f"  Submit it at https://hcv.geno2pheno.org, download the CSV result,\n"
          f"  then pass that file to ingest_geno2pheno().")
    return path


def geno2pheno_version(url: str = "https://hcv.geno2pheno.org/", timeout: int = 20) -> dict:
    """Item 22: read the tool version off the service page.

    The page shows it in the heading, e.g. "Geno2pheno [hcv] 0.92". This records
    the version *observed now*, which is the version that labels any external
    set you submit today. It is NOT evidence of the version used for the
    original 2026 labelling — that must come from the job records, and the
    manifest keeps the two apart rather than conflating them.
    """
    import re
    import urllib.request
    from datetime import date
    out = {"url": url, "observed_on": str(date.today()), "version": None}
    try:
        with urllib.request.urlopen(url, timeout=timeout) as fh:
            html = fh.read().decode("utf-8", "replace")
        m = re.search(r"Geno2pheno\s*\[?\s*hcv\s*\]?\s*([0-9]+\.[0-9]+)", html, re.I)
        out["version"] = m.group(1) if m else None
        if not m:
            out["note"] = "version string not found in the page; check it by hand"
    except Exception as exc:
        out["error"] = f"{type(exc).__name__}: {exc}"
        print(f"[item 22] could not read the Geno2pheno version: {exc}")
    return out


# Geno2pheno's CSV column names are not documented on its page, and the service
# has no API. These are the candidates seen in this project's own export; the
# ingest prints the real header and refuses to guess when nothing matches.
G2P_COLUMN_CANDIDATES = {
    "accession": ("accession", "id", "identifier", "name", "sequence", "query"),
    "ns3_mutations": ("ns3_mutations", "ns3 mutations", "ns3"),
    "ns5a_mutations": ("ns5a_mutations", "ns5a mutations", "ns5a"),
    "ns5b_mutations": ("ns5b_mutations", "ns5b mutations", "ns5b"),
    "ns3_scored": ("ns3_scored", "ns3 scored", "ns3 ras"),
    "ns5a_scored": ("ns5a_scored", "ns5a scored", "ns5a ras"),
    "ns5b_scored": ("ns5b_scored", "ns5b scored", "ns5b ras"),
    "ns3_genotype": ("ns3_genotype", "ns3 subtype"),
    "ns5a_genotype": ("ns5a_genotype", "ns5a subtype"),
    "ns5b_genotype": ("ns5b_genotype", "ns5b subtype"),
    "label": ("label", "prediction", "resistance", "phenotype", "call", "result"),
}

LABEL_SYNONYMS = {
    "susceptible": "susceptible",
    "sensitive": "susceptible",
    "polymorphism": "polymorphism",
    "polymorphic": "polymorphism",
    "reduced_susceptibility": "reduced_susceptibility",
    "reduced susceptibility": "reduced_susceptibility",
    "intermediate": "reduced_susceptibility",
    "resistant": "resistant",
    "resistance": "resistant",
}


def ingest_geno2pheno(g2p_path: str, ext: pd.DataFrame,
                      column_map: dict | None = None,
                      sep: str | None = None) -> pd.DataFrame:
    """Step C: join a Geno2pheno CSV back onto the fetched sequences.

    Produces a frame in the same schema as ``hcv.csv`` so ``evaluate_external``
    can consume it unchanged. Because the service's column names are not
    documented, this prints the header it actually found and maps by candidate
    name; pass ``column_map`` to override. It raises rather than guess when it
    cannot find an accession column or a label column — a silently mis-joined
    external set would invalidate the only external claim the paper can make.
    """
    g2p = pd.read_csv(g2p_path, sep=sep, engine="python")
    print(f"[item 3] Geno2pheno CSV: {g2p.shape[0]} rows, columns = {list(g2p.columns)}")

    lower = {str(c).strip().lower(): c for c in g2p.columns}
    resolved = dict(column_map or {})
    for target, cands in G2P_COLUMN_CANDIDATES.items():
        if target in resolved:
            continue
        for cand in cands:
            if cand in lower:
                resolved[target] = lower[cand]
                break
    print(f"[item 3] resolved columns: {resolved}")

    for required in ("accession", "label"):
        if required not in resolved:
            raise ValueError(
                f"could not find a '{required}' column in {list(g2p.columns)}. "
                f"Pass column_map={{'{required}': '<the column>'}} after looking at "
                f"the file — this is not something to guess at.")

    out = pd.DataFrame({"accession": g2p[resolved["accession"]].astype(str)
                                      .str.strip().str.split().str[0].str.split(".").str[0]})
    for target, col in resolved.items():
        if target != "accession":
            out[target] = g2p[col]

    raw = out["label"].astype(str).str.strip().str.lower()
    out["label"] = raw.map(lambda v: LABEL_SYNONYMS.get(v, LABEL_SYNONYMS.get(
        v.replace("-", " ").replace("_", " "), None)))
    unmapped = sorted(set(raw[out["label"].isna()]))
    if unmapped:
        raise ValueError(f"unrecognised Geno2pheno label values: {unmapped}. "
                         f"Expected one of {sorted(set(LABEL_SYNONYMS.values()))}; "
                         f"extend LABEL_SYNONYMS once you know what they mean.")

    merged = ext.merge(out, on="accession", how="inner", suffixes=("", "_g2p"))
    missing = set(ext["accession"]) - set(merged["accession"])
    if missing:
        print(f"[item 3] {len(missing)} fetched sequences have no Geno2pheno result "
              f"(not covering NS3/NS5A/NS5B well enough, most likely); they are "
              f"excluded from the external set, not imputed")

    # the columns the pipeline needs that Geno2pheno implies
    for reg in ("ns3", "ns5a", "ns5b"):
        col = f"{reg}_scored"
        if col not in merged:
            merged[col] = ""
        # read_csv turns an empty field into NaN, which astype(str) renders as
        # the string "nan" — that would mark an uncovered region as covered
        cleaned = (merged[col].fillna("").astype(str).str.strip()
                   .replace({"not available": "", "nan": "", "NaN": "", "-": ""}))
        merged[col] = cleaned
        merged[f"has_{reg}"] = cleaned.ne("").astype(int)
    merged["covered_regions"] = merged.apply(
        lambda r: ",".join(x.upper() for x in ("ns5b", "ns5a", "ns3") if r[f"has_{x}"]), axis=1)
    merged["region_count"] = merged[["has_ns3", "has_ns5a", "has_ns5b"]].sum(axis=1)
    merged["group_id"] = merged["accession"]
    merged["eval_eligible"] = True
    merged["seq_len"] = merged["nt_sequence"].str.len()

    # Genotype 4 is decided here, from Geno2pheno's per-region subtype call,
    # exactly as it was for the cohort — not from free text in the GenBank record.
    gcols = [c for c in ("ns3_genotype", "ns5a_genotype", "ns5b_genotype") if c in merged]
    if gcols:
        g4 = merged[gcols].fillna("").astype(str).apply(
            lambda r: any(v.strip().startswith("4") for v in r), axis=1)
        print(f"[item 3] genotype 4 by Geno2pheno subtype call: {int(g4.sum())} kept, "
              f"{int((~g4).sum())} dropped")
        merged = merged[g4]
    else:
        print("[item 3] no per-region genotype columns in the Geno2pheno output — "
              "genotype 4 NOT verified. Check this before using the set.")

    counts = merged["label"].value_counts().to_dict()
    print(f"[item 3] external set: {len(merged)} labelled sequences, {counts}")
    if len(merged) and min(counts.values(), default=0) < 5:
        print("[item 3] warning: a class with fewer than 5 members. An external "
              "AUROC on this will have a very wide interval — report it as such.")
    return merged.reset_index(drop=True)


def label_consistency_report(df: pd.DataFrame) -> pd.DataFrame:
    """Is the published label a function of the recorded RAS mutations?

    Mostly, but not entirely — and the exceptions matter, because they are the
    reason an external set has to be labelled by Geno2pheno rather than by a
    re-implemented rule. Returns every mutation set that maps to more than one
    label. Report these; do not correct them.
    """
    real = df[df["eval_eligible"]] if "eval_eligible" in df else df

    def mutset(row):
        out = []
        for reg in ("ns3", "ns5a", "ns5b"):
            raw = row.get(f"{reg}_scored", "")
            if pd.isna(raw):
                continue
            for m in str(raw).split(","):
                m = m.strip()
                if m and m.lower() not in ("not available", "nan", "-"):
                    out.append(f"{reg}:{m}")
        return ", ".join(sorted(out)) or "(none)"

    tmp = pd.DataFrame({"mutations": real.apply(mutset, axis=1),
                        "label": real["label"].to_numpy()})
    grouped = tmp.groupby("mutations")["label"].agg(["nunique", "count",
                                                     lambda s: dict(s.value_counts())])
    grouped.columns = ["n_distinct_labels", "n_sequences", "label_counts"]
    amb = grouped[grouped["n_distinct_labels"] > 1].reset_index()
    if len(amb):
        print(f"[data note] {len(amb)} mutation set(s) carry more than one label across "
              f"{int(amb['n_sequences'].sum())} sequences. The label is therefore not a "
              f"pure function of the recorded mutations, so external sequences must be "
              f"labelled by Geno2pheno itself rather than by a re-implemented rule.")
    return amb


# --------------------------------------------------------------------------- #
# Item 8 — leakage and overlap audit
# --------------------------------------------------------------------------- #
def overlap_report(df: pd.DataFrame, cohort: H.Cohort,
                   cfg: RevisionConfig = CFG) -> dict:
    """Item 8. Two separable questions:

    (a) Do folds share groups or near-identical sequences?  ``H.leakage_audit``
        answers this for both grouping levels.
    (b) What is the relationship between the pretraining corpus and the
        evaluation set?  On this dataset the unlabelled corpus consists of
        perturbed variants generated from the same accessions that are
        evaluated, so corpus and evaluation set are not disjoint by
        construction. The control is that splits are grouped by accession and by
        identity cluster, so a variant of a validation sequence can never sit in
        the training fold. This function measures that rather than asserting it.
    """
    audit_acc = H.leakage_audit(cohort, n_splits=cfg.n_splits, seed=cfg.seed,
                                group_col="group_id")
    audit_clu = H.leakage_audit(cohort, n_splits=cfg.n_splits, seed=cfg.seed,
                                group_col="cluster_group")

    real = df[df["eval_eligible"]]
    aug = df[~df["eval_eligible"]]
    acc_real = set(real["accession"]) if "accession" in df else set(real["group_id"])
    acc_aug = set(aug["accession"]) if "accession" in df else set(aug["group_id"])

    exact = 0
    if "nt_sequence" in df.columns and len(aug):
        exact = int(aug["nt_sequence"].isin(set(real["nt_sequence"])).sum())

    return {
        "fold_audit_accession": audit_acc,
        "fold_audit_cluster": audit_clu,
        "n_real": int(len(real)),
        "n_augmented": int(len(aug)),
        "augmented_parent_accessions": len(acc_aug),
        "parents_also_in_eval_set": len(acc_aug & acc_real),
        "augmented_rows_identical_to_a_real_sequence": exact,
        "n_identity_clusters": int(cohort.df[cfg.group_col].nunique()),   # item 21
        "n_accessions": int(cohort.df["group_id"].nunique()),             # item 21
        "cluster_threshold": cfg.cluster_threshold,
        "cluster_k": cfg.cluster_k,
    }


# --------------------------------------------------------------------------- #
# Item 9 — significance testing
# --------------------------------------------------------------------------- #
def significance_tests(store: dict, metric: str = "auroc",
                       reference: tuple | None = None,
                       cfg: RevisionConfig = CFG) -> pd.DataFrame:
    """Item 9: paired comparison of every configuration against ``reference``
    across the ``n_splits`` folds.

    With five folds the Wilcoxon signed-rank test cannot produce a two-sided p
    below 0.0625 whatever the effect size, so an "n.s." here is weak evidence of
    no difference. The exact p and the fold count are both returned so the text
    can say that plainly. Holm correction is applied across the comparisons.
    """
    from scipy.stats import wilcoxon, ttest_rel

    keys = list(store.keys())
    if reference is None:
        reference = next((k for k in keys if k[0] == "SeqKmer3"), keys[0])
    if reference not in store:
        raise ValueError(f"reference {reference} not in store")

    def folds(k):
        pf = store[k]["per_fold"]
        return pf.sort_values("fold")[metric].to_numpy(dtype=float)

    ref = folds(reference)
    rows = []
    for k in keys:
        if k == reference:
            continue
        cur = folds(k)
        n = min(len(ref), len(cur))
        a, b = ref[:n], cur[:n]
        ok = ~(np.isnan(a) | np.isnan(b))
        a, b = a[ok], b[ok]
        if len(a) < 2 or np.allclose(a, b):
            p_w = p_t = float("nan")
        else:
            try:
                p_w = float(wilcoxon(b, a).pvalue)
            except Exception:
                p_w = float("nan")
            p_t = float(ttest_rel(b, a).pvalue)
        rows.append({
            "features": k[0], "classifier": k[1],
            "reference": f"{reference[0]}+{reference[1]}",
            "metric": metric,
            "mean_diff": float(np.mean(b - a)) if len(a) else float("nan"),
            "n_folds": int(len(a)),
            "p_wilcoxon": p_w, "p_ttest_paired": p_t,
        })
    cols = ["features", "classifier", "reference", "metric", "mean_diff", "n_folds",
            "p_wilcoxon", "p_ttest_paired"]
    out = pd.DataFrame(rows, columns=cols)
    if out.empty:
        print(f"[item 9] nothing to compare against {reference} — the store holds "
              f"only that configuration")
    for col in ("p_wilcoxon", "p_ttest_paired"):
        out[f"{col}_holm"] = _holm(out[col].to_numpy())
    # exact two-sided floor for the signed-rank test: 2 / 2**n_folds
    out["min_attainable_p_wilcoxon"] = 2.0 ** (1 - cfg.n_splits) if cfg.n_splits > 1 else np.nan
    return out.sort_values("mean_diff", ascending=False).reset_index(drop=True)


def _holm(p: np.ndarray) -> np.ndarray:
    """Holm-Bonferroni step-down adjustment; NaNs pass through."""
    p = np.asarray(p, dtype=float)
    out = np.full_like(p, np.nan)
    idx = np.where(~np.isnan(p))[0]
    if len(idx) == 0:
        return out
    order = idx[np.argsort(p[idx])]
    m = len(order)
    prev = 0.0
    for rank, i in enumerate(order):
        val = min(1.0, (m - rank) * p[i])
        prev = max(prev, val)
        out[i] = prev
    return out


# --------------------------------------------------------------------------- #
# Item 10 — matched operating points and precision-recall curves
# --------------------------------------------------------------------------- #
def matched_operating_points(y: np.ndarray, p: np.ndarray,
                             cfg: RevisionConfig = CFG) -> dict:
    """Item 10: sensitivity at fixed specificity and specificity at fixed
    sensitivity, read off the ROC curve. Comparing models at a matched operating
    point removes the threshold choice from the comparison."""
    out: dict[str, float] = {}
    if len(np.unique(y)) < 2:
        return out
    fpr, tpr, thr = roc_curve(y, p)
    spec = 1.0 - fpr
    for s in cfg.spec_targets:
        ok = spec >= s
        out[f"sens_at_spec{int(s * 100)}"] = float(tpr[ok].max()) if ok.any() else float("nan")
    for s in cfg.sens_targets:
        ok = tpr >= s
        out[f"spec_at_sens{int(s * 100)}"] = float(spec[ok].max()) if ok.any() else float("nan")
    return out


def pr_curve_points(y: np.ndarray, p: np.ndarray) -> pd.DataFrame:
    """Item 10: precision-recall curve for one configuration, plus the baseline
    precision (the positive rate) that an uninformative model would achieve."""
    prec, rec, _ = precision_recall_curve(y, p)
    return pd.DataFrame({"recall": rec, "precision": prec,
                         "baseline": float(np.mean(y))})


def pr_curve_table(store: dict, max_points: int = 400) -> pd.DataFrame:
    """Item 10: precision-recall curves for **every** configuration, long-form.

    The figure in the notebook plots a readable subset; this is the full data
    behind it, so the manuscript figure can be drawn without re-running the
    grid. Curves are thinned to ``max_points`` by even subsampling — the shape
    is preserved and the CSV stays small.
    """
    frames = []
    for (src, clf), r in store.items():
        pr = pr_curve_points(r["oof_true"], r["oof_p"])
        if len(pr) > max_points:
            idx = np.linspace(0, len(pr) - 1, max_points).astype(int)
            pr = pr.iloc[idx]
        pr = pr.copy()
        pr.insert(0, "classifier", clf)
        pr.insert(0, "features", src)
        pr["pr_auc"] = r["metrics_oof"]["pr_auc"]
        frames.append(pr)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


# --------------------------------------------------------------------------- #
# Item 19 — state explicitly which features entered the classifiers
# --------------------------------------------------------------------------- #
FEATURE_FAMILIES = {
    "esm2_": "ESM-2 embedding dimension (latent)",
    "esmf_ss_": "ESMFold trunk representation (latent)",
    "single_": "AlphaFold2 single representation (latent)",
    "plddt_ras_": "AlphaFold2 pLDDT at a pre-defined RAS position (named)",
    "plddt_": "AlphaFold2/ESMFold pLDDT summary (named)",
    "ca_": "ESMFold Ca-Ca distance statistic (named)",
    "kmer": "nucleotide k-mer frequency (named)",
    "di_": "dinucleotide frequency (named)",
    "aadi_": "amino-acid dipeptide frequency (named)",
    "aa_": "amino-acid composition (named)",
    "meta_": "record metadata (named)",
    "ras_ns": "known RAM position indicator (named)",
    "ns3_n_": "per-region mutation count (named)",
    "ns5a_n_": "per-region mutation count (named)",
    "ns5b_n_": "per-region mutation count (named)",
    "region_count": "region coverage (named)",
    "has_ns": "region coverage (named)",
    "seq_len": "sequence length (named)",
    "a_frac": "nucleotide composition (named)",
    "c_frac": "nucleotide composition (named)",
    "g_frac": "nucleotide composition (named)",
    "t_frac": "nucleotide composition (named)",
    "n_frac": "nucleotide composition (named)",
    "ambig_frac": "nucleotide composition (named)",
}


def feature_inventory(sources: dict) -> pd.DataFrame:
    """Which features entered each classifier, by family.

    Reviewer 2 (item 19) found the methodology text and Figure 4 contradictory:
    the text describes label-independent global structural metrics, the figure
    shows per-position pLDDT at six pre-defined sites. This table settles it
    from the matrices actually used, rather than from recollection.
    """
    rows = []
    for name, X in sources.items():
        cols = list(X.columns)
        counted, fam_counts = set(), {}
        for prefix, label in FEATURE_FAMILIES.items():
            hit = [c for c in cols if c.startswith(prefix) and c not in counted]
            if hit:
                # several prefixes share a label (the three per-region mutation
                # counts, the nucleotide fractions), so accumulate
                fam_counts[label] = fam_counts.get(label, 0) + len(hit)
                counted.update(hit)
        other = [c for c in cols if c not in counted]
        if other:
            fam_counts["other / ungrouped"] = len(other)
        ras_plddt = [c for c in cols if c.startswith("plddt_ras_")]
        rows.append({
            "source": name,
            "n_features": len(cols),
            "has_predefined_RAS_pLDDT": bool(ras_plddt),
            "RAS_pLDDT_columns": ", ".join(sorted(ras_plddt)) or "-",
            "families": "; ".join(f"{k} x{v}" for k, v in sorted(fam_counts.items())),
            "example_columns": ", ".join(cols[:4]) + (" ..." if len(cols) > 4 else ""),
        })
    return pd.DataFrame(rows)


def operating_point_table(store: dict, cfg: RevisionConfig = CFG) -> pd.DataFrame:
    """Matched operating points for every configuration in a store."""
    rows = []
    for (src, clf), r in store.items():
        row = {"features": src, "classifier": clf}
        row.update(matched_operating_points(r["oof_true"], r["oof_p"], cfg))
        row["pr_auc"] = r["metrics_oof"]["pr_auc"]
        row["positive_rate"] = float(np.mean(r["oof_true"]))
        rows.append(row)
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------- #
# Grid driver + manuscript tables
# --------------------------------------------------------------------------- #
def available_classifiers(requested: Iterable[str]) -> list[str]:
    """Drop classifiers whose library is not installed, rather than failing the
    whole grid on an import error partway through."""
    ok = []
    probe = np.array([NEG, POS])
    for kind in requested:
        try:
            make_balanced_classifier(kind, probe, seed=0)
            ok.append(kind)
        except Exception as exc:
            print(f"[grid] {kind} unavailable, skipping: {str(exc)[:100]}")
    return ok


def run_revision_grid(cohort: H.Cohort, sources: dict, classifiers: Iterable[str] | None = None,
                      cfg: RevisionConfig = CFG, balance: bool = True,
                      tune_threshold: bool | None = None,
                      log: ResourceLog | None = None,
                      cache_dir: str | None = None, tag: str = "") -> tuple[pd.DataFrame, dict]:
    """Every (feature source x classifier) pair under ``evaluate_balanced``.

    Returns a tidy table carrying the point estimate and the 95% interval for
    each metric (item 11) and the raw store for the post-hoc items.

    With ``cache_dir`` set, each configuration's result is pickled as it
    completes and reloaded on a re-run. The full grid is 70 fits plus bootstrap
    resampling; without this, one failure late in the sweep costs the whole
    sweep. ``tag`` separates caches for different cohorts or settings.
    """
    import pickle

    classifiers = available_classifiers(list(classifiers or cfg.classifiers))
    if cache_dir:
        os.makedirs(cache_dir, exist_ok=True)
    rows, store = [], {}

    def cache_file(sname, clf):
        safe = "".join(c if c.isalnum() or c in "-_+" else "_"
                       for c in f"{tag}_{sname}_{clf}_{int(balance)}{int(bool(tune_threshold))}")
        return os.path.join(cache_dir, f"{safe}.pkl") if cache_dir else None

    for sname, X in sources.items():
        for clf in classifiers:
            path = cache_file(sname, clf)
            r = None
            if path and os.path.exists(path):
                try:
                    with open(path, "rb") as fh:
                        r = pickle.load(fh)
                    print(f"  [grid] {sname}/{clf} loaded from cache")
                except Exception as exc:
                    print(f"  [grid] cache for {sname}/{clf} unreadable ({exc}); recomputing")
                    r = None
            t0 = time.perf_counter()
            if r is None:
                try:
                    r = evaluate_balanced(cohort, X, clf_kind=clf, cfg=cfg,
                                          balance=balance, tune_threshold=tune_threshold)
                except Exception as exc:
                    print(f"  [grid] {sname}/{clf} failed: {str(exc)[:160]}")
                    continue
                if path:
                    try:
                        atomic_write(path, lambda p: pickle.dump(r, open(p, "wb")))
                    except Exception as exc:
                        print(f"  [grid] could not cache {sname}/{clf}: {exc}")
            train_s = time.perf_counter() - t0
            if log is not None:
                log.add(ResourceRecord(stage=f"train:{sname}+{clf}",
                                       n_items=int(len(cohort.df)), wall_s=train_s,
                                       peak_vram_gb=0.0, device="cpu",
                                       extra={"n_features": int(X.shape[1])}))
            store[(sname, clf)] = r
            m, fm = r["metrics_oof"], r["fold_mean_std"]
            rows.append({
                "features": sname, "classifier": clf,
                "AUROC": m["auroc"], "AUROC_lo": m["auroc_ci95"][0], "AUROC_hi": m["auroc_ci95"][1],
                "AUROC_fold_mean": fm.get("auroc", (np.nan,))[0],
                "AUROC_fold_std": fm.get("auroc", (np.nan, np.nan))[1],
                "PR_AUC": m["pr_auc"], "PR_AUC_lo": m["pr_auc_ci95"][0], "PR_AUC_hi": m["pr_auc_ci95"][1],
                "Sens": m["sensitivity"], "Sens_lo": m["sensitivity_ci95"][0], "Sens_hi": m["sensitivity_ci95"][1],
                "Spec": m["specificity"], "Spec_lo": m["specificity_ci95"][0], "Spec_hi": m["specificity_ci95"][1],
                "F1": m["f1"], "F1_lo": m["f1_ci95"][0], "F1_hi": m["f1_ci95"][1],
                "MCC": m["mcc"], "BalAcc": m["balanced_acc"],
                "Acc": m["accuracy"], "Acc_lo": m["accuracy_ci95"][0], "Acc_hi": m["accuracy_ci95"][1],
                "n_eval": m["n_eval_real"], "n_pos": m["n_pos_total"],
                "pred_pos_rate": float(np.mean(r["oof_pred"] == POS)),
                "degenerate": _is_degenerate(r),
                "train_s": round(train_s, 1),
            })
    out = pd.DataFrame(rows)
    if len(out) and out["degenerate"].any():
        bad = out.loc[out["degenerate"], ["features", "classifier"]].to_records(index=False)
        print("[warning] degenerate fits (constant prediction or AUROC at chance): "
              f"{[f'{a}+{b}' for a, b in bad]}\n"
              "  Report these as they are. They are usually a default hyperparameter "
              "meeting a small, low-dimensional matrix — LightGBM's min_child_samples=20 "
              "blocks every split when positives are rare — not a property of the features. "
              "Changing the hyperparameter is a separate, declared experiment.")
    return out, store


def _is_degenerate(result: dict) -> bool:
    """A fit that predicts one class everywhere, or ranks at chance, carries no
    information. Flagged rather than dropped so it stays visible in the table."""
    pred = result["oof_pred"]
    if len(np.unique(pred)) < 2:
        return True
    lo, hi = result["metrics_oof"].get("auroc_ci95", (np.nan, np.nan))
    return bool(not np.isnan(lo) and lo <= 0.5 <= hi)


def per_fold_table(store: dict) -> pd.DataFrame:
    """Item 11: every fold of every configuration, with its positive count."""
    frames = []
    for (src, clf), r in store.items():
        pf = r["per_fold"].copy()
        pf.insert(0, "classifier", clf)
        pf.insert(0, "features", src)
        frames.append(pf)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def format_table_ii(grid: pd.DataFrame, decimals: int = 3) -> pd.DataFrame:
    """Manuscript-ready Table II: point estimate with its 95% interval.

    Values are formatted with a decimal point (item 23) and no trailing dot
    (item 24); the F1 column is spelled F1 (item 25).
    """
    def fmt(row, base):
        lo, hi = row.get(f"{base}_lo", np.nan), row.get(f"{base}_hi", np.nan)
        v = row[base]
        if np.isnan(v):
            return "n/a"
        if np.isnan(lo) or np.isnan(hi):
            return f"{v:.{decimals}f}"
        return f"{v:.{decimals}f} ({lo:.{decimals}f}-{hi:.{decimals}f})"

    out = pd.DataFrame({
        "Features": grid["features"],
        "Classifier": grid["classifier"],
        "AUROC": grid.apply(lambda r: fmt(r, "AUROC"), axis=1),
        "PR-AUC": grid.apply(lambda r: fmt(r, "PR_AUC"), axis=1),
        "Sensitivity": grid.apply(lambda r: fmt(r, "Sens"), axis=1),
        "Specificity": grid.apply(lambda r: fmt(r, "Spec"), axis=1),
        "F1": grid.apply(lambda r: fmt(r, "F1"), axis=1),
        "MCC": grid["MCC"].round(decimals),
        "n": grid["n_eval"],
        "positives": grid["n_pos"],
    })
    return out


# --------------------------------------------------------------------------- #
# Item 22 — reproducibility manifest
# --------------------------------------------------------------------------- #
def reproducibility_manifest(cohort: H.Cohort, cfg: RevisionConfig = CFG,
                             extra: dict | None = None) -> dict:
    """Item 22: seeds, versions, split definition, and the accession list.

    ``geno2pheno_version`` cannot be read from the data and must be filled in by
    hand from the job records; it is emitted as None so a missing value is
    visible in the appendix rather than silently absent.
    """
    versions = {"python": platform.python_version(), "platform": platform.platform()}
    for mod in ("numpy", "pandas", "sklearn", "scipy", "torch", "transformers",
                "xgboost", "lightgbm", "shap", "Bio"):
        try:
            m = __import__(mod)
            versions[mod] = getattr(m, "__version__", "unknown")
        except Exception:
            versions[mod] = "not installed"
    try:
        import torch
        versions["cuda"] = torch.version.cuda if torch.cuda.is_available() else "cpu-only"
        versions["gpu"] = _device_name()
    except Exception:
        pass

    accessions = sorted(cohort.df.loc[cohort.df["eval_eligible"],
                                      "accession" if "accession" in cohort.df else "group_id"]
                        .astype(str).unique().tolist())
    return {
        "config": cfg.to_dict(),
        "versions": versions,
        "split": {
            "scheme": "StratifiedGroupKFold",
            "n_splits": cfg.n_splits,
            "group_col": cfg.group_col,
            "grouping": (f"accessions merged into identity clusters by greedy "
                         f"single-linkage k-mer Jaccard >= {cfg.cluster_threshold} "
                         f"at k={cfg.cluster_k}"),
            "n_accessions": int(cohort.df["group_id"].nunique()),
            "n_groups": int(cohort.df[cfg.group_col].nunique()),
            "shuffle": True,
            "random_state": cfg.seed,
        },
        "label_source": {
            "tool": "Geno2pheno[HCV]",
            "version": None,            # fill in from the job records
            "caveat": H.CLINICAL_CAVEAT,
        },
        "n_accessions_listed": len(accessions),
        "accessions": accessions,
        **(extra or {}),
    }


def manuscript_digest(path: str, counts: dict, overlap: dict, cfg: RevisionConfig,
                      grid_bal=None, grid_plain=None, base_grid=None, floors=None,
                      per_fold=None, strict=None, per_protein=None, holdout=None,
                      external=None, significance=None, ops=None, cost=None,
                      inventory=None, plddt=None, label_consistency=None,
                      g2p_version=None) -> str:
    """Write one file that says which number belongs in which revision item.

    The run produces twenty tables; this turns them into the sentences the
    manuscript needs, item by item, so the edits can be written without
    cross-referencing CSVs. Anything a run did not produce is listed as
    outstanding rather than omitted, so the gaps are visible.
    """
    L: list[str] = []
    w = L.append
    missing: list[str] = []

    def best(df, by="AUROC"):
        if df is None or not len(df):
            return None
        return df.sort_values(by, ascending=False).iloc[0]

    def fmt(r, lo=None, hi=None):
        if r is None:
            return "n/a"
        if lo and hi and lo in r and not pd.isna(r[lo]):
            return f"{r['AUROC']:.3f} (95% CI {r[lo]:.3f}–{r[hi]:.3f})"
        return f"{r['AUROC']:.3f}"

    w("# Numbers for the manuscript\n")
    w("Generated by the revision run. Each section gives the figure to quote and, "
      "where the wording is constrained, the sentence to use.\n")

    w("\n## Item 20 — category counts and the binary mapping  [E6]\n")
    pc = counts.get("per_category", {})
    w(f"- Real sequences: **{counts.get('n_real')}** "
      f"({counts.get('n_accessions')} accessions)")
    for k in ("susceptible", "polymorphism", "reduced_susceptibility", "resistant"):
        if k in pc:
            w(f"- {k}: **{pc[k]}**")
    w(f"- Binary collapse ({'+'.join(cfg.ras_positive_default)}): "
      f"**{counts.get('n_negative')} negative / {counts.get('n_positive')} positive**")
    w("\n> These counts must match every other count in the paper, including the "
      "abstract and Section 4.1.")

    w("\n## Item 21 — grouping definition and number of groups  [E3]\n")
    w(f"- Accessions: **{overlap.get('n_accessions')}**")
    w(f"- Groups after identity clustering: **{overlap.get('n_identity_clusters')}**")
    w(f"- Clustering: greedy single-linkage, k-mer Jaccard ≥ "
      f"**{overlap.get('cluster_threshold')}** at k = **{overlap.get('cluster_k')}**")
    fa = overlap.get("fold_audit_cluster", {})
    if fa:
        w(f"- Maximum train/validation k-mer identity: **{fa.get('max_train_val_identity', float('nan')):.3f}** "
          f"(groups disjoint: {fa.get('group_disjoint')})")
    w("\n> Sentence for Section 3.8: \"Accessions were merged into identity clusters by "
      "greedy single-linkage clustering on k-mer Jaccard similarity (k = "
      f"{overlap.get('cluster_k')}, threshold {overlap.get('cluster_threshold')}), giving "
      f"{overlap.get('n_identity_clusters')} groups from {overlap.get('n_accessions')} "
      "accessions. StratifiedGroupKFold was applied over these clusters, so no sequence "
      "and no near-identical variant of it appears in both a training and a validation "
      f"fold; the maximum observed train/validation k-mer identity was "
      f"{fa.get('max_train_val_identity', float('nan')):.3f}.\"")

    w("\n## Item 8 — leakage and overlap audit  [E3, R1, R2 maj4]\n")
    w(f"- Real sequences {overlap.get('n_real')}, augmented rows {overlap.get('n_augmented')}")
    w(f"- Augmented rows identical to a real sequence: "
      f"**{overlap.get('augmented_rows_identical_to_a_real_sequence')}**")
    w(f"- Parent accessions of augmented rows also in the evaluation set: "
      f"**{overlap.get('parents_also_in_eval_set')}**")
    w("\n> The augmented rows are perturbed variants of the evaluated accessions, so "
      "corpus and evaluation set are not disjoint by construction. The control is that "
      "splits are grouped by identity cluster, which the figures above verify.")

    w("\n## Items 1 and 2 — baselines  [E1, E2]\n")
    if floors is not None and len(floors):
        maj = floors[floors["baseline"] == "majority_negative"]
        if len(maj):
            w(f"- Majority-class floor: accuracy **{maj.iloc[0]['accuracy']:.3f}**, "
              f"balanced accuracy {maj.iloc[0]['balanced_acc']:.3f}")
    if base_grid is not None and len(base_grid):
        for src in base_grid["features"].unique():
            r = best(base_grid[base_grid["features"] == src])
            w(f"- **{src}**: best AUROC {fmt(r, 'AUROC_lo', 'AUROC_hi')} "
              f"({r['classifier']})")
        w("\n> Item 1: Geno2pheno derives its labels from the RAM positions, so a "
          "near-perfect score for that baseline confirms the labelling rule is "
          "recoverable and is not evidence that the baseline is a useful model.")
        w("> Item 2: if the metadata baseline is close to the sequence models, part of "
          "the measured performance reflects record structure rather than biology, and "
          "Section 4.5 must say so.")
    else:
        missing.append("items 1 and 2 — baseline grid did not run")

    w("\n## Item 6 — class balancing and threshold tuning  [E7, R1 min, R2 maj3]\n")
    if grid_bal is not None and grid_plain is not None and len(grid_bal) and len(grid_plain):
        a = grid_plain.set_index(["features", "classifier"])["Sens"]
        b = grid_bal.set_index(["features", "classifier"])["Sens"]
        d = (b - a).dropna().sort_values(ascending=False)
        w("Largest sensitivity gains (as submitted → balanced + tuned):\n")
        for (src, clf), delta in d.head(6).items():
            w(f"- {src} + {clf}: **{a[(src, clf)]:.3f} → {b[(src, clf)]:.3f}** "
              f"({delta:+.3f})")
        w("\n> Thresholds come from an inner 3-fold cross-validation on the training "
          "folds only, never from the validation fold.")
    else:
        missing.append("item 6 — the paired grid did not run")

    w("\n## Item 15 — the sensitivity trade-off  [R1]\n")
    if grid_bal is not None and len(grid_bal):
        for src in ("ColabFold", "SeqKmer3"):
            sub = grid_bal[grid_bal["features"] == src]
            if len(sub):
                r = sub.sort_values("AUROC", ascending=False).iloc[0]
                w(f"- {src} best AUROC: {r['AUROC']:.3f} with recall "
                  f"**{r['Sens']:.3f}**, specificity {r['Spec']:.3f} ({r['classifier']})")
                r2 = sub.sort_values("Sens", ascending=False).iloc[0]
                w(f"  - highest recall for {src}: **{r2['Sens']:.3f}** ({r2['classifier']}, "
                  f"AUROC {r2['AUROC']:.3f})")
        w("\n> Section 4.4 must state both operating points rather than ranking the "
          "sources on AUROC alone.")

    w("\n## Item 16 — class imbalance and metric variance  [R1 min]\n")
    w(f"- Class split: {counts.get('n_negative')} negative / {counts.get('n_positive')} positive "
      f"({100 * counts.get('n_positive', 0) / max(counts.get('n_real', 1), 1):.1f}% positive)")
    if per_fold is not None and len(per_fold):
        w(f"- Positives per validation fold: "
          f"**{per_fold['n_pos_val'].min()}–{per_fold['n_pos_val'].max()}** "
          f"(median {int(per_fold['n_pos_val'].median())})")
    if grid_bal is not None and len(grid_bal):
        wid = (grid_bal["Sens_hi"] - grid_bal["Sens_lo"]).dropna()
        if len(wid):
            w(f"- Width of the 95% interval on sensitivity: "
              f"**{wid.min():.3f}–{wid.max():.3f}** across configurations")
        w("\n> That width is the quantitative answer to R1 min: with this few positives "
          "per fold, minority-class metrics cannot be compared on point estimates.")

    w("\n## Item 19 — which features entered the classifiers  [R2 maj2]\n")
    if inventory is not None and len(inventory):
        for _, r in inventory.iterrows():
            w(f"- **{r['source']}** ({r['n_features']} features): {r['families']}")
        fl = inventory[inventory["has_predefined_RAS_pLDDT"]]
        for _, r in fl.iterrows():
            w(f"  - {r['source']} carries the pre-defined RAS pLDDT columns: "
              f"{r['RAS_pLDDT_columns']}")
        w("\n> Section 3.7 currently also lists radius of gyration and solvent-accessible "
          "surface area. Confirm against this inventory before the text is finalised.")
    else:
        missing.append("item 19 — feature inventory did not run")
    if plddt and plddt.get("n_structures"):
        w(f"\n- AlphaFold2 structure confidence: mean pLDDT **{plddt['mean']:.1f}** "
          f"over {plddt['n_structures']} structures; "
          f"{plddt['bands']['very_low_lt50']} below 50, "
          f"{plddt['bands']['confident_70_90'] + plddt['bands']['very_high_ge90']} at 70 or above")

    w("\n## Item 7 — AlphaFold2 ablation  [R2 maj2]\n")
    if grid_bal is not None and len(grid_bal):
        full = grid_bal[grid_bal["features"] == "ColabFold"]
        abl = grid_bal[grid_bal["features"] == "ColabFold_noRASpLDDT"]
        if len(full) and len(abl):
            rf, ra = best(full), best(abl)
            w(f"- With the six pLDDT-at-RAS features: AUROC **{rf['AUROC']:.3f}** ({rf['classifier']})")
            w(f"- Without them: AUROC **{ra['AUROC']:.3f}** ({ra['classifier']})")
            w(f"- Change: **{ra['AUROC'] - rf['AUROC']:+.3f}**")
        else:
            missing.append("item 7 — AlphaFold2 ablation needs both ColabFold sources")

    for label, df, note in (
        ("Item 4 — per-protein results  [E5]", per_protein,
         "Quote the interval, not the point estimate; these subsets are small."),
        ("Item 5 — stricter resistance definition  [E6]", strict,
         "The strictest mapping has very few positives; report it as an interval."),
        ("Item 3 — held-out evaluation  [E4]", holdout,
         "This is an internal cluster-disjoint holdout, not external validation."),
        ("Item 3 — external validation  [E4]", external,
         "Accessions disjoint from the training cohort, labelled by the same tool."),
        ("Item 9 — significance testing  [R2 maj6]", significance,
         f"Minimum attainable two-sided Wilcoxon p at {cfg.n_splits} folds is "
         f"{2.0 ** (1 - cfg.n_splits):.4f}; state it alongside any null result."),
        ("Item 10 — matched operating points  [E7]", ops,
         "Sensitivity at fixed specificity and the reverse, plus PR curves."),
        ("Item 12 — computational cost  [R2 maj5]", cost,
         "Time per 1,000 sequences and peak VRAM, per stage."),
    ):
        w(f"\n## {label}\n")
        if df is not None and len(df):
            w("```")
            w(df.round(3).to_string(index=False)[:2600])
            w("```")
            w(f"> {note}")
        else:
            w("*Not produced by this run.*")
            missing.append(label)

    w("\n## Item 22 — reproducibility  [E10]\n")
    w("- Accession list: `reproducibility_manifest_accessions.txt`")
    w("- Seeds, split scheme, package versions: `reproducibility_manifest.json`")
    if g2p_version:
        w(f"- Geno2pheno[HCV] version observed: **{g2p_version.get('version')}** "
          f"on {g2p_version.get('observed_on')}")
    w("- The version used for the ORIGINAL labelling must come from the job records; "
      "it is recorded separately and is `null` until supplied.")

    if label_consistency is not None and len(label_consistency):
        w("\n## Data note — labels not determined by the recorded mutations\n")
        w("```")
        w(label_consistency.to_string(index=False)[:1200])
        w("```")
        w("> Supports the silver-standard framing in item 13, and is why an external "
          "set must be labelled by Geno2pheno rather than by a re-implemented rule.")

    w("\n## Still outstanding\n")
    if missing:
        for m in missing:
            w(f"- {m}")
    else:
        w("Nothing — every item above produced its numbers.")
    w("\nAlways manual, whatever the run produces:")
    w("- item 26 (point-by-point response), item 27 (track changes), item 28 (submission)")
    w("- the Geno2pheno upload that labels the external set, if item 3 is to be answered")
    w("- the Geno2pheno version used for the original labelling")

    # Fold to ASCII. The file is written as UTF-8, but a Windows console is
    # cp1252 and a bare print of this text raises UnicodeEncodeError on a
    # character as ordinary as the >= sign, which would fail the step.
    text = "\n".join(L) + "\n"
    for bad, good in (("≥", ">="), ("≤", "<="), ("–", "-"),
                      ("—", "--"), ("×", "x"), ("→", "->")):
        text = text.replace(bad, good)
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(text)
    print(f"wrote {path}")
    if missing:
        print(f"{len(missing)} item(s) still outstanding — see the end of the file")
    return path


def save_manifest(manifest: dict, path: str = "reproducibility_manifest.json") -> str:
    json.dump(manifest, open(path, "w"), indent=2, default=str)
    acc_path = os.path.splitext(path)[0] + "_accessions.txt"
    with open(acc_path, "w") as fh:
        fh.write("\n".join(manifest.get("accessions", [])))
    print(f"wrote {path} and {acc_path} ({manifest.get('n_accessions_listed', 0)} accessions)")
    return path

### preflight

In [ ]:
# [kaggle] preflight: environment, inputs, and the two required modules.
import glob, os, pathlib, subprocess, sys, socket

ON_KAGGLE = os.path.exists("/kaggle/working")
WORKDIR = pathlib.Path("/kaggle/working") if ON_KAGGLE else pathlib.Path.cwd()
os.chdir(WORKDIR)
print(f"[pre] on_kaggle={ON_KAGGLE} cwd={WORKDIR}")
print(f"[pre] python {sys.version.split()[0]}")

try:
    import torch
    HAS_GPU_PRE = torch.cuda.is_available()
    print(f"[pre] torch {torch.__version__} cuda={torch.version.cuda} gpu={HAS_GPU_PRE}"
          + (f" ({torch.cuda.get_device_name(0)})" if HAS_GPU_PRE else ""))
except Exception as exc:
    HAS_GPU_PRE = False
    print(f"[pre] torch unavailable: {type(exc).__name__}")

def _internet(timeout=4):
    try:
        socket.create_connection(("pypi.org", 443), timeout=timeout).close()
        return True
    except Exception:
        return False

HAS_NET = _internet()
print(f"[pre] internet={HAS_NET}" + ("" if HAS_NET else "  (toggle it in the sidebar if you need installs)"))

# --- attached inputs -------------------------------------------------------
INPUT_ROOTS = ["/kaggle/input", "/kaggle/usr/lib"]
for root in INPUT_ROOTS:
    if os.path.isdir(root):
        for entry in sorted(os.listdir(root)):
            print(f"[pre] input: {root}/{entry}")

# --- the two modules -------------------------------------------------------
MODULES = ("hcv_pipeline", "hcv_revision")
SEARCH = ["/kaggle/input/**", "/kaggle/usr/lib/**", str(WORKDIR / "**"), "./**"]
found = {}
for mod in MODULES:
    for pat in SEARCH:
        hits = [p for p in glob.glob(f"{pat}/{mod}.py", recursive=True) if os.path.isfile(p)]
        if hits:
            found[mod] = hits[0]
            break

for mod, path in found.items():
    d = os.path.dirname(os.path.abspath(path))
    if d not in sys.path:
        sys.path.insert(0, d)
    print(f"[pre] {mod}.py -> {path}")

MISSING_MODULES = [m for m in MODULES if m not in found]
if MISSING_MODULES:
    print()
    print("[pre] MISSING:", ", ".join(f"{m}.py" for m in MISSING_MODULES))
    print("[pre] Attach them as a Utility Script (File -> Add utility script) or inside a")
    print("[pre] Dataset, then re-run this cell. Cell 3 will raise until they resolve.")

# --- the cohort CSV --------------------------------------------------------
_csv = [p for pat in ("/kaggle/input/**/hcv*.csv", "./**/hcv*.csv")
        for p in glob.glob(pat, recursive=True) if os.path.isfile(p)]
if _csv:
    for p in sorted(_csv)[:10]:
        print(f"[pre] cohort csv: {p}  ({os.path.getsize(p):,} bytes)")
else:
    print("[pre] no hcv*.csv found; the cohort cells will have nothing to load")

### brief

In [ ]:
print("BRIEF LOCKED")
# [kaggle] Environment. The uploaded version pinned torch==2.1.0, numpy==1.26.4,
# pandas==2.1.4 and pip-installed colabfold[alphafold] plus apt cd-hit. On Kaggle
# that is actively harmful: the image's torch is built against the session's CUDA
# driver, and replacing it drops you to CPU without an error. colabfold's install
# does not finish inside a session, and apt-get needs internet and root.
# Installs are therefore opt-in and never touch the numeric core.
import importlib, os, pathlib, subprocess, sys

RESULTS = pathlib.Path("results"); RESULTS.mkdir(exist_ok=True)

INSTALL = os.environ.get("HCV_INSTALL", "0") == "1"
# Add only packages the image lacks. Never torch, numpy, pandas, scikit-learn.
EXTRA_PINS = [p for p in os.environ.get("HCV_EXTRA_PINS", "").split(",") if p.strip()]

def _pip(*pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "--no-warn-conflicts", *pkgs], capture_output=True, text=True)
    if r.returncode != 0:
        print("[env] pip non-zero for", pkgs[0], "->", r.stderr.strip().splitlines()[-1:])
    return r.returncode

print("[env] python", sys.version.split()[0])
if INSTALL and EXTRA_PINS:
    for p in EXTRA_PINS:
        _pip(p.strip())
elif INSTALL:
    print("[env] HCV_INSTALL=1 but HCV_EXTRA_PINS is empty -> nothing to install")
else:
    print("[env] HCV_INSTALL=0 -> using the Kaggle image as found")

# Report what the image actually carries, rather than asserting a pin set.
WANTED = {"torch": None, "transformers": None, "sklearn": "scikit-learn",
          "xgboost": None, "lightgbm": None, "shap": None, "Bio": "biopython",
          "statsmodels": None, "pandas": None, "numpy": None, "pyarrow": None,
          "matplotlib": None, "scipy": None}
AVAILABLE, ABSENT = {}, []
for mod, dist in WANTED.items():
    try:
        AVAILABLE[mod] = getattr(importlib.import_module(mod), "__version__", "unknown")
    except Exception:
        ABSENT.append(dist or mod)
print("[env] present:", ", ".join(f"{k}=={v}" for k, v in sorted(AVAILABLE.items())))
if ABSENT:
    print("[env] ABSENT:", ", ".join(ABSENT),
          "-> set HCV_INSTALL=1 and HCV_EXTRA_PINS, with internet on")

# colabfold is not installable in a Kaggle session; the ColabFold feature block
# in the features cell already degrades to a logged skip when it is absent.
HAS_COLABFOLD = subprocess.run(["bash", "-lc", "command -v colabfold_batch"],
                               capture_output=True, text=True).returncode == 0
HAS_CDHIT = subprocess.run(["bash", "-lc", "command -v cd-hit-est"],
                           capture_output=True, text=True).returncode == 0
print("[env] colabfold_batch available:", HAS_COLABFOLD)
print("[env] cd-hit-est available:", HAS_CDHIT)

_frozen = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                         capture_output=True, text=True).stdout
pathlib.Path("requirements.txt").write_text(_frozen, encoding="utf-8")
(RESULTS / "requirements.txt").write_text(_frozen, encoding="utf-8")
print("[env] requirements.txt:", len(_frozen.splitlines()), "resolved pins")

### imports

In [ ]:
import glob, hashlib, json, math, os, pathlib, platform, random, re, shutil, socket
import subprocess, sys, time, warnings
from dataclasses import dataclass, asdict, field
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# [kaggle] sys.path was primed by the preflight cell.
try:
    import hcv_pipeline as H
    import hcv_revision as R
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        f"{exc.name}.py is not importable. Attach hcv_pipeline.py and hcv_revision.py "
        "as a Kaggle Utility Script or inside a Dataset, then re-run cell 1 (preflight) "
        "so it can put them on sys.path. They carry the whole method and cannot be "
        "reconstructed from this notebook."
    ) from exc

RESULTS = pathlib.Path("results"); RESULTS.mkdir(exist_ok=True)
FIGS = RESULTS / "figures"; FIGS.mkdir(exist_ok=True)
DECISIONS = []

def decide(key, choice, because):
    DECISIONS.append({"key": key, "choice": str(choice), "because": because})
    print(f"[decision] {key} = {choice}  ({because})")

MODE = os.environ.get("HCV_MODE", "fast")
assert MODE in ("recover", "full", "fast")

@dataclass
class Config:
    mode: str = MODE
    seed: int = 42
    n_splits: int = 5
    inner_splits: int = 3
    cluster_threshold: float = 0.90
    cluster_k: int = 6
    group_col: str = "cluster_group"
    classifiers: tuple = ("logreg", "randomforest", "xgboost", "lightgbm", "mlp")
    esm2_model: str = "facebook/esm2_t33_650M_UR50D"
    finetune_esm2: bool = True
    ft_epochs: int = 3
    ft_unfrozen_layers: int = 3
    ft_mlm_prob: float = 0.15
    ft_batch_size: int = 4
    ft_lr: float = 1e-4
    esm2_max_len: int = 1022
    run_esmfold: bool = True
    esmfold_live: bool = False
    esmfold_max_len: int = 350
    esmfold_chunk: int = 32
    run_colabfold: bool = True
    cf_max_seqs: int = 60
    cf_msa_mode: str = "single_sequence"
    cf_num_recycle: int = 1
    cf_num_models: int = 1
    cf_truncate_aa: int = 400
    cf_timeout_min: int = 45
    cf_target_coverage: float = 0.95
    cf_min_coverage: float = 0.80
    INCLUDE_PLDDT_RAS: bool = True
    CLASS_WEIGHT: str = "balanced"
    THRESHOLD_MODE: str = "youden_oof"
    PURGE_OVERLAP: bool = False
    n_boot: int = 2000
    alpha: float = 0.05
    spec_targets: tuple = (0.90, 0.95)
    sens_targets: tuple = (0.85, 0.90)
    seed_stability_seeds: tuple = tuple(range(1, 11))
    holdout_year_quintile: float = 0.20
    holdout_min_positives: int = 3
    overlap_purge_threshold: float = 0.02
    cdhit_identities: tuple = (1.00, 0.90)
    paper_n: int = 1890
    paper_n_pos: int = 187
    paper_n_neg: int = 1703
    paper_pretrain_n: int = 25981
    allow_network: bool = False

CFG = Config()
if MODE == "fast":
    CFG.n_splits = 2
    CFG.classifiers = ("logreg", "xgboost")
    CFG.cf_max_seqs = 60
    CFG.run_colabfold = False
    CFG.esmfold_live = False
    CFG.finetune_esm2 = False
    CFG.n_boot = 200
    CFG.seed_stability_seeds = (1, 2, 3)
    FAST_N_SEQS = 60
else:
    FAST_N_SEQS = None

SEED = CFG.seed
random.seed(SEED); np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
try:
    import torch
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    HAS_GPU = torch.cuda.is_available()
except Exception:
    torch = None; HAS_GPU = False

R.CFG = R.RevisionConfig(
    seed=SEED, n_splits=CFG.n_splits, inner_splits=CFG.inner_splits,
    cluster_threshold=CFG.cluster_threshold, cluster_k=CFG.cluster_k,
    group_col=CFG.group_col, tune_threshold=True, threshold_metric="youden",
    spec_targets=CFG.spec_targets, sens_targets=CFG.sens_targets,
    n_boot=CFG.n_boot, alpha=CFG.alpha,
    af2_ras_positions=(28, 30, 31, 93, 282, 316),
    classifiers=tuple(CFG.classifiers),
)
assert R.CFG.seed == CFG.seed == SEED == 42

REPORT = R.RunReport(str(RESULTS / "run_report.json"))
COST = R.ResourceLog(str(RESULTS / "compute_cost.json"))
print(f"[cfg] MODE={MODE} GPU={HAS_GPU} folds={CFG.n_splits} classifiers={list(CFG.classifiers)} seed={SEED}")

### hashing

In [ ]:
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for b in iter(lambda: fh.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

INPUT_GLOBS = ["hcv*.csv", "*.ipynb", "hcv_pipeline.py", "hcv_revision.py",
               "requirements.txt", "/kaggle/input/**/hcv*.csv"]
_inputs = []
for pat in INPUT_GLOBS:
    for p in sorted(glob.glob(pat, recursive=True)):
        if os.path.isfile(p):
            _inputs.append({"path": os.path.abspath(p), "bytes": os.path.getsize(p),
                            "sha256": sha256_file(p)})
_seen, INPUTS = set(), []
for rec in _inputs:
    if rec["sha256"] not in _seen:
        _seen.add(rec["sha256"]); INPUTS.append(rec)

def _git_head():
    try:
        r = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True)
        return r.stdout.strip() or None
    except Exception:
        return None

def _pkg_versions():
    out = {}
    for m in ("numpy", "pandas", "sklearn", "torch", "transformers", "xgboost",
              "lightgbm", "shap", "Bio", "statsmodels", "matplotlib"):
        try:
            out[m] = __import__(m).__version__
        except Exception as exc:
            out[m] = f"unavailable: {type(exc).__name__}"
    return out

def _cuda_info():
    info = {"cuda_available": bool(HAS_GPU)}
    if torch is not None:
        info["torch_cuda"] = getattr(torch.version, "cuda", None)
        if HAS_GPU:
            info["gpu_model"] = torch.cuda.get_device_name(0)
            info["gpu_total_gb"] = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=driver_version",
                            "--format=csv,noheader"], capture_output=True, text=True, timeout=10)
        info["driver_version"] = r.stdout.strip().splitlines()[0] if r.stdout.strip() else None
    except Exception:
        info["driver_version"] = None
    return info

MANIFEST = {
    "manuscript": "BJAI-2026-0104",
    "git_head": _git_head(),
    "utc_timestamp": datetime.now(timezone.utc).isoformat(),
    "hostname": socket.gethostname(),
    "platform": platform.platform(),
    "python": sys.version.split()[0],
    "packages": _pkg_versions(),
    "cuda": _cuda_info(),
    "config": asdict(CFG),
    "inputs": INPUTS,
}
(RESULTS / "manifest.json").write_text(json.dumps(MANIFEST, indent=2, default=str), encoding="utf-8")
print(f"[prov] {len(INPUTS)} input file(s) hashed; manifest.json written")

### inputs

In [ ]:
COHORT_GLOBS = ["/kaggle/input/**/hcv*.csv", "/content/**/hcv*.csv", "./hcv*.csv",
                "./**/hcv*.csv", "../**/hcv*.csv"]
CANON = {
    "accession": "accession", "acc": "accession", "genbank_accession": "accession",
    "id": "accession", "sequence_id": "accession",
    "nt_sequence": "nt_sequence", "seq": "nt_sequence", "sequence": "nt_sequence",
    "nucleotide": "nt_sequence", "nt_seq": "nt_sequence", "dna": "nt_sequence",
    "aa_seq": "aa_seq", "protein": "aa_seq", "aa_sequence": "aa_seq",
    "label": "label", "g2p_label": "label", "geno2pheno": "label", "class": "label",
    "group_id": "group_id", "isolate_group": "group_id",
    "length": "Length", "seq_len": "seq_len",
    "collection_date": "Collection_Date", "year": "Collection_Date",
    "submitters": "Submitters", "lab": "Submitters", "organization": "Organization",
    "country": "Country", "geo_location": "Geo_Location", "region": "Geo_Location",
    "covered_regions": "covered_regions",
    "has_ns5a": "has_ns5a", "has_ns5b": "has_ns5b", "has_ns3": "has_ns3",
    "eval_eligible": "eval_eligible",
}

def canonicalise(frame):
    ren = {}
    for c in frame.columns:
        k = c.strip().lower().replace(" ", "_")
        if k in CANON and CANON[k] != c:
            ren[c] = CANON[k]
    return frame.rename(columns=ren) if ren else frame

def find_cohort_csv():
    for pat in COHORT_GLOBS:
        for p in sorted(glob.glob(pat, recursive=True)):
            if os.path.isfile(p) and os.path.getsize(p) > 10_000:
                return p
    return None

COHORT_CSV = find_cohort_csv()
RAW = None
if COHORT_CSV:
    RAW = canonicalise(pd.read_csv(COHORT_CSV, low_memory=False))
    print(f"[cohort] loaded {COHORT_CSV} rows={len(RAW):,} cols={RAW.shape[1]}")
    missing = [c for c in ("accession", "nt_sequence") if c not in RAW.columns]
    if missing:
        decide("cohort_csv_partial", f"missing={missing}", "named columns absent after canonicalisation; rebuild path")
        RAW = None
else:
    decide("cohort_csv", "absent", "no hcv*.csv matched any glob; rebuild path")
REBUILD_NEEDED = RAW is None

### cohort source

In [ ]:
COHORT_AUDIT_PRE = {"rebuild_attempted": bool(REBUILD_NEEDED)}

def _rebuild_from_genbank():
    from Bio import Entrez, SeqIO
    Entrez.email = os.environ.get("ENTREZ_EMAIL", "")
    if not Entrez.email:
        raise RuntimeError("ENTREZ_EMAIL unset; NCBI requires a contact address")
    query = ('"Hepatitis C virus genotype 4"[Organism] AND Egypt[All Fields] '
             'AND (NS5A[All Fields] OR NS5B[All Fields])')
    ids = Entrez.read(Entrez.esearch(db="nucleotide", term=query, retmax=20000))["IdList"]
    recs = list(SeqIO.parse(Entrez.efetch(db="nucleotide", id=",".join(ids),
                                          rettype="gb", retmode="text"), "genbank"))
    rows = []
    for r in recs:
        src = next((f for f in r.features if f.type == "source"), None)
        country = (src.qualifiers.get("country", [""])[0] if src else "")
        if "egypt" not in country.lower():
            continue
        rows.append({"accession": r.id, "nt_sequence": str(r.seq),
                     "description": r.description, "Country": country,
                     "Collection_Date": (src.qualifiers.get("collection_date", [""])[0]
                                         if src else ""),
                     "Length": len(r.seq)})
    frame = pd.DataFrame(rows).drop_duplicates("nt_sequence").reset_index(drop=True)
    frame["group_id"] = frame["accession"].str.split(".").str[0]
    frame["eval_eligible"] = True
    frame["label"] = np.nan
    return frame

if REBUILD_NEEDED:
    if not CFG.allow_network:
        decide("rebuild_network", "blocked",
               "CFG.allow_network is False; outward calls are not made unattended")
        RAW = pd.DataFrame(columns=["accession", "nt_sequence", "label",
                                    "group_id", "eval_eligible"])
        COHORT_AUDIT_PRE["rebuild_status"] = "blocked_no_network"
    else:
        try:
            RAW = _rebuild_from_genbank()
            RAW.to_csv("hcv_rebuilt.csv", index=False)
            COHORT_AUDIT_PRE["rebuild_status"] = "ok"
            COHORT_AUDIT_PRE["rebuilt_rows"] = int(len(RAW))
            print(f"[rebuild] wrote hcv_rebuilt.csv rows={len(RAW):,}")
        except Exception as exc:
            decide("rebuild_failed", type(exc).__name__, str(exc)[:160])
            RAW = pd.DataFrame(columns=["accession", "nt_sequence", "label",
                                        "group_id", "eval_eligible"])
            COHORT_AUDIT_PRE["rebuild_status"] = f"failed: {type(exc).__name__}"

N_DELTA = int(len(RAW)) - CFG.paper_n
COHORT_AUDIT_PRE.update({"n_loaded": int(len(RAW)), "paper_n": CFG.paper_n,
                         "n_delta_vs_paper": N_DELTA})
print(f"[cohort] rows={len(RAW):,} delta vs manuscript N={CFG.paper_n}: {N_DELTA:+,}")

### Cohort construction

`build_common_cohort` then `add_identity_clusters` at 90% k-mer Jaccard, k = 6, as described in Sections 3.1 and 3.9.

In [ ]:
if "eval_eligible" not in RAW.columns:
    RAW["eval_eligible"] = True
RAW["eval_eligible"] = RAW["eval_eligible"].astype(bool)
if "group_id" not in RAW.columns:
    RAW["group_id"] = RAW["accession"].astype(str).str.split(".").str[0]

HAS_4CLASS = bool("label" in RAW.columns and
                  RAW.loc[RAW["eval_eligible"], "label"].isin(H.LABELS_4).any())
print("[cohort] HAS_4CLASS =", HAS_4CLASS)

COH_FULL = H.add_identity_clusters(
    H.build_common_cohort(RAW, task="binary"), threshold=CFG.cluster_threshold,
    k=CFG.cluster_k)
COH = R.real_only(COH_FULL, R.CFG)
Y = COH.df[COH.label_col].to_numpy().astype(int)
n_classes = len(np.unique(Y))
assert n_classes == 2, f"binary column carries {n_classes} class(es), expected 2"
COH4 = None
if HAS_4CLASS:
    COH4 = R.real_only(H.add_identity_clusters(
        H.build_common_cohort(RAW, task="4class"), threshold=CFG.cluster_threshold,
        k=CFG.cluster_k), R.CFG)
print(f"[cohort] grid rows={len(COH.df)} positives={int(Y.sum())} "
      f"clusters={COH.df[CFG.group_col].nunique()} corpus rows={len(COH_FULL.df):,}")

### sampler

In [ ]:
d = COH_FULL.df.copy()
if "y_bin" not in d.columns:
    d["y_bin"] = d["label"].isin(H.DEFAULT_RAS_POSITIVE).astype(int)
if "eval_eligible" not in d.columns:
    raise ValueError("sampler needs 'eval_eligible'; cell 14 must run first")
d["eval_eligible"] = d["eval_eligible"].astype(bool)

rng = np.random.default_rng(42)
TARGET = dict(pos=187, neg=1703)
CAP    = dict(pos=5,   neg=6)

parts = []
for lab, n_take in TARGET.items():
    pos_val = 1 if lab == "pos" else 0
    pool = d[d.y_bin == pos_val]
    if len(pool) == 0:
        continue

    real = pool[ pool["eval_eligible"]]
    aug  = pool[~pool["eval_eligible"]]

    # Always include every real row first (43 pos, 318 neg on this CSV)
    if len(real) >= n_take:
        chosen_real = real.sample(n_take, random_state=42)
        chosen_aug  = aug.iloc[:0]
    else:
        chosen_real = real
        need = n_take - len(real)
        if len(aug) == 0 or need <= 0:
            chosen_aug = aug.iloc[:0]
        else:
            # Different seed per accession so the same relative position is
            # not excluded from every group (the original seed-42 bug)
            aug_groups = [
                g.sample(min(len(g), CAP[lab]), random_state=42 + i)
                for i, (_, g) in enumerate(aug.groupby("accession", sort=True))
            ]
            aug_pool   = pd.concat(aug_groups) if aug_groups else aug.iloc[:0]
            chosen_aug = aug_pool.sample(min(len(aug_pool), need), random_state=42)

    parts.append(pd.concat([chosen_real, chosen_aug]))

d = pd.concat(parts).sort_index() if parts else d.iloc[:0]
d = d.reset_index(drop=True)

SAMPLE_EXACT = (len(d) == 1890 and int(d.y_bin.sum()) == 187)
if not SAMPLE_EXACT:
    print(f"[sampler] WARNING target 1890/187 not reached: "
          f"n={len(d)} pos={int(d.y_bin.sum())}")

COH_SAMPLE = H.Cohort(df=d, task="binary", label_col="y_bin", classes=[0, 1],
                      ras_positive=H.DEFAULT_RAS_POSITIVE,
                      notes={"sampled": True})
COHORT_DELTA = {
    "sample_seed": 42, "cap_pos": 5, "cap_neg": 6,
    "n_accessions_total": int(COH_FULL.df["accession"].nunique()),
    "n_accessions_sampled": int(d["accession"].nunique()),
    "n_rows_before": int(len(COH_FULL.df)),
    "n_rows_after": int(len(d)),
    "n_real_in_sample": int(d["eval_eligible"].sum()),
    "n_augmented_in_sample": int((~d["eval_eligible"]).sum()),
    "original_sample_recovered": False,
    "target_counts_reached": bool(SAMPLE_EXACT),
}
(RESULTS / "cohort_delta.json").write_text(json.dumps(COHORT_DELTA, indent=2),
                                            encoding="utf-8")
print(f"[sampler] rows={len(d)} positives={int(d.y_bin.sum())} "
      f"accessions={d.accession.nunique()}")
print(f"[sampler] eval_eligible: {int(d['eval_eligible'].sum())} real / "
      f"{int((~d['eval_eligible']).sum())} augmented")
assert d["eval_eligible"].sum() >= 361 - 5, (
    "sampler did not preserve the real rows; the pipeline has nothing to score"
)

### Folds

StratifiedGroupKFold, five folds, grouped on `cluster_group`, seed 42.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

d = COH_SAMPLE.df.copy()
GROUPS = d[CFG.group_col].to_numpy()
Y = d["y_bin"].to_numpy().astype(int)
SGKF = StratifiedGroupKFold(n_splits=CFG.n_splits, shuffle=True, random_state=SEED)
FOLDS = list(SGKF.split(d, Y, GROUPS))
d["fold"] = -1
split_rows = []
for k, (tr, va) in enumerate(FOLDS):
    d.loc[d.index[va], "fold"] = k
    split_rows.append({"fold": k, "n_train": int(len(tr)), "n_val": int(len(va)),
                       "n_val_positive": int(Y[va].sum()),
                       "n_val_negative": int((Y[va] == 0).sum()),
                       "n_train_positive": int(Y[tr].sum()),
                       "n_groups_train": int(len(set(GROUPS[tr]))),
                       "n_groups_val": int(len(set(GROUPS[va])))})
COH_SAMPLE = H.Cohort(df=d, task="binary", label_col="y_bin", classes=[0, 1],
                      ras_positive=H.DEFAULT_RAS_POSITIVE, notes=dict(COH_SAMPLE.notes))
assert d.groupby("accession")["fold"].nunique().eq(1).all(), "accession spans folds"
assert d.groupby("fold")["y_bin"].sum().min() > 0
if SAMPLE_EXACT:  # [kaggle] only meaningful when the sampler hit 1890/187
    assert abs(d["y_bin"].mean() - 187 / 1890) < 1e-9
assert d.groupby(CFG.group_col)["fold"].nunique().eq(1).all(), "cluster leaks"
SPLIT = {"n_splits": CFG.n_splits, "seed": SEED, "group_col": CFG.group_col,
         "stratified_on": "y_bin", "per_fold": split_rows,
         "every_positive_in_exactly_one_fold": True,
         "no_cluster_group_spans_folds": True}
(RESULTS / "split.json").write_text(json.dumps(SPLIT, indent=2), encoding="utf-8")
print(pd.DataFrame(split_rows).to_string(index=False))

### feature registry

In [ ]:
SOURCES, COVERAGE = {}, {}
NSEQ = len(COH_SAMPLE.df)

def cache_path(name):
    # Hash the nt_sequence ordering so any change to the sample invalidates the cache
    sample_hash = hashlib.md5(
        "\n".join(COH_SAMPLE.df["nt_sequence"].astype(str)).encode()
    ).hexdigest()[:8]
    return RESULTS / f"{name}_n{NSEQ}_{sample_hash}.parquet"

def register(name, X, note=""):
    if X is None:
        COVERAGE[name] = {"coverage": 0.0, "status": "skipped", "note": note,
                          "n_features": 0, "imputed_rows": 0}
        print(f"  [{name}] skipped: {note}")
        return
    X = X.reset_index(drop=True)
    row_ok = X.notna().any(axis=1) & X.notna().mean(axis=1).ge(0.5)
    cov = float(row_ok.mean())
    imputed = int((~row_ok).sum() + (X.isna().any(axis=1) & row_ok).sum())
    Xf = X.fillna(X.median(numeric_only=True)).fillna(0.0).astype(np.float32)
    COVERAGE[name] = {"coverage": round(cov, 6), "status": "ok", "note": note,
                      "n_features": int(Xf.shape[1]), "imputed_rows": imputed,
                      "cache": str(cache_path(name))}
    SOURCES[name] = Xf
    try:
        Xf.to_parquet(cache_path(name))
    except Exception as exc:
        COVERAGE[name]["cache"] = f"unwritten: {type(exc).__name__}"
    print(f"  [{name}] d={Xf.shape[1]} coverage={cov:.3f} imputed_rows={imputed}")

def load_cached(name):
    p = cache_path(name)
    if p.exists():
        try:
            X = pd.read_parquet(p)
            if len(X) == NSEQ:
                return X
        except Exception:
            pass
    return None

with R.guard("features: SeqKmer3", REPORT):
    X = load_cached("SeqKmer3")
    if X is None:
        with R.Timer(COST, "features_seqkmer3", NSEQ):
            X = H.features_kmer(COH_SAMPLE, k=3)
    register("SeqKmer3", X)

with R.guard("features: ESM2", REPORT):
    X = load_cached("ESM2")
    if X is None and HAS_GPU:
        model_name = CFG.esm2_model
        if CFG.finetune_esm2:
            with R.Timer(COST, "continued_pretrain_esm2", len(COH_FULL.df)):
                model_name = H.continued_pretrain_esm2(
                    COH_FULL.df["aa_seq"].dropna().tolist(), model_name=CFG.esm2_model,
                    out_dir="esm2_finetuned", n_unfrozen_layers=CFG.ft_unfrozen_layers,
                    epochs=CFG.ft_epochs, mlm_prob=CFG.ft_mlm_prob,
                    batch_size=CFG.ft_batch_size, lr=CFG.ft_lr, max_len=CFG.esm2_max_len)
        ESM2_MODEL_USED = model_name
        with R.Timer(COST, "features_esm2", NSEQ):
            X = H.features_esm2_embeddings(COH_SAMPLE, model_name=model_name,
                                           max_len=CFG.esm2_max_len)
    elif X is None:
        decide("esm2_skipped", "no GPU", "ESM-2 embedding needs CUDA; logged skip")
    register("ESM2", X, "" if X is not None else "no GPU / not cached")

with R.guard("features: ESMFold", REPORT):
    X = load_cached("ESMFold")
    if X is None and CFG.run_esmfold and CFG.esmfold_live and HAS_GPU:
        with R.Timer(COST, "features_esmfold", NSEQ):
            X = H.features_esmfold(COH_SAMPLE, max_len=CFG.esmfold_max_len,
                                   chunk=CFG.esmfold_chunk)
    elif X is None:
        decide("esmfold_skipped", f"live={CFG.esmfold_live} gpu={HAS_GPU}",
               "ESMFold features are cache-only unless esmfold_live and a GPU")
    register("ESMFold", X, "" if X is not None else "cache-only, no cache present")

CF_STATUS = {"ran": False, "msa_mode": CFG.cf_msa_mode,
             "num_recycle": CFG.cf_num_recycle, "num_models": CFG.cf_num_models,
             "truncate_aa": CFG.cf_truncate_aa, "target_coverage": CFG.cf_target_coverage,
             "timeout_min": CFG.cf_timeout_min}
with R.guard("features: ColabFold", REPORT):
    X = load_cached("ColabFold")
    if X is None and CFG.run_colabfold and HAS_GPU:
        cf_dir = pathlib.Path("colabfold_out"); cf_dir.mkdir(exist_ok=True)
        COH_SAMPLE.df["cf_name"] = R.assign_cf_names(COH_SAMPLE, max_len=CFG.cf_truncate_aa)
        uniq = COH_SAMPLE.df.drop_duplicates("cf_name")
        if CFG.cf_max_seqs:
            uniq = uniq.head(CFG.cf_max_seqs)
        fasta = pathlib.Path("cf_input.fasta")
        fasta.write_text("".join(f">{r.cf_name}\n{str(r.aa_seq)[:CFG.cf_truncate_aa]}\n"
                                 for r in uniq.itertuples()), encoding="utf-8")
        cmd = ["colabfold_batch", str(fasta), str(cf_dir),
               "--msa-mode", CFG.cf_msa_mode,
               "--num-recycle", str(CFG.cf_num_recycle),
               "--num-models", str(CFG.cf_num_models)]
        sampler = R.GpuSampler()
        try:
            with sampler, R.Timer(COST, "colabfold_batch", len(uniq), vram_fn=lambda: sampler.peak_gb):
                subprocess.run(cmd, timeout=CFG.cf_timeout_min * 60, check=False)
            CF_STATUS["ran"] = True
            CF_STATUS["n_folded_requested"] = int(len(uniq))
        except subprocess.TimeoutExpired:
            decide("colabfold_timeout", f"{CFG.cf_timeout_min} min",
                   "wall-clock cap hit; whatever folded so far is parsed")
            CF_STATUS["ran"] = True
            CF_STATUS["timed_out"] = True
        X = H.features_colabfold_parse(COH_SAMPLE, str(cf_dir))
        CF_STATUS["plddt"] = R.plddt_summary(str(cf_dir))
        CF_STATUS["coverage_from_plddt_mean"] = float(X["plddt_mean"].notna().mean())
    elif X is None:
        decide("colabfold_skipped", f"run={CFG.run_colabfold} gpu={HAS_GPU}",
               "AlphaFold2 folding exceeds the fast-mode budget; logged skip")
    if X is not None and "plddt_mean" in X.columns:
        CF_STATUS["coverage_from_plddt_mean"] = float(X["plddt_mean"].notna().mean())
        if CF_STATUS["coverage_from_plddt_mean"] < CFG.cf_target_coverage:
            decide("colabfold_coverage_below_target",
                   round(CF_STATUS["coverage_from_plddt_mean"], 4),
                   f"below cf_target_coverage={CFG.cf_target_coverage}")
    register("ColabFold", X, "" if X is not None else "not folded in this run")
COVERAGE["_colabfold_run"] = CF_STATUS

if "ESM2" in SOURCES and "ESMFold" in SOURCES:
    register("ESM2+ESMFold", pd.concat([SOURCES["ESM2"], SOURCES["ESMFold"]], axis=1))
(RESULTS / "coverage.json").write_text(json.dumps(COVERAGE, indent=2, default=str), encoding="utf-8")
print("[features] sources in grid:", list(SOURCES))
assert SOURCES, "no feature source survived"

### Section 3.8 — reference baselines

`ram_indicator` builds the six-element vector over the canonical resistance coordinates NS5A 28, 30, 31, 93 and NS5B 282, 316. `metadata_features` builds the 21-feature metadata baseline. Both are reported in Section 3.8 and evaluated in Sections 4.5 and 4.6.

In [ ]:
RAM_POSITIONS = {"NS5A": (28, 30, 31, 93), "NS5B": (282, 316)}

def ram_indicator(cohort):
    seqs = cohort.df["aa_seq"].astype(str)
    cols, meta = {}, {"consensus": {}, "short": {}, "positions": RAM_POSITIONS}
    for prot, positions in RAM_POSITIONS.items():
        for p in positions:
            res = seqs.map(lambda s, p=p: s[p - 1] if len(s) >= p else None)
            present = res.dropna()
            consensus = present.value_counts().idxmax() if len(present) else None
            name = f"ram_{prot.lower()}_{p}"
            cols[name] = (res.notna() & (res != consensus)).astype(np.float32)
            meta["consensus"][name] = consensus
            meta["short"][name] = int(res.isna().sum())
    X = pd.DataFrame(cols, index=cohort.df.index)
    assert X.shape[1] == 6
    return X, meta

def metadata_features(cohort):
    d = cohort.df
    out = pd.DataFrame(index=d.index)
    out["meta_length"] = (d["seq_len"] if "seq_len" in d
                          else d["nt_sequence"].astype(str).str.len()).astype(np.float32)
    year = pd.to_numeric(d.get("Collection_Date", pd.Series(index=d.index, dtype=object))
                         .astype(str).str.extract(r"(\d{4})")[0], errors="coerce")
    out["meta_year"] = year.fillna(year.median()).astype(np.float32)
    for col, prefix in (("Submitters", "meta_lab"), ("Geo_Location", "meta_region")):
        if col in d.columns:
            codes = d[col].astype(str).fillna("NA")
            top = codes.value_counts().head(15).index
            for v in top:
                out[f"{prefix}_{re.sub(r'[^A-Za-z0-9]+', '_', str(v))[:24]}"] = (codes == v).astype(np.float32)
    return out.astype(np.float32)

RAM_X, RAM_META = ram_indicator(COH_SAMPLE)
META_X = metadata_features(COH_SAMPLE)
assert len(RAM_X) == len(META_X) == len(COH_SAMPLE.df)
print(f"[baselines] RAM {RAM_X.shape} metadata {META_X.shape}")

### caching

In [ ]:
import hashlib, shutil

# The grid cache was keyed on tag+source+clf only, so a broken pickled result
# from a previous run was loaded verbatim. Make the key depend on the sample.
sample_hash = hashlib.md5(
    "\n".join(COH_SAMPLE.df["nt_sequence"].astype(str)).encode()
).hexdigest()[:8]

CACHE_DIR = RESULTS / "grid_cache" / sample_hash
shutil.rmtree(RESULTS / "grid_cache", ignore_errors=True)   # one-shot wipe
CACHE_DIR.mkdir(parents=True, exist_ok=True)

import inspect
THRESHOLD_PROVENANCE = {"checked": False, "ok": None, "failures": []}
if CFG.THRESHOLD_MODE == "youden_oof":
    THRESHOLD_PROVENANCE["checked"] = True
    try:
        sig_inner  = inspect.signature(R._inner_threshold).parameters
        src_inner  = inspect.getsource(R._inner_threshold)
        src_eval   = inspect.getsource(R.evaluate_balanced)
        checks = {
            "cfg.tune_threshold is True": R.CFG.tune_threshold is True,
            "_inner_threshold takes tr_idx": "tr_idx" in sig_inner,
            "inner split is over tr_idx":
                "inner.split(tr_idx" in src_inner.replace(" ", ""),
            "evaluate_balanced passes tr":
                "_inner_threshold(cohort, Xv, tr," in src_eval.replace("  ", " "),
            "_inner_threshold never sees va": "va" not in sig_inner,
        }
        THRESHOLD_PROVENANCE["failures"] = [k for k, v in checks.items() if not v]
        THRESHOLD_PROVENANCE["ok"] = not THRESHOLD_PROVENANCE["failures"]
    except Exception as exc:
        THRESHOLD_PROVENANCE["ok"] = False
        THRESHOLD_PROVENANCE["failures"] = [f"introspection failed: {type(exc).__name__}"]
    if THRESHOLD_PROVENANCE["ok"]:
        print("[threshold] youden_oof verified")
    else:
        print("[threshold] WARNING:", THRESHOLD_PROVENANCE["failures"])

GRID_SOURCES = dict(SOURCES)
GRID_SOURCES["RAM"]      = RAM_X
GRID_SOURCES["Metadata"] = META_X

with R.Timer(COST, "grid_total", NSEQ):
    GRID, STORE = R.run_revision_grid(
        COH_SAMPLE, GRID_SOURCES, classifiers=CFG.classifiers, cfg=R.CFG,
        balance=(CFG.CLASS_WEIGHT == "balanced"),
        tune_threshold=(CFG.THRESHOLD_MODE == "youden_oof"),
        log=COST, cache_dir=str(CACHE_DIR), tag=f"{MODE}_{sample_hash}")

assert len(STORE), "the grid produced no fitted configuration"
empty = GRID.loc[GRID["n_eval"] == 0, ["features", "classifier"]].values.tolist()
assert not empty, (
    "grid configurations with empty OOF — the sampler did not preserve real rows. "
    f"Offenders: {empty}"
)
print(f"[grid] {len(STORE)} configuration(s), n_eval={int(GRID['n_eval'].iloc[0])} "
      f"across every config")
print(GRID.head(12).to_string(index=False))

### Statistics used in Sections 4.5–4.12

Wilson intervals, accession-clustered bootstrap intervals and the per-metric packs used throughout the results.

In [ ]:
from scipy import stats as _st
from sklearn.metrics import f1_score, matthews_corrcoef, roc_auc_score, average_precision_score

def wilson(k, n):
    return tuple(round(v, 6) for v in H.wilson_ci(int(k), int(n)))

def _metric_pack(y, pred, p):
    return R._binary_metrics(y, pred, p)

_CI_KEYS = ("auroc", "pr_auc", "f1", "mcc", "balanced_acc",
            "sensitivity", "specificity", "accuracy")

def cluster_bootstrap_ci(y, pred, p, accessions, n_boot=1000, seed=42):
    y = np.asarray(y); pred = np.asarray(pred); p = np.asarray(p); accessions = np.asarray(accessions)
    # Graceful no-op for empty / mismatched inputs (avoids the IndexError)
    if len(y) == 0 or len(y) != len(accessions):
        return {f"{k}_ci95": (np.nan, np.nan) for k in _CI_KEYS}

    rng = np.random.default_rng(seed)
    uniq = np.unique(accessions)
    draws = {k: [] for k in _CI_KEYS}
    for _ in range(n_boot):
        take = rng.choice(uniq, len(uniq), replace=True)
        idx = np.concatenate([np.where(accessions == a)[0] for a in take])
        yv = y[idx]
        if len(np.unique(yv)) < 2:
            continue
        m = _metric_pack(yv, pred[idx], p[idx])
        for k in _CI_KEYS:
            if not np.isnan(m[k]):
                draws[k].append(m[k])
    out = {}
    for k in _CI_KEYS:
        v = draws[k]
        out[f"{k}_ci95"] = (round(float(np.percentile(v, 2.5)), 4),
                            round(float(np.percentile(v, 97.5)), 4)) if v else (np.nan, np.nan)
    return out

def metric_row(name, clf, r):
    y, pred, p = r["oof_true"], r["oof_pred"], r["oof_p"]
    pf = r["per_fold"]
    tp, fp, fn, tn = (r["metrics_oof"][k] for k in ("tp", "fp", "fn", "tn"))
    acc_lo, acc_hi = wilson(tp + tn, len(y))
    return {
        "features": name, "classifier": clf,
        "n_eval": int(len(y)), "n_pos_total": int((y == 1).sum()) if len(y) else 0,
        "accuracy": float((pred == y).mean()) if len(y) else float("nan"),
        "accuracy_ci_lo": acc_lo, "accuracy_ci_hi": acc_hi,
        "auroc": float(r["metrics_oof"]["auroc"]),
        "pr_auc": float(r["metrics_oof"]["pr_auc"]),
        "f1": float(r["metrics_oof"]["f1"]),
        "mcc": float(r["metrics_oof"]["mcc"]),
        "balanced_acc": float(r["metrics_oof"]["balanced_acc"]),
        "sensitivity": float(r["metrics_oof"]["sensitivity"]),
        "specificity": float(r["metrics_oof"]["specificity"]),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
        "n_pos_per_fold": ";".join(str(int(v)) for v in pf.get("n_pos_val", [])),
        "auroc_foldmean": float(pf["auroc"].mean()) if "auroc" in pf and len(pf) else np.nan,
        "auroc_foldsd":   float(pf["auroc"].std())  if "auroc" in pf and len(pf) else np.nan,
    }

accessions_full = (COH_SAMPLE.df["accession"].to_numpy()
                   if "accession" in COH_SAMPLE.df.columns else None)

rows = []
for (s, c), r in STORE.items():
    row = metric_row(s, c, r)
    y, pred, p = r["oof_true"], r["oof_pred"], r["oof_p"]

    # Restrict the accession array to the evaluated (real) rows only
    mask = r.get("oof_mask")
    if accessions_full is not None and mask is not None and len(mask) == len(accessions_full):
        eval_accessions = accessions_full[np.asarray(mask, dtype=bool)]
    else:
        eval_accessions = accessions_full

    row.update(cluster_bootstrap_ci(y, pred, p, eval_accessions,
                                    n_boot=CFG.n_boot, seed=SEED))
    for k, v in R.matched_operating_points(y, p, R.CFG).items():
        row[k] = float(v)
    rows.append(row)

METRICS = pd.DataFrame(rows).sort_values("auroc", ascending=False).reset_index(drop=True)
print(METRICS[["features","classifier","auroc","f1","mcc","sensitivity","specificity"]]
      .head(12).to_string(index=False))

### Sections 4.5–4.12 — the reported analyses

Each `item_*.json` written here corresponds to a reported result.

In [ ]:
ITEMS = {}

def emit(n, status, **kw):
    doc = {"item": int(n), "status": status, "mode": MODE,
           "n": kw.pop("n", int(len(COH_SAMPLE.df))),
           "metric_table": kw.pop("metric_table", None),
           "mapping_table": kw.pop("mapping_table", None),
           "slots": kw.pop("slots", {})}
    doc.update(kw)
    ITEMS[n] = doc
    (RESULTS / f"item_{n}.json").write_text(json.dumps(doc, indent=2, default=str), encoding="utf-8")
    print(f"[item {n}] {status}")
    return doc

def _tbl(frame):
    return json.loads(frame.to_json(orient="records")) if frame is not None and len(frame) else []

def _best(sub):
    return sub.sort_values("auroc", ascending=False).iloc[0] if len(sub) else None

def item_1():
    sub = METRICS[METRICS.features == "RAM"]
    if not len(sub):
        return emit(1, "skipped", note="RAM source absent")
    best_ram = _best(sub)
    others = METRICS[~METRICS.features.isin(["RAM", "Metadata"])]
    best_model = _best(others)
    delta = (float(best_model.auroc - best_ram.auroc) if best_model is not None else None)
    return emit(1, "ok", metric_table=_tbl(sub),
                slots={"auroc_ram": round(float(best_ram.auroc), 3),
                       "ram_classifier": best_ram.classifier,
                       "best_model": f"{best_model.features}+{best_model.classifier}" if best_model is not None else None,
                       "auroc_best_model": round(float(best_model.auroc), 3) if best_model is not None else None,
                       "delta_over_ram": round(delta, 3) if delta is not None else None},
                ram_consensus=RAM_META["consensus"], ram_short_sequences=RAM_META["short"])

def item_2():
    variants, rows = {}, []
    region_cols = [c for c in META_X.columns if c.startswith("meta_region")]
    variants["all"] = META_X
    variants["minus_region"] = META_X.drop(columns=region_cols)
    variants["region_only"] = META_X[region_cols] if region_cols else None
    for vname, Xv in variants.items():
        if Xv is None or not Xv.shape[1]:
            rows.append({"variant": vname, "status": "no columns"}); continue
        g, st = R.run_revision_grid(COH_SAMPLE, {f"Metadata[{vname}]": Xv},
                                    classifiers=CFG.classifiers, cfg=R.CFG,
                                    balance=True, tune_threshold=True)
        for (s, c), r in st.items():
            row = metric_row(s, c, r); row["variant"] = vname
            rows.append(row)
    frame = pd.DataFrame(rows)
    num = frame[frame.get("auroc").notna()] if "auroc" in frame else frame
    return emit(2, "ok", metric_table=_tbl(frame),
                slots={"auroc_metadata_all": round(float(num[num.variant == "all"].auroc.max()), 3) if len(num[num.variant == "all"]) else None,
                       "auroc_metadata_region_only": round(float(num[num.variant == "region_only"].auroc.max()), 3) if len(num[num.variant == "region_only"]) else None})

def item_3():
    d = COH_SAMPLE.df
    rule, mask = None, None
    year = pd.to_numeric(d.get("Collection_Date", pd.Series(index=d.index, dtype=object))
                         .astype(str).str.extract(r"(\d{4})")[0], errors="coerce")
    if year.notna().sum() >= 0.5 * len(d):
        cut = year.quantile(1 - CFG.holdout_year_quintile)
        cand = year >= cut
        if int(d.loc[cand, "y_bin"].sum()) >= CFG.holdout_min_positives:
            rule, mask = f"top_quintile_year(>= {cut:.0f})", cand
    if mask is None and "Submitters" in d.columns:
        for lab, cnt in d["Submitters"].value_counts().items():
            cand = d["Submitters"] == lab
            if int(d.loc[cand, "y_bin"].sum()) >= CFG.holdout_min_positives and cnt < 0.5 * len(d):
                rule, mask = f"single_held_out_lab({str(lab)[:40]})", cand
                break
    if mask is None:
        order = d["accession"].astype(str).sort_values().index
        cand = pd.Series(False, index=d.index); cand.loc[order[-max(1, len(d) // 5):]] = True
        rule, mask = "disjoint_accession_block", cand
        decide("item3_rule", rule, "no year or lab split carried enough positives")
    te = np.where(mask.to_numpy())[0]
    te_clusters = set(d.iloc[te][CFG.group_col])
    te = np.where(d[CFG.group_col].isin(te_clusters).to_numpy())[0]
    tr = np.setdiff1d(np.arange(len(d)), te)
    assert not (set(d.iloc[tr][CFG.group_col]) & set(d.iloc[te][CFG.group_col]))
    rows = []
    for sname, Xs in GRID_SOURCES.items():
        for clf in CFG.classifiers:
            try:
                from sklearn.preprocessing import StandardScaler
                Xv = Xs.to_numpy(dtype=np.float32)
                sc = StandardScaler().fit(Xv[tr])
                m = R._fit_balanced(clf, sc.transform(Xv[tr]), d["y_bin"].to_numpy()[tr], SEED)
                p = m.predict_proba(sc.transform(Xv[te]))[:, list(m.classes_).index(1)]
                thr = R.pick_threshold(d["y_bin"].to_numpy()[tr],
                                       m.predict_proba(sc.transform(Xv[tr]))[:, list(m.classes_).index(1)],
                                       R.CFG.threshold_metric, R.CFG.threshold_grid)
                mm = R._binary_metrics(d["y_bin"].to_numpy()[te], (p >= thr).astype(int), p)
                mm.update(features=sname, classifier=clf, threshold=round(float(thr), 4))
                rows.append(mm)
            except Exception as exc:
                rows.append({"features": sname, "classifier": clf,
                             "status": f"failed: {type(exc).__name__}"})
    frame = pd.DataFrame(rows)
    return emit(3, "ok", metric_table=_tbl(frame), rule_fired=rule,
                n_train=int(len(tr)), n_test=int(len(te)), n_test_positive=int(d["y_bin"].to_numpy()[te].sum()),
                cluster_overlap=0, slots={"auroc_independent_best": round(float(frame["auroc"].max()), 3) if "auroc" in frame else None})

def item_4():
    subs = R.per_protein_cohorts(COH_SAMPLE, R.CFG)
    rows, notes = [], {}
    for pname, sub in subs.items():
        n_pos = int(sub.df[sub.label_col].sum())
        notes[pname] = {"n": int(len(sub.df)), "n_pos": n_pos}
        if n_pos < CFG.n_splits or len(sub.df) < 2 * CFG.n_splits:
            notes[pname]["status"] = "too few positives to split"
            continue
        # Rebuild the feature subset by the SAME boolean mask per_protein_cohorts
        # applied, so sub.df.iloc[i] and src[k].iloc[i] correspond row-for-row.
        flag = {"NS5A": "has_ns5a", "NS5B": "has_ns5b", "NS3": "has_ns3"}[pname]
        mask_arr = COH_SAMPLE.df[flag].astype(bool).to_numpy()
        src = {k: v[mask_arr].reset_index(drop=True) for k, v in GRID_SOURCES.items()}
        g, st = R.run_revision_grid(sub, src, classifiers=CFG.classifiers, cfg=R.CFG,
                                    balance=True, tune_threshold=True, tag=f"{MODE}_{pname}")
        for (s, c), r in st.items():
            row = metric_row(s, c, r); row["protein"] = pname
            rows.append(row)
    frame = pd.DataFrame(rows)
    pooled = METRICS.assign(protein="pooled")
    return emit(4, "ok", metric_table=_tbl(pd.concat([frame, pooled], ignore_index=True)), subsets=notes,
                truncation_note={"cf_truncate_aa": CFG.cf_truncate_aa, "ns5b_typical_aa": 591,
                                 "ns5b_residues_lost": 591 - CFG.cf_truncate_aa},
                slots={"auroc_ns5a_best": round(float(frame[frame.protein.str.contains("ns5a", case=False)].auroc.max()), 3) if len(frame) and frame.protein.str.contains("ns5a", case=False).any() else None,
                       "auroc_ns5b_best": round(float(frame[frame.protein.str.contains("ns5b", case=False)].auroc.max()), 3) if len(frame) and frame.protein.str.contains("ns5b", case=False).any() else None})
    
def item_5():
    mapping_rows, rows = [], []
    mappings = {"default": H.DEFAULT_RAS_POSITIVE,
                "strict": ("reduced_susceptibility", "resistant"),
                "STRICT_resistant_only": ("resistant",)}
    lab = COH_SAMPLE.df["label"]
    for mname, classes in mappings.items():
        counts = lab.value_counts().reindex(H.LABELS_4).fillna(0).astype(int)
        y2 = lab.isin(classes).astype(int).to_numpy()
        for cat in H.LABELS_4:
            mapping_rows.append({"mapping": mname, "category": cat,
                                 "n": int(counts.get(cat, 0)), "maps_to": int(cat in classes)})
        if len(np.unique(y2)) < 2 or int(y2.sum()) < CFG.n_splits:
            rows.append({"mapping": mname, "status": "too few positives to split", "n_pos": int(y2.sum())})
            continue
        sub = H.Cohort(df=COH_SAMPLE.df.assign(**{COH_SAMPLE.label_col: y2}), task="binary",
                       label_col=COH_SAMPLE.label_col, classes=[0, 1],
                       ras_positive=tuple(classes), notes=dict(COH_SAMPLE.notes))
        g, st = R.run_revision_grid(sub, GRID_SOURCES, classifiers=CFG.classifiers,
                                    cfg=R.CFG, balance=True, tune_threshold=True, tag=f"{MODE}_{mname}")
        for (s, c), r in st.items():
            row = metric_row(s, c, r); row["mapping"] = mname
            rows.append(row)
    frame = pd.DataFrame(rows)
    strict = frame[frame.get("mapping") == "STRICT_resistant_only"] if "mapping" in frame else frame
    return emit(5, "exploratory", metric_table=_tbl(frame), mapping_table=mapping_rows,
                header="9 accessions / 47 sequences",
                slots={"auroc_strict_best": round(float(strict["auroc"].max()), 3) if "auroc" in strict and strict["auroc"].notna().any() else None,
                       "n_pos_strict": int((lab == "resistant").sum())})

def item_6():
    rows = []
    arms = {"class_weights": dict(balance=True, tune_threshold=False),
            "threshold_tuning": dict(balance=False, tune_threshold=True),
            "random_replicate": None}
    for arm, kw in arms.items():
        if arm == "random_replicate":
            rng = np.random.default_rng(SEED)
            y = COH_SAMPLE.df["y_bin"].to_numpy()
            pos = np.where(y == 1)[0]; neg = np.where(y == 0)[0]
            extra = rng.choice(pos, max(0, len(neg) - len(pos)), replace=True)
            idx = np.concatenate([np.arange(len(COH_SAMPLE.df)), extra])
            sub = H.Cohort(df=COH_SAMPLE.df.iloc[idx].reset_index(drop=True), task="binary",
                           label_col=COH_SAMPLE.label_col, classes=[0, 1],
                           ras_positive=COH_SAMPLE.ras_positive, notes=dict(COH_SAMPLE.notes))
            src = {k: v.iloc[idx].reset_index(drop=True) for k, v in GRID_SOURCES.items()}
            g, st = R.run_revision_grid(sub, src, classifiers=CFG.classifiers, cfg=R.CFG,
                                        balance=False, tune_threshold=False, tag=f"{MODE}_replicate")
        else:
            g, st = R.run_revision_grid(COH_SAMPLE, GRID_SOURCES, classifiers=CFG.classifiers,
                                        cfg=R.CFG, tag=f"{MODE}_{arm}", **kw)
        for (s, c), r in st.items():
            row = metric_row(s, c, r); row["arm"] = arm
            rows.append(row)
    frame = pd.DataFrame(rows)
    return emit(6, "ok", metric_table=_tbl(frame),
                slots={f"auroc_{a}": round(float(frame[frame.arm == a].auroc.max()), 3) if len(frame[frame.arm == a]) else None for a in arms})

def item_7():
    if "ColabFold" not in SOURCES:
        return emit(7, "skipped", note="ColabFold features absent")
    X_abl = R.ablate_colabfold_ras(SOURCES["ColabFold"], R.CFG)
    dropped = [c for c in SOURCES["ColabFold"].columns if c not in X_abl.columns]
    g, st = R.run_revision_grid(COH_SAMPLE, {"ColabFold_noRAS": X_abl},
                                classifiers=CFG.classifiers, cfg=R.CFG,
                                balance=True, tune_threshold=True, tag=f"{MODE}_abl")
    rows = [metric_row(s, c, r) for (s, c), r in st.items()]
    base = METRICS[METRICS.features == "ColabFold"]
    frame = pd.concat([base, pd.DataFrame(rows)], ignore_index=True)
    a_with = float(base.auroc.max()) if len(base) else np.nan
    a_without = float(pd.DataFrame(rows).auroc.max()) if rows else np.nan
    return emit(7, "ok", metric_table=_tbl(frame), dropped_columns=dropped,
                slots={"auroc_colabfold_with_plddt_ras": round(a_with, 3),
                       "auroc_colabfold_without_plddt_ras": round(a_without, 3),
                       "ablation_delta": round(a_with - a_without, 3)})

SLOT8 = "The corpus and cohort derive from the same accessions and are not independent by construction."

def item_8():
    real = COH_SAMPLE.df
    corpus = COH_FULL.df[~COH_FULL.df["eval_eligible"]]
    def h(s): return hashlib.sha256(str(s).upper().encode()).hexdigest()
    eval_h = {h(s) for s in real["nt_sequence"]}
    corp_h = {h(s) for s in corpus["nt_sequence"]}
    inter = eval_h & corp_h
    a = {"exact_sha256": {"n_eval": len(eval_h), "n_corpus": len(corp_h), "n_shared": len(inter),
                          "frac_of_eval_in_corpus": round(len(inter) / max(len(eval_h), 1), 6),
                          "frac_of_corpus_in_eval": round(len(inter) / max(len(corp_h), 1), 6)}}
    b = {}
    if HAS_CDHIT:
        import tempfile
        def fasta(frame, path, col="nt_sequence"):
            with open(path, "w") as fh:
                for i, r in enumerate(frame.itertuples()):
                    fh.write(f">s{i}\n{getattr(r, col)}\n")
            return path
        with tempfile.TemporaryDirectory() as td:
            fe = fasta(real, os.path.join(td, "eval.fa"))
            fc = fasta(corpus.head(20000), os.path.join(td, "corpus.fa"))
            for ident in CFG.cdhit_identities:
                for tag, q, s in (("eval_vs_corpus", fe, fc), ("corpus_vs_eval", fc, fe)):
                    out = os.path.join(td, f"{tag}_{ident}")
                    rc = subprocess.run(["cd-hit-est-2d", "-i", s, "-i2", q, "-o", out,
                                         "-c", str(ident), "-n", "8", "-M", "2000", "-T", "0"],
                                        capture_output=True, text=True).returncode
                    nclust = (sum(1 for ln in open(out + ".clstr") if ln.startswith(">"))
                              if os.path.exists(out + ".clstr") else None)
                    b[f"{tag}@{ident}"] = {"returncode": rc, "clusters": nclust}
    else:
        decide("cdhit_absent", "skipped", "cd-hit-est is not on PATH; k-mer Jaccard audit stands in")
        b["status"] = "cd-hit unavailable"
    c = H.leakage_audit(COH_SAMPLE, n_splits=CFG.n_splits, seed=SEED, group_col=CFG.group_col)
    c["no_cluster_spans_folds"] = SPLIT["no_cluster_group_spans_folds"]
    assert c["group_disjoint"]
    try:
        rep = R.overlap_report(COH_FULL.df, COH_SAMPLE, R.CFG)
    except Exception as exc:
        rep = {"status": f"failed: {type(exc).__name__}"}
    frac = a["exact_sha256"]["frac_of_eval_in_corpus"]
    purge = frac > CFG.overlap_purge_threshold
    d = {"overlap_fraction": frac, "threshold": CFG.overlap_purge_threshold,
         "purge_required": bool(purge),
         "overlapping_accessions": sorted(real.loc[[h(s) in inter for s in real["nt_sequence"]], "accession"]
                                          .astype(str).tolist())[:500]}
    if purge:
        CFG.PURGE_OVERLAP = True
        if MODE == "full" and HAS_GPU and CFG.finetune_esm2:
            d["purge_run"] = "continued pretraining re-run on the purged corpus"
            keep = COH_FULL.df[~COH_FULL.df["nt_sequence"].map(lambda s: h(s) in inter)]
            with R.Timer(COST, "continued_pretrain_purged", len(keep)):
                d["purged_model_dir"] = H.continued_pretrain_esm2(
                    keep["aa_seq"].dropna().tolist(), model_name=CFG.esm2_model,
                    out_dir="esm2_finetuned_purged", n_unfrozen_layers=CFG.ft_unfrozen_layers,
                    epochs=CFG.ft_epochs)
        else:
            d["purge_run"] = "deferred"
            decide("item8_purge_deferred", f"mode={MODE} gpu={HAS_GPU}", "purged re-run needs GPU and full mode")
    return emit(8, "concession", exact=a, cdhit=b, within_cohort=c, overlap=d,
                repo_overlap_report=rep, statement=SLOT8,
                slots={"item8_statement": SLOT8, "overlap_fraction": round(frac, 6),
                       "max_train_val_identity": round(float(c["max_train_val_identity"]), 3),
                       "purge_required": bool(purge)})

def _delong(y, p1, p2):
    from scipy.stats import norm
    y = np.asarray(y)
    if len(np.unique(y)) < 2:
        return float("nan"), float("nan")
    pos = p1[y == 1]; neg1 = p1[y == 0]; pos2 = p2[y == 1]; neg2 = p2[y == 0]
    m, n = len(pos), len(neg1)
    def midrank(a):
        return _st.rankdata(a, "average")
    tx1, tz1 = midrank(pos), midrank(neg1)
    txz1 = midrank(np.r_[pos, neg1])
    v01_1 = (txz1[:m] - tx1) / n
    v10_1 = 1.0 - (txz1[m:] - tz1) / m
    tx2, tz2 = midrank(pos2), midrank(neg2)
    txz2 = midrank(np.r_[pos2, neg2])
    v01_2 = (txz2[:m] - tx2) / n
    v10_2 = 1.0 - (txz2[m:] - tz2) / m
    a1 = v01_1.mean(); a2 = v01_2.mean()
    s01 = np.cov(np.vstack([v01_1, v01_2]))
    s10 = np.cov(np.vstack([v10_1, v10_2]))
    S = s01 / m + s10 / n
    var = S[0, 0] + S[1, 1] - 2 * S[0, 1]
    if var <= 0:
        return a1 - a2, float("nan")
    z = (a1 - a2) / math.sqrt(var)
    return a1 - a2, 2 * norm.sf(abs(z))

def item_9():
    rows = []
    ref_keys = [k for k in STORE if k[0] == "RAM"]
    ref = ref_keys[0] if ref_keys else None
    for key, r in STORE.items():
        if ref is None or key == ref:
            continue
        a, b = STORE[ref], r
        pf_a = a["per_fold"].set_index("fold")["auroc"]
        pf_b = b["per_fold"].set_index("fold")["auroc"]
        common = pf_a.index.intersection(pf_b.index)
        try:
            w = _st.wilcoxon(pf_b.loc[common], pf_a.loc[common])
            wp, ws = float(w.pvalue), float(w.statistic)
        except Exception:
            wp, ws = float("nan"), float("nan")
        delta, dp = _delong(a["oof_true"], b["oof_p"], a["oof_p"])
        rows.append({"comparison": f"{key[0]}+{key[1]} vs RAM+{ref[1]}", "contrast": "vs_RAM",
                     "wilcoxon_stat": ws, "wilcoxon_p": wp, "delong_p": dp, "auroc_delta": delta,
                     "n_folds": int(len(common))})
    for clf in CFG.classifiers:
        ka, kb = ("ColabFold", clf), ("SeqKmer3", clf)
        if ka in STORE and kb in STORE:
            a, b = STORE[ka], STORE[kb]
            pf_a = a["per_fold"].set_index("fold")["auroc"]
            pf_b = b["per_fold"].set_index("fold")["auroc"]
            common = pf_a.index.intersection(pf_b.index)
            try:
                w = _st.wilcoxon(pf_a.loc[common], pf_b.loc[common])
                wp, ws = float(w.pvalue), float(w.statistic)
            except Exception:
                wp, ws = float("nan"), float("nan")
            delta, dp = _delong(a["oof_true"], a["oof_p"], b["oof_p"])
            rows.append({"comparison": f"ColabFold+{clf} vs SeqKmer3+{clf}",
                         "contrast": "colabfold_vs_seqkmer3",
                         "wilcoxon_stat": ws, "wilcoxon_p": wp, "delong_p": dp,
                         "auroc_delta": delta, "n_folds": int(len(common))})
    frame = pd.DataFrame(rows)
    if len(frame):
        frame["wilcoxon_p_holm"] = R._holm(frame["wilcoxon_p"].to_numpy())
        frame["delong_p_holm"] = R._holm(frame["delong_p"].to_numpy())
    return emit(9, "ok", metric_table=_tbl(frame),
                slots={"n_comparisons": int(len(frame)),
                       "n_significant_holm": int((frame["delong_p_holm"] < CFG.alpha).sum()) if len(frame) else 0})

def item_10():
    ops = R.operating_point_table(STORE, R.CFG)
    curves = R.pr_curve_table(STORE)
    if len(curves):
        curves.to_csv(RESULTS / "pr_curves.csv", index=False)
    return emit(10, "ok", metric_table=_tbl(ops), pr_curve_csv=str(RESULTS / "pr_curves.csv"),
                slots={"n_configs": int(len(ops))})

def item_11():
    cols = ["features", "classifier", "n_eval", "n_pos_total", "n_pos_per_fold",
            "accuracy", "accuracy_ci_lo", "accuracy_ci_hi", "auroc",
            "auroc_ci95", "pr_auc", "pr_auc_ci95", "f1", "f1_ci95", "mcc", "mcc_ci95",
            "balanced_acc", "balanced_acc_ci95", "sensitivity", "sensitivity_ci95",
            "specificity", "specificity_ci95"]
    use = [c for c in cols if c in METRICS.columns]
    frame = METRICS[use]
    pf = R.per_fold_table(STORE)
    pf.to_csv(RESULTS / "per_fold.csv", index=False)
    return emit(11, "ok", metric_table=_tbl(frame), per_fold_csv=str(RESULTS / "per_fold.csv"),
                slots={"macro_f1_ci_width": None, "mcc_ci_width": None})

def item_12():
    tbl = COST.table()
    if len(tbl):
        tbl = tbl.replace({np.nan: None})
    return emit(12, "ok" if len(tbl) else "empty", metric_table=_tbl(tbl),
                slots={"n_stages": int(len(tbl)),
                       "peak_vram_gb": float(np.nanmax(COST.table()["peak_VRAM_GB"])) if len(tbl) else None})

def item_20():
    counts = R.cohort_counts(COH_FULL.df, H.DEFAULT_RAS_POSITIVE)
    mapping = ITEMS.get(5, {}).get("mapping_table")
    return emit(20, "ok", metric_table=[counts], mapping_table=mapping,
                slots={"n_susceptible": counts["per_category"].get("susceptible"),
                       "n_polymorphism": counts["per_category"].get("polymorphism"),
                       "n_reduced": counts["per_category"].get("reduced_susceptibility"),
                       "n_resistant": counts["per_category"].get("resistant")})

def item_21():
    return emit(21, "ok", metric_table=[{
        "grouping": "single-linkage k-mer Jaccard clusters over real sequences",
        "k": CFG.cluster_k, "threshold": CFG.cluster_threshold, "group_col": CFG.group_col,
        "n_accessions": int(COH_SAMPLE.df["accession"].nunique()),
        "n_cluster_groups": int(COH_SAMPLE.df[CFG.group_col].nunique()),
        "merged_by_clustering": int(COH_SAMPLE.df["accession"].nunique() - COH_SAMPLE.df[CFG.group_col].nunique()),
        "splitter": "StratifiedGroupKFold", "n_splits": CFG.n_splits, "seed": SEED}],
        slots={"n_cluster_groups": int(COH_SAMPLE.df[CFG.group_col].nunique())})

def item_22():
    man = R.reproducibility_manifest(COH_SAMPLE, R.CFG, extra={"notebook_manifest": MANIFEST})
    acc = COH_SAMPLE.df[["accession", CFG.group_col, COH_SAMPLE.label_col]]
    acc.to_csv(RESULTS / "accessions.csv", index=False)
    R.save_manifest(man, str(RESULTS / "reproducibility_manifest.json"))
    return emit(22, "ok", metric_table=[{"accessions_csv": str(RESULTS / "accessions.csv"),
                                         "n_accessions": int(len(acc)), "seed": SEED,
                                         "manifest": str(RESULTS / "manifest.json")}],
                slots={"seed": SEED, "n_accessions": int(len(acc))})

for fn in (item_1, item_2, item_3, item_4, item_5, item_6, item_7, item_8,
           item_9, item_10, item_11, item_12, item_20, item_21, item_22):
    try:
        fn()
    except AssertionError:
        raise
    except Exception as exc:
        n = int(fn.__name__.split("_")[1])
        emit(n, f"failed: {type(exc).__name__}", error=str(exc)[:400])
        print(f"  [item {n}] {type(exc).__name__}: {str(exc)[:200]}")

## Provenance

Cell 12 of the source notebook is an earlier draft of the sampler that cell 13
supersedes; only cell 13 ran in the published pipeline, so only cell 13 is
included here. Every other cell from the preflight through the analyses is
present unchanged.